### PowerFlow with GNN 
Simple network with 9 buses and PyPSA as comparison.

In [ ]:
# Standard library imports
import logging
import time
import random
import warnings
from collections import defaultdict
from typing import List, Dict, Optional, Tuple  # If you use type hints
#Scientific computing
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, lil_matrix

# Visualization  
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# NetworkX
import networkx as nx

# PyTorch and PyTorch Geometric
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.utils import to_networkx

# Progress bars
from tqdm import tqdm

# Power system analysis
import pypsa

# Set up logging
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)


## Setting up the conventional power flow network and solutions

In [ ]:
# In the official version generators are connected to bus 1, 2 and 3. 
# This is used as default, but can be changed by specifying bus numbers for each gen.


def create_9_bus_network_topology_variants(
    # Topology parameters
    gen_buses: List[int] = [1, 2, 3],
    load_buses: List[int] = [5, 7, 9],
    
    # Time series parameters
    steps: int = 5,
    
    # Generator setpoints (p.u.)
    p_set_generators: Dict[int, float] = None,  # {bus_id: p_set}
    
    # Load parameters
    load_base_p: Dict[int, Tuple[float, float]] = None,  # {bus_id: (min, max)}
    load_base_q: Dict[int, Tuple[float, float]] = None,
    load_volatility: float = 0.1,
    
    # Line/transformer modifications
    line_modifications: Dict[str, Dict[str, float]] = None,  # {line_name: {param: value}}
    
    # Additional lines for new load buses (beyond bus 9)
    additional_lines: List[Dict] = None,  # [{"from": int, "to": int, "r": float, "x": float, ...}]
    
    # System parameters
    sbase: float = 1.0,
    vnom: float = 1.0,
    
    # Other parameters
    plot: bool = False,
    seed: Optional[int] = None
) -> pypsa.Network:
    """
    Create IEEE 9-bus network with flexible topology variants for GNN training.
    
    Parameters
    ----------
    gen_buses : List[int]
        Bus indices where generators are connected (1-3 generators required)
    load_buses : List[int]
        Bus indices where loads are connected. Can extend beyond bus 9.
        If a load bus conflicts with a generator bus, it will be skipped.
    steps : int
        Number of time steps for the simulation
    p_set_generators : Dict[int, float], optional
        Active power setpoints for non-slack generators {bus_id: p_set in p.u.}
        If None, uses default values
    load_base_p : Dict[int, Tuple[float, float]], optional
        Active load ranges {bus_id: (min_MW, max_MW)} for random initialization
        If None or incomplete, uses defaults for missing buses
    load_base_q : Dict[int, Tuple[float, float]], optional
        Reactive load ranges {bus_id: (min_MVAr, max_MVAr)}
        If None or incomplete, uses defaults for missing buses
    load_volatility : float
        Volatility parameter for Brownian motion load variations (0-1)
    line_modifications : Dict[str, Dict[str, float]], optional
        Line/transformer parameter modifications
        Example: {'Line 1': {'r': 0.015, 's_nom': 1.2}}
    additional_lines : List[Dict], optional
        Additional lines/transformers for extended topology
        Example: [{"from": 9, "to": 10, "r": 0.01, "x": 0.08, "b": 0.15, 
                   "s_nom": 1.0, "tap": 1.0, "type": "Line"}]
    sbase : float
        System base power in MVA (use 1.0 for per-unit GNN training)
    vnom : float
        Nominal voltage in p.u.
    plot : bool
        Whether to plot the network
    seed : Optional[int]
        Random seed for reproducibility
        
    Returns
    -------
    pypsa.Network
        Configured PyPSA network object
        
    Raises
    ------
    AssertionError
        If validation constraints are violated
    """
    
    # ============================================
    # VALIDATION (Priority: generators first)
    # ============================================
    
    # Validate generator buses (1-3 generators required)
    assert 1 <= len(gen_buses) <= 3, (
        f"Must have 1-3 generators, got {len(gen_buses)}"
    )
    assert len(set(gen_buses)) == len(gen_buses), (
        "Generator buses must be unique"
    )
    assert all(isinstance(b, int) and b >= 1 for b in gen_buses), (
        "Generator bus indices must be positive integers"
    )
    
    logger.info(f"Generator buses: {gen_buses}")
    
    # Validate and filter load buses
    assert len(load_buses) >= 1, "Must have at least 1 load bus"
    assert all(isinstance(b, int) and b >= 1 for b in load_buses), (
        "Load bus indices must be positive integers"
    )
    
    # Check for conflicts: load buses cannot be generator buses
    original_load_buses = load_buses.copy()
    load_buses_filtered = []
    skipped_buses = []
    
    for bus in load_buses:
        if bus in gen_buses:
            skipped_buses.append(bus)
            logger.warning(
                f"Load bus {bus} conflicts with generator bus. Skipping load at bus {bus}."
            )
        elif bus in load_buses_filtered:
            logger.warning(
                f"Duplicate load bus {bus} found. Skipping duplicate."
            )
        else:
            load_buses_filtered.append(bus)
    
    load_buses = load_buses_filtered
    
    assert len(load_buses) >= 1, (
        f"After removing conflicts with generator buses {gen_buses}, "
        f"no valid load buses remain from {original_load_buses}"
    )
    
    logger.info(f"Valid load buses after conflict resolution: {load_buses}")
    if skipped_buses:
        logger.info(f"Skipped load buses due to generator conflicts: {skipped_buses}")
    
    # Determine maximum bus index needed
    max_bus_needed = max(max(gen_buses), max(load_buses))
    if additional_lines:
        for line_spec in additional_lines:
            max_bus_needed=max(max_bus_needed,line_spec["from"],line_spec["to"])
    if max_bus_needed > 9:
        logger.info(f"Extended topology: creating {max_bus_needed} buses (original IEEE: 9)")
    
    # ============================================
    # SET DEFAULTS
    # ============================================
    
    if p_set_generators is None:
        # Assign default setpoints to non-slack generators
        p_set_generators = {}
        non_slack_gens = [b for b in gen_buses if b != gen_buses[0]]
        default_setpoints = [1.63, 0.85]  # For 2nd and 3rd generator
        for i, bus in enumerate(non_slack_gens):
            if i < len(default_setpoints):
                p_set_generators[bus] = default_setpoints[i]
            else:
                p_set_generators[bus] = 0.5  # Default for additional generators
    
    # Handle load_base_p defaults - ensure all load buses have entries
    if load_base_p is None:
        load_base_p = {}
    
    # Default IEEE 9-bus load values (in p.u.)
    default_loads_p = {5: (0.9, 1.25), 7: (0.9, 1.25), 9: (0.9, 1.25)}
    
    # Fill in missing entries for all load buses
    for bus in load_buses:
        if bus not in load_base_p:
            if bus in default_loads_p:
                load_base_p[bus] = default_loads_p[bus]
            else:
                # For extended buses not specified, use moderate default
                load_base_p[bus] = (0.5, 1.0)
                logger.info(f"Using default load_base_p for bus {bus}: {load_base_p[bus]}")
    
    # Handle load_base_q defaults - ensure all load buses have entries
    if load_base_q is None:
        load_base_q = {}
    
    default_loads_q = {5: (0.3, 0.5), 7: (0.3, 0.5), 9: (0.3, 0.5)}
    
    # Fill in missing entries for all load buses
    for bus in load_buses:
        if bus not in load_base_q:
            if bus in default_loads_q:
                load_base_q[bus] = default_loads_q[bus]
            else:
                # For extended buses not specified, use moderate default
                load_base_q[bus] = (0.2, 0.4)
                logger.info(f"Using default load_base_q for bus {bus}: {load_base_q[bus]}")
    
    # ============================================
    # INITIALIZE NETWORK
    # ============================================
    
    network = pypsa.Network()
    snapshots = pd.date_range("2025-01-01 00:00", periods=steps, freq="h")
    network.set_snapshots(snapshots)
    network.sbase = sbase
    
    # ============================================
    # BUS COORDINATES
    # ============================================
    
    # IEEE 9-bus layout coordinates
    base_coordinates = {
        1: (0, 0),      # Bottom center
        2: (4, 6),      # Top right
        3: (-4, 6),     # Top left
        4: (0, 2),      # Lower middle
        5: (-2, 4),     # Mid left
        6: (-2, 6),     # Upper left
        7: (0, 6),      # Top center
        8: (2, 6),      # Upper right
        9: (2, 4)       # Mid right
    }
    
    # Extend coordinates for buses beyond 9
    bus_coordinates = base_coordinates.copy()
    if max_bus_needed > 9:
        # Simple extension pattern: place additional buses in in a grid outside of the existing buses but add jitter to avoid overlapping lines
        angle_step = 360 / (max_bus_needed - 9)
        radius = 8
        for i in range(10, max_bus_needed + 1):
            angle = np.radians((i - 10) * angle_step)
            x = radius * np.cos(angle) + np.random.uniform(-1, 1)
            y = radius * np.sin(angle) + np.random.uniform(-1, 1)
            bus_coordinates[i] = (x, y)

        
    # ============================================
    # ADD BUSES
    # ============================================
    
    for i in range(1, max_bus_needed + 1):
        # Determine bus type
        if i == gen_buses[0]:
            bus_type = "Slack"
            v_set = 1.04
        elif i in gen_buses[1:]:
            bus_type = "PV"
            v_set = 1.025
        else:
            bus_type = "PQ"
            v_set = 1.0
        
        bus_data = {
            "x": bus_coordinates[i][0],
            "y": bus_coordinates[i][1],
            "v_nom": vnom,
            "v_mag_pu_set": v_set,
            "v_mag_pu_min": 0.9,
            "v_mag_pu_max": 1.1,
            "type": bus_type
        }
        network.add("Bus", f"Bus {i}", **bus_data)
    
    logger.info(f"Added {max_bus_needed} buses to network")
    
    # ============================================
    # ADD LOADS
    # ============================================
    
    if seed is not None:
        main_rng = np.random.default_rng(seed)
    else:
        main_rng = np.random.default_rng()
    
    load_rngs = main_rng.spawn(len(load_buses))
    
    for idx, bus in enumerate(load_buses):
        rng = load_rngs[idx]
        
        # Random base load within specified range
        base_p = rng.uniform(load_base_p[bus][0], load_base_p[bus][1]) * sbase
        base_q = rng.uniform(load_base_q[bus][0], load_base_q[bus][1]) * sbase
        
        # Brownian motion for temporal variation
        random_walk = np.cumsum(rng.standard_normal(steps))
        normalized_walk = (random_walk - random_walk.min()) / (
            random_walk.max() - random_walk.min() + 1e-10
        )
        
        # Scale to [-load_volatility, +load_volatility]
        variation = load_volatility * (normalized_walk - 0.5) * 2
        
        load_p_set = base_p * (1 + variation)
        load_q_set = base_q * (1 + variation)
        
        network.add("Load", f"Load {bus}",
                   bus=f"Bus {bus}",
                   p_set=pd.Series(load_p_set, index=snapshots),
                   q_set=pd.Series(load_q_set, index=snapshots))
    
    logger.info(f"Added {len(load_buses)} loads to network")
    
    # ============================================
    # ADD GENERATORS
    # ============================================
    
    # Default generator configurations (from IEEE 9-bus)
    default_gen_configs = [
        {
            "p_nom": 5.12 * 0.9 * sbase,
            "p_min_pu": 10 / (5.12 * 0.9),
            "control": "Slack"
        },
        {
            "p_nom": 2.70 * 0.85 * sbase,
            "p_min_pu": 10 / (2.70 * 0.85),
            "control": "PV"
        },
        {
            "p_nom": 1.25 * 0.85 * sbase,
            "p_min_pu": 10 / (1.25 * 0.85),
            "control": "PV"
        }
    ]
    
    for i, bus in enumerate(gen_buses):
        # Use default config if available, otherwise create generic PV
        if i < len(default_gen_configs):
            config = default_gen_configs[i].copy()
        else:
            config = {
                "p_nom": 1.0 * sbase,
                "p_min_pu": 0.1,
                "control": "PV"
            }
        
        gen_data = {
            "bus": f"Bus {bus}",
            "control": config["control"],
            "p_nom": config["p_nom"],
            "p_min_pu": config["p_min_pu"]
        }
        
        # Set p_set for non-slack generators
        if config["control"] == "Slack":
            gen_data["p_set"] = pd.Series([0.0] * steps, index=snapshots)
        else:
            p_set_value = p_set_generators.get(bus, 0.5)
            gen_data["p_set"] = pd.Series([p_set_value] * steps, index=snapshots)
        
        network.add("Generator", f"Gen {i+1}", **gen_data)
    
    logger.info(f"Added {len(gen_buses)} generators to network")
    
    # ============================================
    # ADD BASE LINES AND TRANSFORMERS
    # ============================================
    
    # Base IEEE 9-bus branch data: [from_bus, to_bus, r, x, b, s_nom, tap_ratio]
    base_branch_data = [
        [1, 4, 0.0000, 0.0576, 0.0000, 1.50, 1.0],  # Transformer
        [4, 5, 0.0100, 0.0920, 0.1580, 1.00, 1.0],  # Line
        [5, 6, 0.0390, 0.1700, 0.3580, 0.75, 1.0],  # Line
        [3, 6, 0.0000, 0.0586, 0.0000, 1.50, 1.0],  # Transformer
        [6, 7, 0.0119, 0.1008, 0.2090, 1.00, 1.0],  # Line
        [7, 8, 0.0085, 0.0720, 0.1490, 1.00, 1.0],  # Line
        [8, 2, 0.0000, 0.0625, 0.0000, 1.75, 1.0],  # Transformer
        [9, 8, 0.0320, 0.1610, 0.3060, 1.00, 1.0],  # Line
        [9, 4, 0.0100, 0.0850, 0.1760, 1.00, 1.0]   # Line
    ]
    
    # Add base lines and transformers
    branch_count = 0
    for i, branch in enumerate(base_branch_data):
        f_bus, t_bus, r, x, b, s_nom_pu, tap = branch
        
        # Only add if both buses exist in current topology
        if f_bus > max_bus_needed or t_bus > max_bus_needed:
            continue
        
        # Scale s_nom by sbase
        s_nom = s_nom_pu * sbase
        
        # Determine component type
        is_transformer = (tap != 1.0) or (r == 0.0 and x > 0.0)
        component_type = "Transformer" if is_transformer else "Line"
        component_name = f"{component_type} {i+1}"
        
        # Base parameters
        params = {
            "bus0": f"Bus {f_bus}",
            "bus1": f"Bus {t_bus}",
            "r": r,
            "x": x,
            "b": b / 2,
            "s_nom": s_nom
        }
        
        # Add tap_ratio for transformers
        if is_transformer:
            params["tap_ratio"] = tap
        
        # Apply modifications if specified
        if line_modifications and component_name in line_modifications:
            for param, value in line_modifications[component_name].items():
                if param in params:
                    params[param] = value
                    logger.info(f"Modified {component_name}: {param} = {value}")
        
        # Add component to network
        network.add(component_type, component_name, **params)
        branch_count += 1
    
    logger.info(f"Added {branch_count} base lines/transformers")
    
    # ============================================
    # ADD ADDITIONAL LINES (for extended topology)
    # ============================================
    
    if additional_lines:
        for i, line_spec in enumerate(additional_lines):
            f_bus = line_spec["from"]
            t_bus = line_spec["to"]
            
             # Validate buses exist (they should now, since we included them in max_bus_needed)
            if f_bus > max_bus_needed or t_bus > max_bus_needed:
                logger.error(f"Additional line {i+1} references non-existent bus: "
                           f"from={f_bus}, to={t_bus}, max_bus={max_bus_needed}")
                raise ValueError(f"Additional line {i+1} references non-existent bus")
            
            # Determine component type
            line_type = line_spec.get("type", "Line")
            is_transformer = (line_type == "Transformer") or (
                line_spec.get("tap", 1.0) != 1.0
            )
            component_type = "Transformer" if is_transformer else "Line"
            
            # Count existing components to assign unique name
            existing_count = len(network.lines) + len(network.transformers)
            component_name = f"{component_type} {existing_count + 1}"
            
            # Build parameters
            params = {
                "bus0": f"Bus {f_bus}",
                "bus1": f"Bus {t_bus}",
                "r": line_spec.get("r", 0.01),
                "x": line_spec.get("x", 0.08),
                "b": line_spec.get("b", 0.15) / 2,
                "s_nom": line_spec.get("s_nom", 1.0) * sbase
            }
            
            if is_transformer:
                params["tap_ratio"] = line_spec.get("tap", 1.0)
            
            network.add(component_type, component_name, **params)
            logger.info(f"Added {component_name}: Bus {f_bus} -> Bus {t_bus}")
        
        logger.info(f"Added {len(additional_lines)} additional lines/transformers")
    
    # ============================================
    # FINALIZE
    # ============================================
    
    if plot:
        plot_network(network)
        #network.plot(bus_sizes=0.05, line_widths=0.5)
    
    logger.info(f"Network created successfully: {len(network.buses)} buses, "
               f"{len(network.generators)} generators, {len(network.loads)} loads, "
               f"{len(network.lines)} lines, {len(network.transformers)} transformers")
    
    return network


# ============================================
# UTILITY FUNCTION FOR BATCH GENERATION
# ============================================

def generate_topology_variants(
    n_variants: int,
    gen_bus_options: List[List[int]] = None,
    load_bus_options: List[List[int]] = None,
    **kwargs
) -> List[pypsa.Network]:
    """
    Generate multiple topology variants for GNN training dataset.
    
    Parameters
    ----------
    n_variants : int
        Number of network variants to generate
    gen_bus_options : List[List[int]], optional
        List of generator bus configurations to cycle through
        Example: [[1,2,3], [1,3,2], [2,1,3]]
    load_bus_options : List[List[int]], optional
        List of load bus configurations to cycle through
        Example: [[5,7,9], [4,6,8], [5,9]]
    **kwargs
        Additional arguments passed to create_9_bus_network_topology_variants
        
    Returns
    -------
    List[pypsa.Network]
        List of network objects with different topologies
    """
    if gen_bus_options is None:
        gen_bus_options = [[1, 2, 3]]
    
    if load_bus_options is None:
        load_bus_options = [[5, 7, 9]]
    
    networks = []
    for i in range(n_variants):
        # Cycle through topology options
        gen_buses = gen_bus_options[i % len(gen_bus_options)]
        load_buses = load_bus_options[i % len(load_bus_options)]
        
        # Use unique seed for each variant
        seed = kwargs.get('seed', None)
        if seed is not None:
            seed = seed + i
        
        logger.info(f"\n{'='*60}")
        logger.info(f"Generating network variant {i+1}/{n_variants}")
        logger.info(f"{'='*60}")
        
        network = create_9_bus_network_topology_variants(
            gen_buses=gen_buses,
            load_buses=load_buses,
            seed=seed,
            **{k: v for k, v in kwargs.items() if k != 'seed'}
        )
        networks.append(network)
    
    return networks


#Helper function for load pattern:
# no longer used as the load patters are now created using brownian motion
def create_pattern(base_pattern, num_steps):
    """
    Creates a pattern using numpy's repeat function
    Base pattern with 5 , modified to handle snapshots < 5 and snapshots > 5
    """
    pattern_length = len(base_pattern)
    repeats = np.ceil(num_steps / pattern_length).astype(int)
    extended_pattern = np.repeat(base_pattern, repeats)
    return extended_pattern[:num_steps]


In [ ]:
#to plot the network this is called within the create_9_bus_network if plot=True
#using first snapshot values 
def plot_network(n):
    plt.figure(figsize=(12, 8))
    title="IEEE \n9-bus \nSystem"
    plt.title(title,loc='left')
    # Plot base network
    n.plot(
        bus_sizes=0.01,          
        line_widths=2,          
        bus_colors='red',       
        line_colors='blue',     
        margin=0.15,            
        geomap=False           
    )

    # Add bus labels with nominal voltage
    for bus in n.buses.index:
        x = n.buses.x[bus]
        y = n.buses.y[bus]
        v_nom = n.buses.v_nom[bus]
        bus_type = n.buses.type[bus]
        plt.annotate(
            f"Bus {bus}\n{v_nom}kV\n{bus_type}",
            xy=(x, y),
            xytext=(10, 10),
            textcoords='offset points',
            bbox=dict(facecolor='white', alpha=0.7),
            fontsize=8
        )

    # Add generator labels with setpoints
    for gen in n.generators.index:
        bus = n.generators.bus[gen]
        x = n.buses.x[bus]
        y = n.buses.y[bus]
        # Access first snapshot of p_set time series
        p_set = n.generators_t.p_set[gen].iloc[0]
        p_nom = n.generators.p_nom[gen]
        plt.annotate(
            f"Gen {gen}\nP={p_set:.1f}MW\nPmax={p_nom:.1f}MW",
            xy=(x, y),
            xytext=(-75, -40),
            textcoords='offset points',
            color='magenta',
            bbox=dict(facecolor='white', alpha=0.7),
            fontsize=8
        )

    # Add load labels with setpoints
    for load in n.loads.index:
        bus = n.loads.bus[load]
        x = n.buses.x[bus]
        y = n.buses.y[bus]
        p = n.loads_t.p_set.loc[n.snapshots[0],load]
        q = n.loads_t.q_set.loc[n.snapshots[0],load]
        plt.annotate(
            f"Load {load}\nP={p:.1f}MW\nQ={q:.1f}MVAr",
            xy=(x, y),
            xytext=(-60, 0),
            textcoords='offset points',
            color='brown',
            bbox=dict(facecolor='white', alpha=0.7),
            fontsize=8
        )
    #add S_nom labels for branches
    for branch_type in ['Line','Transformer']:
        for branch in getattr(n,branch_type.lower()+'s').index:
            bus0=getattr(n,branch_type.lower()+'s').bus0[branch]
            bus1=getattr(n,branch_type.lower()+'s').bus1[branch]
            s_nom = getattr(n, branch_type.lower() + 's').s_nom[branch]
            r=getattr(n,branch_type.lower()+'s').r[branch]
            x_val=getattr(n,branch_type.lower()+'s').x[branch]
            x0, y0 = n.buses.x[bus0], n.buses.y[bus0]
            x1, y1 = n.buses.x[bus1], n.buses.y[bus1]
            mid_x, mid_y = ((x0 + x1) / 2)-0.25, ((y0 + y1)/2)-0.5
            plt.annotate(
                f"{s_nom:.1f}MVA \n r={r:.4f}\nx={x_val:.4f}",
                xy=(mid_x, mid_y),
                xytext=(0, 10),
                textcoords='offset points',
                color='green',
                bbox=dict(facecolor='white', alpha=0.7),
                fontsize=8,
                ha='center'
            )

    plt.grid(True)
    plt.axis('equal')
    
    # Add legend
    
    legend_elements = [
        Line2D([0], [0], marker='o', color='red', label='Buses', 
               markersize=8, linestyle='None'),
        Line2D([0], [0], color='blue', label='Lines/Transformers', 
               linewidth=2),
        Line2D([0], [0], marker='s', color='magenta', label='Generators',
               markersize=8, linestyle='None'),
        Line2D([0], [0], marker='s', color='brown', label='Loads',
               markersize=8, linestyle='None')
    ]
    plt.legend(handles=legend_elements, loc='lower right')
    plt.show()

In [ ]:

def model_results_old(n, print_results=False, plot_results=False):# assumes IEEE 9-bus system
    #n.lpf()
    n.pf(use_seed=True)
    # Create DataFrame with bus results for all snapshots
    bus_results = pd.DataFrame(
        index=n.snapshots,
        columns=pd.MultiIndex.from_product([
            n.buses.index,
            ['P (MW)', 'Q (MVAr)', 'V (pu)', 'Angle (deg)']
        ])
    )

    # Fill results for each snapshot
    for t in n.snapshots:
        for bus in n.buses.index:
            bus_results.loc[t, (bus, 'P (MW)')] = n.buses_t.p.loc[t, bus]
            bus_results.loc[t, (bus, 'Q (MVAr)')] = n.buses_t.q.loc[t, bus]
            bus_results.loc[t, (bus, 'V (pu)')] = n.buses_t.v_mag_pu.loc[t, bus]
            bus_results.loc[t, (bus, 'Angle (deg)')] = n.buses_t.v_ang.loc[t, bus]
    # Create DataFrame for line results
    line_results = pd.DataFrame(
        index=n.snapshots,
        columns=pd.MultiIndex.from_product([
            n.lines.index,
            ['P0 (MW)', 'P1 (MW)', 'Q0 (MVAr)', 'Q1 (MVAr)']
        ])
    )

    # Fill line results
    for t in n.snapshots:
        for line in n.lines.index:
            line_results.loc[t, (line, 'P0 (MW)')] = n.lines_t.p0.loc[t, line]
            line_results.loc[t, (line, 'P1 (MW)')] = n.lines_t.p1.loc[t, line]
            line_results.loc[t, (line, 'Q0 (MVAr)')] = n.lines_t.q0.loc[t, line]
            line_results.loc[t, (line, 'Q1 (MVAr)')] = n.lines_t.q1.loc[t, line]
    
    if print_results:
        print("\nBus Results:") 
        print(bus_results)
        print("\nLine Results:")
        print(line_results)
        #for s in results_df.index:
        #    print(results_df.loc[s])
    if plot_results:
        plot_network_results(n, bus_results)
        plot_line_results(n, line_results)

    return bus_results, line_results


def model_results(n, print_results=False, plot_results=False):
    """
    Run power flow and extract results for all buses and lines.
    Works with any network topology.
    """
    # Run power flow
    pf_results = n.pf(use_seed=True)
    
    # Check convergence
    if pf_results is not None and 'converged' in pf_results:
        converged_df = pf_results['converged']
        if isinstance(converged_df, pd.DataFrame):
            converged = converged_df.all().all()
        elif isinstance(converged_df, pd.Series):
            converged = converged_df.all()
        else:
            converged = bool(converged_df)
        
        if not converged:
            logger.warning("Power flow did not converge!")
    
    # Create DataFrame with bus results for all snapshots
    bus_results = pd.DataFrame(
        index=n.snapshots,
        columns=pd.MultiIndex.from_product([
            n.buses.index,
            ['P (MW)', 'Q (MVAr)', 'V (pu)', 'Angle (deg)']
        ])
    )

    # Fill results for each snapshot
    for t in n.snapshots:
        for bus in n.buses.index:
            bus_results.loc[t, (bus, 'P (MW)')] = n.buses_t.p.loc[t, bus]
            bus_results.loc[t, (bus, 'Q (MVAr)')] = n.buses_t.q.loc[t, bus]
            bus_results.loc[t, (bus, 'V (pu)')] = n.buses_t.v_mag_pu.loc[t, bus]
            bus_results.loc[t, (bus, 'Angle (deg)')] = n.buses_t.v_ang.loc[t, bus]
    
    # Create DataFrame for line results
    line_results = pd.DataFrame(
        index=n.snapshots,
        columns=pd.MultiIndex.from_product([
            n.lines.index,
            ['P0 (MW)', 'P1 (MW)', 'Q0 (MVAr)', 'Q1 (MVAr)']
        ])
    )

    # Fill line results
    for t in n.snapshots:
        for line in n.lines.index:
            line_results.loc[t, (line, 'P0 (MW)')] = n.lines_t.p0.loc[t, line]
            line_results.loc[t, (line, 'P1 (MW)')] = n.lines_t.p1.loc[t, line]
            line_results.loc[t, (line, 'Q0 (MVAr)')] = n.lines_t.q0.loc[t, line]
            line_results.loc[t, (line, 'Q1 (MVAr)')] = n.lines_t.q1.loc[t, line]
    
    if print_results:
        print("\nBus Results:") 
        print(bus_results)
        print("\nLine Results:")
        print(line_results)
    
    if plot_results:
        plot_network_results(n, bus_results)
        plot_line_results(n, line_results)

    return bus_results, line_results


def plot_network_results_old(network, results_df):# assumes IEEE 9-bus system
    """
    Plot network results in 4 subplots: P, Q, V_mag, and V_ang with bus types in legend
    """
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
    
    # Get data for each bus
    for bus in network.buses.index:
        # Determine bus type for legend
        if bus in network.loads.bus.values:
            bus_type = f"{bus} (Load)"
        elif bus in network.generators.bus.values:
            gen_idx = network.generators[network.generators.bus == bus].index[0]
            if network.generators.loc[gen_idx, 'control'] == 'Slack':
                bus_type = f"{bus} (Gen-Slack)"
            else:
                bus_type = f"{bus} (Gen-PV)"
        else:
            bus_type = bus
            
        # Active Power (P)
        ax1.plot(results_df.index, results_df[bus]['P (MW)'], marker='o', label=bus_type)
        ax1.set_ylabel('P (MW)')
        ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax1.grid(True)
        
        # Reactive Power (Q)
        ax2.plot(results_df.index, results_df[bus]['Q (MVAr)'], marker='o', label=bus_type)
        ax2.set_ylabel('Q (MVAr)')
        ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax2.grid(True)
        
        # Voltage Magnitude
        ax3.plot(results_df.index, results_df[bus]['V (pu)'], marker='o', label=bus_type)
        ax3.set_ylabel('Voltage (p.u.)')
        ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax3.grid(True)
        
        # Voltage Angle
        ax4.plot(results_df.index, results_df[bus]['Angle (deg)'], marker='o', label=bus_type)
        ax4.set_ylabel('Angle (degrees)')
        ax4.set_xlabel('Time')
        ax4.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax4.grid(True)

    plt.tight_layout()
    plt.show()

def plot_line_results_old(network, results_df):# assumes IEEE 9-bus system
    """
    Plot line flow results with one subplot per line, showing P0/P1 and Q0/Q1
    """
    n_lines = len(network.lines)
    fig, axs = plt.subplots(n_lines, 2, figsize=(12, 2*n_lines), sharex=True)
    
    # Get data for each line
    for i, line in enumerate(network.lines.index):
        # Get from/to bus names for title
        from_bus = network.lines.loc[line, 'bus0']
        to_bus = network.lines.loc[line, 'bus1']
        line_name = f"{line} ({from_bus}->{to_bus})"
        
        # Get S_nom for reference line
        s_nom = network.lines.loc[line, 's_nom']
        
        # Active Power (P) - Left subplot
        axs[i, 0].plot(results_df.index, results_df[line]['P0 (MW)'], marker='o', 
                label=f"P-in (from {from_bus})")
        axs[i, 0].plot(results_df.index, -results_df[line]['P1 (MW)'], marker='s', 
                label=f"P-out (to {to_bus})")
        # Add S_nom reference lines
        axs[i, 0].axhline(y=s_nom, color='r', linestyle='--', 
                          label=f"S_nom = {s_nom} MVA")
        axs[i, 0].axhline(y=-s_nom, color='r', linestyle='--', 
                          label=f"S_nom = {-s_nom} MVA")
        axs[i, 0].set_ylabel('P (MW)')
        axs[i, 0].set_title(f"Active Power Flow - {line_name}")
        axs[i, 0].legend()
        axs[i, 0].grid(True)
        
        # Reactive Power (Q) - Right subplot
        axs[i, 1].plot(results_df.index, results_df[line]['Q0 (MVAr)'], marker='o', 
                label=f"Q-in (from {from_bus})")
        axs[i, 1].plot(results_df.index, -results_df[line]['Q1 (MVAr)'], marker='s', 
                label=f"Q-out(to {to_bus})")
        axs[i, 1].set_ylabel('Q (MVAr)')
        axs[i, 1].set_title(f"Reactive Power Flow - {line_name}")
        axs[i, 1].legend()
        axs[i, 1].grid(True)
        
    # Add common x-axis label
    fig.text(0.5, 0.04, 'Time', ha='center')
    
    plt.tight_layout()
    plt.show()

def get_bus_type_label(network, bus):
    """
    Determine the type and label for a bus dynamically.
    
    Returns:
        str: Label indicating bus type (e.g., "Bus 5 (Load)", "Bus 1 (Gen-Slack)")
    """
    # Check if bus has a load
    has_load = bus in network.loads.bus.values
    
    # Check if bus has a generator
    has_gen = bus in network.generators.bus.values
    
    # Determine bus type
    if has_gen and has_load:
        # Bus has both generator and load
        gen_mask = network.generators.bus == bus
        if gen_mask.any():
            gen_idx = network.generators[gen_mask].index[0]
            control_type = network.generators.loc[gen_idx, 'control']
            if control_type == 'Slack':
                return f"Bus {bus} (Gen-Slack+Load)"
            else:
                return f"Bus {bus} (Gen-PV+Load)"
    elif has_gen:
        # Bus has only generator
        gen_mask = network.generators.bus == bus
        gen_idx = network.generators[gen_mask].index[0]
        control_type = network.generators.loc[gen_idx, 'control']
        if control_type == 'Slack':
            return f"Bus {bus} (Gen-Slack)"
        else:
            return f"Bus {bus} (Gen-PV)"
    elif has_load:
        # Bus has only load
        return f"Bus {bus} (Load)"
    else:
        # Bus is neither (transit bus)
        return f"Bus {bus} (Transit)"


def plot_network_results(network, results_df):
    """
    Plot network results in 4 subplots: P, Q, V_mag, and V_ang with bus types in legend.
    Works with any network topology.
    """
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
    
    # Sort buses for consistent plotting (slack first, then gen, then load, then transit)
    def bus_sort_key(bus):
        """Sort key: Slack=0, Gen=1, Load=2, Transit=3, then by bus number"""
        is_gen = bus in network.generators.bus.values
        is_load = bus in network.loads.bus.values
        
        if is_gen:
            gen_mask = network.generators.bus == bus
            if gen_mask.any():
                gen_idx = network.generators[gen_mask].index[0]
                if network.generators.loc[gen_idx, 'control'] == 'Slack':
                    return (0, int(bus) if str(bus).isdigit() else 999)
            return (1, int(bus) if str(bus).isdigit() else 999)
        elif is_load:
            return (2, int(bus) if str(bus).isdigit() else 999)
        else:
            return (3, int(bus) if str(bus).isdigit() else 999)
    
    sorted_buses = sorted(network.buses.index, key=bus_sort_key)
    
    # Plot data for each bus
    for bus in sorted_buses:
        # Get bus type label
        bus_label = get_bus_type_label(network, bus)
        
        # Active Power (P)
        ax1.plot(results_df.index, results_df[bus]['P (MW)'], 
                marker='o', label=bus_label, linewidth=2)
        
        # Reactive Power (Q)
        ax2.plot(results_df.index, results_df[bus]['Q (MVAr)'], 
                marker='o', label=bus_label, linewidth=2)
        
        # Voltage Magnitude
        ax3.plot(results_df.index, results_df[bus]['V (pu)'], 
                marker='o', label=bus_label, linewidth=2)
        
        # Voltage Angle
        ax4.plot(results_df.index, results_df[bus]['Angle (deg)'], 
                marker='o', label=bus_label, linewidth=2)
    
    # ===================================================================
    # ADD FORMATTING AND REFERENCE LINES AFTER THE LOOP (ONLY ONCE)
    # ===================================================================
    
    # Subplot 1: Active Power
    ax1.set_ylabel('P (MW)', fontsize=12)
    ax1.set_title('Active Power Injection by Bus', fontsize=13, fontweight='bold')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # Subplot 2: Reactive Power
    ax2.set_ylabel('Q (MVAr)', fontsize=12)
    ax2.set_title('Reactive Power Injection by Bus', fontsize=13, fontweight='bold')
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    # Subplot 3: Voltage Magnitude
    ax3.set_ylabel('Voltage (p.u.)', fontsize=12)
    ax3.set_title('Voltage Magnitude by Bus', fontsize=13, fontweight='bold')
    # Add reference line ONCE, after all bus data is plotted
    ax3.axhline(y=1.0, color='k', linestyle='--', alpha=0.5, linewidth=1.5, 
                label='Nominal (1.0 pu)')
    ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    ax3.grid(True, alpha=0.3)
    
    # Subplot 4: Voltage Angle
    ax4.set_ylabel('Angle (degrees)', fontsize=12)
    ax4.set_xlabel('Time Step', fontsize=12)
    ax4.set_title('Voltage Angle by Bus', fontsize=13, fontweight='bold')
    # Add reference line ONCE, after all bus data is plotted
    ax4.axhline(y=0.0, color='k', linestyle='--', alpha=0.5, linewidth=1.5, 
                label='Reference (0°)')
    ax4.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()



def plot_line_results(network, results_df):
    """
    Plot line flow results with one subplot per line, showing P0/P1 and Q0/Q1.
    Works with any network topology (lines only, not transformers).
    """
    if len(network.lines) == 0:
        logger.warning("No lines to plot")
        return
    
    n_lines = len(network.lines)
    fig, axs = plt.subplots(n_lines, 2, figsize=(14, 2.5*n_lines), sharex=True)
    
    # Handle case of single line (axs is 1D instead of 2D)
    if n_lines == 1:
        axs = axs.reshape(1, -1)
    
    # Get data for each line
    for i, line in enumerate(network.lines.index):
        # Get from/to bus names for title
        from_bus = network.lines.loc[line, 'bus0']
        to_bus = network.lines.loc[line, 'bus1']
        
        # Get bus type labels
        from_label = get_bus_type_label(network, from_bus)
        to_label = get_bus_type_label(network, to_bus)
        
        line_name = f"{line}: {from_bus}→{to_bus}"
        
        # Get S_nom for reference line
        s_nom = network.lines.loc[line, 's_nom']
        
        # Active Power (P) - Left subplot
        axs[i, 0].plot(results_df.index, results_df[line]['P0 (MW)'], 
                      marker='o', linewidth=2, label=f"P₀ from {from_bus}")
        axs[i, 0].plot(results_df.index, -results_df[line]['P1 (MW)'], 
                      marker='s', linewidth=2, label=f"P₁ to {to_bus}")
        
        # Add S_nom reference lines
        axs[i, 0].axhline(y=s_nom, color='r', linestyle='--', alpha=0.7,
                          label=f"S_nom = ±{s_nom:.2f} MVA")
        axs[i, 0].axhline(y=-s_nom, color='r', linestyle='--', alpha=0.7)
        
        axs[i, 0].set_ylabel('P (MW)', fontsize=11)
        axs[i, 0].set_title(f"Active Power - {line_name}", fontsize=11, fontweight='bold')
        axs[i, 0].legend(fontsize=9)
        axs[i, 0].grid(True, alpha=0.3)
        
        # Reactive Power (Q) - Right subplot
        axs[i, 1].plot(results_df.index, results_df[line]['Q0 (MVAr)'], 
                      marker='o', linewidth=2, label=f"Q₀ from {from_bus}")
        axs[i, 1].plot(results_df.index, -results_df[line]['Q1 (MVAr)'], 
                      marker='s', linewidth=2, label=f"Q₁ to {to_bus}")
        
        axs[i, 1].set_ylabel('Q (MVAr)', fontsize=11)
        axs[i, 1].set_title(f"Reactive Power - {line_name}", fontsize=11, fontweight='bold')
        axs[i, 1].legend(fontsize=9)
        axs[i, 1].grid(True, alpha=0.3)
    
    # Add common x-axis label
    fig.text(0.5, 0.02, 'Time Step', ha='center', fontsize=12)
    
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()


def plot_transformer_results(network, results_df=None):
    """
    Plot transformer flow results separately.
    Similar to plot_line_results but for transformers.
    """
    if len(network.transformers) == 0:
        logger.warning("No transformers to plot")
        return
    
    # If results_df not provided, run power flow
    if results_df is None:
        n.pf(use_seed=True)
        results_df = pd.DataFrame(
            index=network.snapshots,
            columns=pd.MultiIndex.from_product([
                network.transformers.index,
                ['P0 (MW)', 'P1 (MW)', 'Q0 (MVAr)', 'Q1 (MVAr)']
            ])
        )
        for t in network.snapshots:
            for trafo in network.transformers.index:
                results_df.loc[t, (trafo, 'P0 (MW)')] = network.transformers_t.p0.loc[t, trafo]
                results_df.loc[t, (trafo, 'P1 (MW)')] = network.transformers_t.p1.loc[t, trafo]
                results_df.loc[t, (trafo, 'Q0 (MVAr)')] = network.transformers_t.q0.loc[t, trafo]
                results_df.loc[t, (trafo, 'Q1 (MVAr)')] = network.transformers_t.q1.loc[t, trafo]
    
    n_trafos = len(network.transformers)
    fig, axs = plt.subplots(n_trafos, 2, figsize=(14, 2.5*n_trafos), sharex=True)
    
    if n_trafos == 1:
        axs = axs.reshape(1, -1)
    
    for i, trafo in enumerate(network.transformers.index):
        from_bus = network.transformers.loc[trafo, 'bus0']
        to_bus = network.transformers.loc[trafo, 'bus1']
        
        from_label = get_bus_type_label(network, from_bus)
        to_label = get_bus_type_label(network, to_bus)
        
        trafo_name = f"{trafo}: {from_bus}→{to_bus}"
        s_nom = network.transformers.loc[trafo, 's_nom']
        
        # Active Power
        axs[i, 0].plot(results_df.index, results_df[trafo]['P0 (MW)'], 
                      marker='o', linewidth=2, label=f"P₀ from {from_bus}")
        axs[i, 0].plot(results_df.index, -results_df[trafo]['P1 (MW)'], 
                      marker='s', linewidth=2, label=f"P₁ to {to_bus}")
        axs[i, 0].axhline(y=s_nom, color='r', linestyle='--', alpha=0.7,
                          label=f"S_nom = ±{s_nom:.2f} MVA")
        axs[i, 0].axhline(y=-s_nom, color='r', linestyle='--', alpha=0.7)
        axs[i, 0].set_ylabel('P (MW)', fontsize=11)
        axs[i, 0].set_title(f"Active Power - {trafo_name}", fontsize=11, fontweight='bold')
        axs[i, 0].legend(fontsize=9)
        axs[i, 0].grid(True, alpha=0.3)
        
        # Reactive Power
        axs[i, 1].plot(results_df.index, results_df[trafo]['Q0 (MVAr)'], 
                      marker='o', linewidth=2, label=f"Q₀ from {from_bus}")
        axs[i, 1].plot(results_df.index, -results_df[trafo]['Q1 (MVAr)'], 
                      marker='s', linewidth=2, label=f"Q₁ to {to_bus}")
        axs[i, 1].set_ylabel('Q (MVAr)', fontsize=11)
        axs[i, 1].set_title(f"Reactive Power - {trafo_name}", fontsize=11, fontweight='bold')
        axs[i, 1].legend(fontsize=9)
        axs[i, 1].grid(True, alpha=0.3)
    
    fig.text(0.5, 0.02, 'Time Step', ha='center', fontsize=12)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()



In [ ]:
# Add load at bus 10 (beyond original 9-bus system)
additional_connections = [
    {"from": 9, "to": 10, "r": 0.01, "x": 0.08, "b": 0.15, "s_nom": 1.0}
]

network = create_9_bus_network_topology_variants(
    gen_buses=[1, 2],  # Only 2 generators
    load_buses=[5, 7, 9, 10],  # Load at extended bus 10
    additional_lines=additional_connections,
    load_base_p={10: (0.8, 1.2)},  # Specify load range for bus 10
    load_base_q={10: (0.25, 0.45)},
    steps=24,
    seed=42
)


In [ ]:
# Create 12-bus system
new_lines = [
    {"from": 9, "to": 10, "r": 0.01, "x": 0.08, "b": 0.15, "s_nom": 1.0},
    {"from": 5, "to": 11, "r": 0.015, "x": 0.09, "b": 0.18, "s_nom": 0.8},
    {"from": 4, "to": 12, "r": 0.012, "x": 0.085, "b": 0.16, "s_nom": 0.9},
    {"from": 3, "to": 13, "r": 0.015, "x": 0.075, "b": 0.17, "s_nom": 0.7}
]

network = create_9_bus_network_topology_variants(
    gen_buses=[1, 2, 13],
    load_buses=[5, 7, 9, 10, 11, 12],
    additional_lines=new_lines,
    steps=48,
    seed=42,
    plot=True
)


In [ ]:
bus_results,line_results=model_results(network,print_results=False, plot_results=False) # run the power flow and store results in dfs. Print if print_results = True, plot results if Plot=True

## Preparing the GNN

In [ ]:
class PowerFlowDataset(Dataset):# Create a custom PyTorch Geometric Dataset for power flow data
    def __init__(self, networks, use_edge_features=True, transform=None):
        """
        Dataset for power flow prediction using PyTorch Geometric
        
        Args:
            networks: List of PyPSA network objects with solved power flow
        """
        super().__init__(transform=transform)
        self.networks = networks
        self.use_edge_features = use_edge_features  #[STSI 08.02.26]: toggle edge features
        self.network_indices = []  # Track which network each sample belongs to
        self.processed_data = [] 
        self.process_networks()
        
        
    def process_networks(self):
        """Pre-process networks to extract features and targets"""
        self.processed_data = []
        
        for net_idx, network in enumerate(self.networks):
            # Extract snapshot data
            for t in network.snapshots:
                # Create graph data for this snapshot
                graph_data = self._create_graph_data(network, t, net_idx) # Create graph data for this snapshot
                self.processed_data.append(graph_data) # Append to processed data list
                self.network_indices.append(net_idx) # Added network index to track to handle topology changes
    
    """earlier version where only whitelisted attributes were passed as inputs
    def prepare_input_features(self,data,bus_types):
        input_features = np.zeros_like(data)
        #include only the known values as input features
        for i, bus_type in enumerate(bus_types):
            if bus_type == 'Slack': #slack bus
                input_features[i,5] = data[i,5]  # v_mag known
                input_features[i, 6] = data[i, 6]  # v_ang known
            elif bus_type == 'PV': #PV bus
                input_features[i, 3] = data[i, 3]  # P known
                input_features[i,5] = data[i,5]  # v_mag known
            elif bus_type=='PQ':  # PQ bus
                input_features[i, 3] = data[i,3]  # P known
                input_features[i, 4] = data[i,4]  # Q known
        return input_features
    """


    def input_feature_filter(self,data,bus_types): #[STSI 240925]: Added input filtering to make sure unknown values are masked to zero for the training
        input_features = data.copy()  # Start with all data
    
        for i, bus_type in enumerate(bus_types):
            if bus_type == 'PQ':
                # KEEP bus types [0:3] and known P[3], Q[4]
                # MASK unknown V_mag[5], V_ang[6]
                input_features[i, 5] = 0.0  
                input_features[i, 6] = 0.0
            elif bus_type == 'PV':
                # KEEP bus types [0:3], P[3], V_mag[5]
                # MASK unknown Q[4], V_ang[6]
                input_features[i, 4] = 0.0
                input_features[i, 6] = 0.0
            elif bus_type == 'Slack':
                # KEEP bus types [0:3], V_mag[5], V_ang[6] 
                # MASK unknown P[3], Q[4]
                input_features[i, 3] = 0.0
                input_features[i, 4] = 0.0
        
        return input_features


    def _create_graph_data(self, network, snapshot, net_idx):
        """Convert PyPSA network at a snapshot to PyG Data object"""
        num_buses = len(network.buses)
        
        # Node features: [bus_type, p, q, v_mag, v_ang]
        
        bus_types = np.zeros((num_buses, 3))
        
        # Extract bus data for this snapshot
        p = network.buses_t.p.loc[snapshot].values # Active power injections
        q = network.buses_t.q.loc[snapshot].values # Reactive power injections
        v_mag = network.buses_t.v_mag_pu.loc[snapshot].values # Voltage magnitudes
        v_ang = network.buses_t.v_ang.loc[snapshot].values # Voltage angles
        
        # One-hot encode bus types: [is_slack, is_pv, is_pq]
        bus_types_list=[] # to store bus types for tracking
        for i, bus in enumerate(network.buses.index):
            bus_type = network.buses.loc[bus, 'type']
            bus_types_list.append(bus_type) # added to track when topology changes
            if bus_type == 'Slack':
                bus_types[i, 0] = 1  # Slack
            elif bus_type == 'PV':
                bus_types[i, 1] = 1  # PV
            else:
                bus_types[i, 2] = 1  # PQ
        
        # Combine features into a node feature tensor
        # Note: bus_types is already a one-hot encoded matrix
        
        #Old version with all values
        #node_features = np.column_stack([bus_types, p, q, v_mag, v_ang]) # Problem here with passing all values as inputs, also the solution from the conventional power flow
        # New version where only the known variables are passed as inputs
        
        node_features = self.input_feature_filter(np.column_stack([bus_types, p, q, v_mag, v_ang]), network.buses['type'].values)
        
 
        # Creating the edge indices and attributes
        edge_index = []
        edge_attr = []
        
    #[STSI 15.01.2026] Creating bus name to index mapping to handle different bus numbering when topology changes
        bus_to_idx={bus_name: idx for idx, bus_name in enumerate(network.buses.index)}

    #[STSI 08.02.26]: Use normalized, physics-informed edge features
        sbase = float(getattr(network, "sbase", 1.0))  # fall back to 1.0 if not set

        # Process lines
        for _, line in network.lines.iterrows():
            from_bus = bus_to_idx[line['bus0']]
            to_bus   = bus_to_idx[line['bus1']]

            edge_index.append([from_bus, to_bus])
            edge_index.append([to_bus, from_bus])

            if self.use_edge_features:
                r = float(line['r'])
                x = float(line['x'])
                b = float(line['b'])
                s_nom_pu = float(line['s_nom'] / sbase)

                if r == 0.0 and x == 0.0:
                    y_mag = 0.0
                else:
                    y_mag = 1.0 / np.sqrt(r**2 + x**2)

                edge_features = [r, x, b, s_nom_pu, y_mag, 0.0]  # 0.0 = line
            else:
                # Minimal placeholder (fixed 1-D feature)
                edge_features = [1.0]

            edge_attr.append(edge_features)
            edge_attr.append(edge_features)  # Same features for reverse direction
        
        # Process transformers if any
        for _, trafo in network.transformers.iterrows():
            from_bus = bus_to_idx[trafo['bus0']]
            to_bus   = bus_to_idx[trafo['bus1']]

            edge_index.append([from_bus, to_bus])
            edge_index.append([to_bus, from_bus])

            if self.use_edge_features:
                r = float(trafo['r'])
                x = float(trafo['x'])
                b = float(trafo['b'])
                s_nom_pu = float(trafo['s_nom'] / sbase)
                tap = float(trafo.get('tap_ratio', 1.0))

                if r == 0.0 and x == 0.0:
                    y_mag = 0.0
                else:
                    y_mag = 1.0 / np.sqrt(r**2 + x**2)

                edge_features = [r, x, b, s_nom_pu, y_mag, 1.0]  # 1.0 = transformer
            else:
                edge_features = [1.0]

            edge_attr.append(edge_features)
            edge_attr.append(edge_features)


        # Target values: what we want to predict
        # For PQ buses: predict v_mag, v_ang
        # For PV buses: predict v_ang, q
        # For Slack bus: predict p, q
        target = np.zeros((num_buses, 4))  # [v_mag, v_ang, p, q]
        
        for i, bus in enumerate(network.buses.index):
            bus_type = network.buses.loc[bus, 'type']
            
            if bus_type == 'PQ':
                # For PQ buses, predict voltage magnitude and angle
                target[i, 0] = v_mag[i]  # v_mag
                target[i, 1] = v_ang[i]  # v_ang
            elif bus_type == 'PV':
                # For PV buses, predict angle and reactive power
                target[i, 1] = v_ang[i]  # v_ang
                target[i, 3] = q[i]      # q # [STSI 24.09.25]:Corrected here, was target [i,2] before
            else:  # Slack
                # For slack bus, predict active and reactive power
                target[i, 2] = p[i]      # p
                target[i, 3] = q[i]      # q
        
        
        # Convert to PyTorch tensors to create Data object
        # Note: PyTorch Geometric expects edge_index to be in COO format (2D tensor)
        # Convert edge_index to a 2D tensor
        x = torch.tensor(node_features, dtype=torch.float)
        # Contigous is used to ensure that the tensor is stored in contiguous memory for faster access
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous() # Transpose to COO format which means basically a 2D tensor
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)
        y = torch.tensor(target, dtype=torch.float)
        
        # Create masks for different bus types, masks are used to filter out the bus types
        slack_mask = torch.tensor(bus_types[:, 0].astype(bool))
        pv_mask = torch.tensor(bus_types[:, 1].astype(bool))
        pq_mask = torch.tensor(bus_types[:, 2].astype(bool))
        
        return Data(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=y,
            slack_mask=slack_mask,
            pv_mask=pv_mask,
            pq_mask=pq_mask,
            network_idx=torch.tensor([net_idx],dtype=torch.long), #[STSI 15.01.2026] the network indice to track when topology is changing
            num_nodes=num_buses #[STSI 15.01.2026]: keep track as number of buses can change
        )
    
    
    def len(self):
        return len(self.processed_data)
    
    def get(self, idx):
        return self.processed_data[idx]
    def get_network_for_sample(self, idx):
        """
        [STSI 15.01.2026]: Added- Get the original network object for a given sample index
        
        Args:
            idx: Sample index
            
        Returns:
            pypsa.Network: The network object this sample came from
        """
        net_idx = self.network_indices[idx]
        return self.networks[net_idx]

In [ ]:
# THe model is a simple GNN with 3 layers and 4 heads, using GATv2Conv for graph convolution
# and GCNConv for the final output layer. The model predicts voltage magnitude, angle, active power, and reactive power.


class PowerFlowGNN(nn.Module):
    def __init__(self, node_features=8, edge_features=6, hidden_dim=64, num_layers=3):
        #Node features  [is_slack, is_pv, is_pq, p, q, v_mag, v_ang] which is only 7, but 8 is used as default for simplicity 
        # edge features [r, x, b, s_nom, y_mag, is_trafo] which is 6 (here we are not using tap_ratio)
        # hidden_dim is the dimension of the hidden layers set to 64 which means 64 neurons in each layer
        super().__init__()
        
        # Initial node embedding
        self.node_embedding = nn.Linear(node_features, hidden_dim) # here node_features are embedded to hidden_dim to get the initial node features
        
        # Graph convolution layers
        self.convs = nn.ModuleList() # module list is used to store the layers
        for _ in range(num_layers):    #  looping through the number of layers to create the layers and append them to the list
            self.convs.append(GATv2Conv(
                in_channels=hidden_dim, # input channels are hidden_dim because the output of the previous layer is the input to the next layer
                out_channels=hidden_dim, # output channels are also hidden_dim to maintain the same dimensionality across layers
                edge_dim=edge_features, # edge features are used to create the edge features for the graph convolution
                heads=4,    # create 4 different attention heads for the graph convolution, one for each layer. The purpose of the attention heads is to create different attention weights for the edges in the graph. this enables the model to capture different relationships between the nodes in the graph.
                concat=False # this is set to false to concatenate the output of the attention heads, which means that the output of the attention heads is averaged instead of concatenated
            ))
        
        # Output layers for different predictions which one prediction head for each of the properties we want to predict
        self.v_mag_pred = nn.Linear(hidden_dim, 1)  # Voltage magnitude
        self.v_ang_pred = nn.Linear(hidden_dim, 1)  # Voltage angle
        self.p_pred = nn.Linear(hidden_dim, 1)      # Active power
        self.q_pred = nn.Linear(hidden_dim, 1)      # Reactive power
        
        self._initialize_weights()

    def forward(self, data): # Forward pass for the model where the features are embedded first, then processed through the graph convolution layers, and finally the predictions are made
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        
        # Initial embedding
        x = self.node_embedding(x) # the node features are embedded to hidden_dim
        x = F.leaky_relu(x) # the activation function used is leaky relu which is a variant of relu that allows a small gradient when the input is negative. 
        
        # Graph convolution layers 
        # The basic idea of convolution is to apply a filter to the input data to extract features. 
        # The filter is applied to the input data in a sliding window fashion, which means that the filter is applied to a small portion of the input data at a time. 
        # This is done by moving the filter across the input data in small steps, which is called the stride. 
        #We then end up with a new set of features that are the result of applying the filter to the input data. 
        # The output of the convolution is then passed through an activation function, which introduces non-linearity into the model. 
        # This is done to allow the model to learn complex relationships between the input data and the output data.
        for conv in self.convs: # conv is the module list that contains the graph convolution layers (as declared at the start of the class)
            x = conv(x, edge_index, edge_attr) # the graph convolution layer is applied to the input data. We use GATv2Conv which applies the graph convolution to the input data and returns the output data.
            x = F.leaky_relu(x) # # The output of the graph convolution layer is then passed through an activation function. F is the functional module in pytorch that contains the activation functions.
        
        # Predictions are made using the output of the last graph convolution layer
        v_mag = self.v_mag_pred(x).squeeze(-1) #squeeze is used to remove the last dimension of the tensor which is 1 in this case. This is done to make the output tensor of shape (batch_size, num_nodes) instead of (batch_size, num_nodes, 1)
        v_ang = self.v_ang_pred(x).squeeze(-1)
        p = self.p_pred(x).squeeze(-1)
        q = self.q_pred(x).squeeze(-1)
        
        # Apply physics constraints based on bus type
        # For PQ buses: predict v_mag, v_ang
        # For PV buses: use known v_mag, predict v_ang, q
        # For Slack bus: use known v_mag, v_ang, predict p, q
        
        # Create output tensor
        output = torch.zeros_like(data.y)
        
        # Fill in predictions based on bus type
        # PQ buses: predict v_mag, v_ang
        output[data.pq_mask, 0] = v_mag[data.pq_mask]  # v_mag
        output[data.pq_mask, 1] = v_ang[data.pq_mask]  # v_ang
        
        # PV buses: use known v_mag (from input), predict v_ang, q
        output[data.pv_mask, 1] = v_ang[data.pv_mask]  # v_ang
        output[data.pv_mask, 3] = q[data.pv_mask]      # q
        
        # Slack bus: use known v_mag, v_ang (from input), predict p, q
        output[data.slack_mask, 2] = p[data.slack_mask]  # p
        output[data.slack_mask, 3] = q[data.slack_mask]  # q
        
        return output
    
    def _initialize_weights(self, gain=1.0): #[STSI 24.09.25] Added "full" initialization for all layers
        """Enhanced weight initialization for the entire network"""
        
        # 1. Initialize node embedding layer Added 24.09.25
        nn.init.xavier_uniform_(self.node_embedding.weight, gain=gain)
        nn.init.zeros_(self.node_embedding.bias)
        
        # 2. Initialize GAT convolution layers, Addded 24.09.25
        for conv in self.convs:
            # GAT layers have multiple linear transformations
            if hasattr(conv, 'lin_l') and conv.lin_l is not None:
                nn.init.xavier_uniform_(conv.lin_l.weight, gain=gain)
            if hasattr(conv, 'lin_r') and conv.lin_r is not None:
                nn.init.xavier_uniform_(conv.lin_r.weight, gain=gain)
            if hasattr(conv, 'lin_edge') and conv.lin_edge is not None:
                nn.init.xavier_uniform_(conv.lin_edge.weight, gain=gain)
        
        # 3. Initialize output layers 
        prediction_layers = [
            (self.v_mag_pred, 1.0),   # Voltage magnitude bias toward 1.0 p.u.
            (self.v_ang_pred, 0.0),   # Voltage angle bias toward 0 rad
            (self.p_pred, 0.0),       # Active power bias toward 0
            (self.q_pred, 0.0)        # Reactive power bias toward 0
        ]
        
        for layer, bias_init in prediction_layers:
            nn.init.xavier_uniform_(layer.weight, gain=gain)
            nn.init.constant_(layer.bias, bias_init)

In [ ]:

def train_power_flow_gnn(networks, num_epochs=200, batch_size=1, lr=0.001, weight_physics=0.01, use_edge_features=True,max_n_test=None):
    """
    Train power flow GNN with proper network-based train/val/test split
    """
    #[STSI25.09.25]: changed the train/test split from snapshots to networks"
    #[STSI 15.01.26]: added updates to handle topology changes 
    # Split networks first, then create datasets 
    #[STSI 08.02.26]: Work on a shuffled copy to avoid modifying original list
    networks_copy = networks.copy()
    
    random.shuffle(networks_copy)

    num_train_networks = int(0.7 * len(networks_copy))
    num_val_networks = int(0.85 * len(networks_copy)) - num_train_networks

    train_networks = networks_copy[:num_train_networks]
    val_networks = networks_copy[num_train_networks:num_train_networks + num_val_networks]
    test_networks = networks_copy[num_train_networks + num_val_networks:]

    
    # Debug: Verify split
    print(f"Network split: Train={len(train_networks)}, Val={len(val_networks)}, Test={len(test_networks)}")
    print(f"Total: {len(train_networks) + len(val_networks) + len(test_networks)} of {len(networks)}")
    
    # Create datasets from split networks
    train_dataset = PowerFlowDataset(train_networks, use_edge_features=use_edge_features)
    val_dataset = PowerFlowDataset(val_networks, use_edge_features=use_edge_features)
    test_dataset = PowerFlowDataset(test_networks, use_edge_features=use_edge_features)
    
    # Debug: Verify dataset sizes
    print(f"Dataset sizes: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")
    
    # Cache y-matrices to reduce computation during training
    logger.info("Precomputing admittance matrices...")
    train_Y_cache = precompute_Y_matrices(train_networks)
    val_Y_cache = precompute_Y_matrices(val_networks)
    test_Y_cache = precompute_Y_matrices(test_networks)
    logger.info("Y-matrices cached successfully")

    # Create data loaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True,   # Shuffle makes sure that the data is randomly sampled during training, training dataset contains multiple networks with different topologies
        follow_batch=['network_idx'] #[STSI 15.01.2026] added to keep track of which network used
        ) 
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, # Don't shuffle validation
        follow_batch=['network_idx'] #[STSI 15.01.2026] added to keep track of which network used
        )  
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, # Don't shuffle test
        follow_batch=['network_idx'] #[STSI 15.01.2026] added to keep track of which network used
        )  
    
    
    # Initialize model
    sample_data = train_dataset[0]
    model = PowerFlowGNN(
        node_features=sample_data.x.size(1), # node features are 8 as defined in the model
        edge_features=sample_data.edge_attr.size(1), # edge features are 4 as defined in the model
        hidden_dim=64,  # hidden dimension is 64 as defined in the model
        num_layers=3    # number of layers is 3 as defined in the model
    )#.to(device) # specifying device to GPU if available, can be skipped for CPU training
    print(model)
    
    # Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, # Optimizer to adjust
        mode='min', # Minimize validation loss
        factor=0.5, # Reduce LR by a factor of 0.5
        patience=10, # Number of epochs with no improvement after which learning rate will be reduced
        verbose=True # Print LR updates
        )
    
    # Training loop
    train_losses = []
    val_losses = []
    best_val_loss = float('inf') # Initialize best validation loss
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        epoch_loss = 0 # setting all losses to zero to inintialize
        epoch_mse = 0 #setting all losses to zero to inintialize 
        epoch_physics = 0 #setting all losses to zero to inintialize
        

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            optimizer.zero_grad()
            #batch = batch.to(device)  # uncomment if you move to GPU

            pred = model(batch)

            if batch_size == 1:
                #[STSI 08.02.26]: Single-graph path (backward compatible)
                if batch.network_idx.dim() == 0:
                    net_idx = int(batch.network_idx.item())
                else:
                    net_idx = int(batch.network_idx[0].item())

                loss, mse, physics = physics_informed_loss_single_graph(
                    pred,
                    batch.y,
                    batch,
                    train_networks[net_idx],
                    train_Y_cache[net_idx],
                    loss_fraction_physics=weight_physics
                )
            else:
                #[STSI 08.02.26]: Batched path using simplified wrapper
                loss, mse, physics = physics_informed_loss_batch(
                    pred,
                    batch.y,
                    batch,
                    train_networks,
                    train_Y_cache,
                    loss_fraction_physics=weight_physics
                )
  

            # Backward pass
            loss.backward() # Pass the loss to backward to compute gradients
            optimizer.step() # Update the model parameters using the optimizer: most important parameters are the learning rate and the weight decay

            epoch_loss += loss.item()
            epoch_mse += mse.item()
            epoch_physics += physics.item()

        train_loss = epoch_loss / len(train_loader)
        train_losses.append(train_loss)
        
        # Validation
        model.eval()
        val_loss = 0
        val_mse = 0
        val_physics = 0
        
        with torch.no_grad():
            for batch in val_loader:
                #batch = batch.to(device)

                pred = model(batch)

                if batch_size == 1:
                    if batch.network_idx.dim() == 0:
                        net_idx = int(batch.network_idx.item())
                    else:
                        net_idx = int(batch.network_idx[0].item())

                    loss, mse, physics = physics_informed_loss_single_graph(
                        pred,
                        batch.y,
                        batch,
                        val_networks[net_idx],
                        val_Y_cache[net_idx],
                        loss_fraction_physics=weight_physics
                    )
                else:
                    loss, mse, physics = physics_informed_loss_batch(
                        pred,
                        batch.y,
                        batch,
                        val_networks,
                        val_Y_cache,
                        loss_fraction_physics=weight_physics
                    )

                val_loss += loss.item()
                val_mse += mse.item()
                val_physics += physics.item()

        val_loss = val_loss / len(val_loader)
        val_losses.append(val_loss)
        
        # Update learning rate
        scheduler.step(val_loss)
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model.pt')
        # Print progress
        print(f"Epoch {epoch+1}/{num_epochs}, "
              f"Train Loss: {train_loss:.6f}, "
              f"Val Loss: {val_loss:.6f}, "
              f"LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Debug to detect data leakage
        if epoch == 0:  # Only first epoch
            print("\n=== DATA LEAKAGE CHECK ===")
            print(f"Input features (batch.x) shape: {batch.x.shape}")
            print(f"Input features sample:\n{batch.x[:5]}")
            print(f"Target features (batch.y) shape: {batch.y.shape}")
            print(f"Target features sample:\n{batch.y[:5]}")
            
            # Critical check: Are targets in inputs?
            print("\n=== CHECKING FOR LEAKAGE ===")
            # For PQ buses, V_mag should ONLY be in targets
            pq_indices = torch.where(batch.pq_mask)[0]
            if len(pq_indices) > 0:
                print(f"PQ bus indices: {pq_indices}")
                print(f"Input at PQ bus 0: {batch.x[pq_indices[0]]}")
                print(f"Target at PQ bus 0: {batch.y[pq_indices[0]]}")

    # Plot training curve
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f"Training and Validation Loss \n Batch Size: {batch_size}, LR: {lr}, Physics Weight: {weight_physics}")
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # Evaluate on test set
    model.load_state_dict(torch.load('best_model.pt'))
    model.eval()
    test_loss = 0
    mse_loss_total = 0
    physics_loss_total = 0
    
    with torch.no_grad():
        for batch in test_loader:
            #batch = batch.to(device)
            pred = model(batch)

            if batch_size == 1:
                if batch.network_idx.dim() == 0:
                    net_idx = int(batch.network_idx.item())
                else:
                    net_idx = int(batch.network_idx[0].item())

                loss, mse, physics = physics_informed_loss_single_graph(
                    pred,
                    batch.y,
                    batch,
                    test_networks[net_idx],
                    test_Y_cache[net_idx],
                    loss_fraction_physics=weight_physics
                )
            else:
                loss, mse, physics = physics_informed_loss_batch(
                    pred,
                    batch.y,
                    batch,
                    test_networks,
                    test_Y_cache,
                    loss_fraction_physics=weight_physics
                )
            test_loss += loss.item()
            mse_loss_total += mse.item()
            physics_loss_total += physics.item()
    
    test_loss = test_loss / len(test_loader)
    mse_loss_avg = mse_loss_total / len(test_loader)
    physics_loss_avg = physics_loss_total / len(test_loader)
    
    print("\\n" + "="*60)
    print("BASIC TEST METRICS (Batched)")
    print("="*60)
    print(f"Test Loss: {test_loss:.6f}")
    print(f"MSE Loss: {mse_loss_avg:.6f}")
    print(f"Physics Loss: {physics_loss_avg:.6f}")

    #[STSI 08.02.26]: Detailed test set evaluation across all test networks
    print("\\n" + "="*60)
    print("DETAILED TEST SET EVALUATION")
    print("="*60)

    test_results = test_model_on_networks(
        model,
        test_networks,
        num_snapshots_per_network=24,  # adjust as needed
        use_edge_features=use_edge_features, 
        max_n_test=max_n_test
    )

    return model, (train_losses, val_losses), test_results

        

In [ ]:
#no longer used keep for reference
"""
def physics_informed_loss(pred, target, data, network, snapshot_idx, loss_fraction_physics=0.5):

    #Physics-informed loss function that combines:
    #1. MSE loss between predictions and targets
    #2. Power balance constraints from Kirchhoff's laws

    # Data-driven MSE loss
 
    mse_loss = 0.0
    
    # Only compute loss for values we're trying to predict
    # For PQ buses: v_mag, v_ang
    pq_mask = data.pq_mask # t
    if pq_mask.any():
        mse_loss += F.mse_loss(pred[pq_mask, :2], target[pq_mask, :2])
    
    # For PV buses: v_ang, q
    pv_mask = data.pv_mask
    if pv_mask.any():
        mse_loss += F.mse_loss(pred[pv_mask, 1], target[pv_mask, 1])  # v_ang
        mse_loss += F.mse_loss(pred[pv_mask, 3], target[pv_mask, 3])  # q
    
    # For Slack bus: p, q
    slack_mask = data.slack_mask
    if slack_mask.any():
        mse_loss += F.mse_loss(pred[slack_mask, 2:], target[slack_mask, 2:])
    
    # Physics-based power balance constraints
    # Extract predicted values
    v_mag = torch.zeros(pred.size(0)) # creating an initial tensor of zeros with the same size as the prediction tensor
    v_ang = torch.zeros(pred.size(0))
    p = torch.zeros(pred.size(0))
    q = torch.zeros(pred.size(0))
    
    # Based on node_features structure:
    # x = [bus_type_one_hot (3 dims), p, q, v_mag, v_ang]
    # Indices are: p=3, q=4, v_mag=5, v_ang=6
    # for the prediction tenso the incices are  v_mag=0, v_ang=1, p=2, q=3
    
    # Fill in known and predicted values
    # PQ buses
    v_mag[pq_mask] = pred[pq_mask, 0]  # Predicted. the pred[pq_mask,0] will return the predicted voltage magnitude for the PQ buses] 
    v_ang[pq_mask] = pred[pq_mask, 1]  # Predicted
    p[pq_mask] = data.x[pq_mask, 3]    # Known (from input)
    q[pq_mask] = data.x[pq_mask, 4]    # Known (from input)
    
    # PV buses
    v_mag[pv_mask] = data.x[pv_mask, 5]  # Known (from input)
    v_ang[pv_mask] = pred[pv_mask, 1]    # Predicted
    p[pv_mask] = data.x[pv_mask, 3]      # Known (from input)
    q[pv_mask] = pred[pv_mask, 3]        # Predicted
    
    # Slack bus
    v_mag[slack_mask] = data.x[slack_mask, 5]  # Known (from input)
    v_ang[slack_mask] = data.x[slack_mask, 6]  # Known (from input)
    p[slack_mask] = pred[slack_mask, 2]        # Predicted
    q[slack_mask] = pred[slack_mask, 3]        # Predicted
    

    # Calculate power flow using admittance matrix
    # Get admittance matrix from PyPSA
    Y_bus = get_ybus_from_pypsa(network)
    Y_bus_tensor = torch.tensor(Y_bus, dtype=torch.complex64) #here we are converting the Y_bus matrix to a complex tensor
    
    # Initialize power balance loss
    power_balance_loss = 0.0
    
    # Get batch information
    if hasattr(data, 'batch'):
        # For batched data
        num_graphs = data.batch.max().item() + 1
        num_nodes_per_graph = len(data.x) // num_graphs
        
        # Process each graph in the batch separately
        for graph_idx in range(num_graphs):
            # Create mask for this graph
            graph_mask = (data.batch == graph_idx)
            
            # Extract data for this graph only
            v_mag_graph = torch.zeros(num_nodes_per_graph)
            v_ang_graph = torch.zeros(num_nodes_per_graph)
            p_graph = torch.zeros(num_nodes_per_graph)
            q_graph = torch.zeros(num_nodes_per_graph)
            
            # Get node indices for this graph
            node_indices = torch.where(graph_mask)[0]
            
            # Fill in values for this graph
            for i, node_idx in enumerate(node_indices):
                # Get bus type masks for this node
                is_pq = data.pq_mask[node_idx]
                is_pv = data.pv_mask[node_idx]
                is_slack = data.slack_mask[node_idx]
                
                # Fill based on bus type 
                if is_pq:
                    v_mag_graph[i] = pred[node_idx, 0]
                    v_ang_graph[i] = pred[node_idx, 1]
                    p_graph[i] = data.x[node_idx, 3]
                    q_graph[i] = data.x[node_idx, 4]
                elif is_pv:
                    v_mag_graph[i] = data.x[node_idx, 5]
                    v_ang_graph[i] = pred[node_idx, 1]
                    p_graph[i] = data.x[node_idx, 3]
                    q_graph[i] = pred[node_idx, 3]
                elif is_slack:
                    v_mag_graph[i] = data.x[node_idx, 5]
                    v_ang_graph[i] = data.x[node_idx, 6]
                    p_graph[i] = pred[node_idx, 2]
                    q_graph[i] = pred[node_idx, 3]
            
            # Calculate power flow for this graph

            
            v_complex_graph = v_mag_graph * torch.exp(1j * v_ang_graph)# [STSI 24.09.25]: Corrected degree to radians conversion here. As angles are already in radians, no need to convert
            S_calc_graph = v_complex_graph * torch.conj(torch.matmul(Y_bus_tensor, v_complex_graph))
            p_calc_graph = torch.real(S_calc_graph)
            
            q_calc_graph = torch.imag(S_calc_graph)
            
            # Add to power balance loss
            power_balance_loss += F.mse_loss(p_calc_graph, p_graph) + F.mse_loss(q_calc_graph, q_graph)
            # Only debug first graph to avoid spam and print only every tenth epoch
        #    if graph_idx == 0 and (snapshot_idx % 10 == 0):  
        #        print(f"Physics Debug (Graph {graph_idx}):")
        #        print(f"  P calc vs target: {torch.mean(torch.abs(p_calc_graph - p_graph)):.6f}")
        #        print(f"  Q calc vs target: {torch.mean(torch.abs(q_calc_graph - q_graph)):.6f}")
        #        print(f"  Graph power balance loss: {F.mse_loss(p_calc_graph, p_graph) + F.mse_loss(q_calc_graph, q_graph):.6f}")
            
        # Average over all graphs in batch
        power_balance_loss /= num_graphs
    else:
        # Convert voltage to complex form
        
        v_complex = v_mag * torch.exp(1j * v_ang) # [STSI 24.09.25]: Corrected degree to radians conversion here. As angles are already in radians, no need to convert
        
        # Calculate power using S = V * (Y*V)* (complex power equation)
        S_calc = v_complex * torch.conj(torch.matmul(Y_bus_tensor, v_complex))
        p_calc = torch.real(S_calc)
        q_calc = torch.imag(S_calc)
        
        # Power balance loss
        power_balance_loss = F.mse_loss(p_calc, p) + F.mse_loss(q_calc, q)
        #print debug for evey tenth epoch
        #if snapshot_idx % 10 == 0:
        #    print(f"Physics Debug:")
        #    print(f"  P calc vs target: {torch.mean(torch.abs(p_calc - p)):.6f}")
        #    print(f"  Q calc vs target: {torch.mean(torch.abs(q_calc - q)):.6f}")
        #    print(f"  Power balance loss: {power_balance_loss:.6f}")
        #    print(f"  MSE loss: {mse_loss:.6f}")
    # Total loss (weighted sum)
    if loss_fraction_physics > 0 and loss_fraction_physics < 1:
        total_loss = loss_fraction_physics * power_balance_loss + (1 - loss_fraction_physics) *mse_loss     
    else:
        total_loss = mse_loss
    #total_loss = mse_loss
    #total_loss = 0.7 * mse_loss + 0.3 * power_balance_loss
    #total_loss = 0.3 * mse_loss + 0.7 * power_balance_loss
    #total_loss = power_balance_loss
    # In physics_informed_loss, add detailed debugging:



    
    return total_loss, mse_loss, power_balance_loss 
"""


In [ ]:
def physics_informed_loss_single_graph(pred, target, batch, network, Y_matrix=None, loss_fraction_physics=0.01):
    """
    Physics-informed loss for a SINGLE graph (batch_size=1).
    Much simpler than batched version!
    """
    #[STSI 08.02.26]: Removed undefined net_idx from debug logging
    logger.debug(f"Processing single-graph loss on network with {len(network.buses)} buses")

    # MSE loss on predictions
    mse_loss = F.mse_loss(pred, target)

    if loss_fraction_physics == 0:
        return mse_loss, mse_loss, torch.tensor(0.0, device=pred.device)

    if Y_matrix is None:
        Y_matrix = compute_admittance_matrix(network, device=pred.device)

    bus_masks = (batch.slack_mask, batch.pv_mask, batch.pq_mask)

    physics_loss = compute_power_flow_residual_from_pred(
        pred,
        batch.x,
        Y_matrix,
        network,
        bus_masks=bus_masks
    )

    total_loss = mse_loss + loss_fraction_physics * physics_loss
    return total_loss, mse_loss, physics_loss


def physics_informed_loss_batch(pred, target, batch, networks, Y_cache, loss_fraction_physics=0.01):
#[STSI 08.02.26]: Simplified batched physics loss, loops over graphs and uses physics_informed_loss_single_graph
    """
    Physics-informed loss with topology awareness for batched graphs.

    Args:
        pred: Predicted values [num_nodes_total, 4]
        target: Target values [num_nodes_total, 4]
        batch: PyG Batch object with .batch (graph ids) and .network_idx per-graph
        networks: List of networks for this split (train/val/test)
        Y_cache: List/array of precomputed Y-matrices aligned with networks
        loss_fraction_physics: Weight for physics loss

    Returns:
        total_loss, mse_loss, physics_loss
    """

    # Global MSE over all nodes in batch
    mse_loss = F.mse_loss(pred, target)

    graph_ids = batch.batch  # shape [num_nodes_total], values 0..num_graphs-1
    num_graphs = int(graph_ids.max().item()) + 1

    total_phys = 0.0
    total_nodes = 0

    for g in range(num_graphs):
        node_mask = (graph_ids == g)
        if not node_mask.any():
            continue

        node_indices = torch.where(node_mask)[0]

        # [STSI 08.02.26]: network_idx is stored per-graph; pick from this graph
        if batch.network_idx.dim() == 0:
            net_idx = int(batch.network_idx.item())
        else:
            net_idx = int(batch.network_idx[g].item())

        network = networks[net_idx]
        Y_matrix = Y_cache[net_idx]

        pred_g = pred[node_mask]
        target_g = target[node_mask]
        x_g = batch.x[node_mask]

        slack_mask_g = batch.slack_mask[node_mask]
        pv_mask_g = batch.pv_mask[node_mask]
        pq_mask_g = batch.pq_mask[node_mask]
        bus_masks = (slack_mask_g, pv_mask_g, pq_mask_g)

        physics_residual = compute_power_flow_residual_from_pred(
            pred_g,
            x_g,
            Y_matrix,
            network,
            bus_masks=bus_masks
        )

        num_nodes_g = node_mask.sum().item()
        total_phys += physics_residual * num_nodes_g
        total_nodes += num_nodes_g

    if total_nodes > 0:
        physics_loss = total_phys / total_nodes
    else:
        physics_loss = torch.tensor(0.0, device=pred.device)

    total_loss = mse_loss + loss_fraction_physics * physics_loss
    return total_loss, mse_loss, physics_loss

def compute_power_flow_residual_from_pred(pred, x, Y_matrix, network, bus_masks=None):
    """
    Compute power flow residual from predictions with proper bus-type handling.
    
    Physics equations must be satisfied:
    - At PQ buses: Use specified P,Q → Check if calculated V matches
    - At PV buses: Use specified P, predicted Q → Check if calculated V matches  
    - At Slack bus: Use specified V → Check if calculated P,Q matches
    
    Args:
        pred: Predictions [num_nodes, 4] - [v_mag, v_ang, p, q]
        x: Node features [num_nodes, feature_dim] - [bus_type(3), p, q, v_mag, v_ang]
        Y_matrix: Tuple of (Y_real, Y_imag)
        network: PyPSA network object
        bus_masks: Tuple of (slack_mask, pv_mask, pq_mask) if available from batch
        
    Returns:
        physics_loss: Scalar tensor
    """

    Y_real, Y_imag = Y_matrix
    num_nodes = pred.size(0)
    device = pred.device
    
    logger.debug(f"num_nodes in batch: {num_nodes}")
    logger.debug(f"Y_real shape: {Y_real.shape}")
    logger.debug(f"Y_imag shape: {Y_imag.shape}")
    logger.debug(f"network has {len(network.buses)} buses")
    # Extract bus type masks from node features if not provided
    if bus_masks is None:
        slack_mask = x[:, 0] == 1.0  # First column is slack indicator
        pv_mask = x[:, 1] == 1.0     # Second column is PV indicator
        pq_mask = x[:, 2] == 1.0     # Third column is PQ indicator
    else:
        slack_mask, pv_mask, pq_mask = bus_masks
    
    # ============================================
    # CONSTRUCT P AND Q INJECTIONS FOR PHYSICS CHECK
    # ============================================
    
    # Initialize P and Q with zeros
    p_inj = torch.zeros(num_nodes, device=device)
    q_inj = torch.zeros(num_nodes, device=device)
    
    # For PQ buses: Use KNOWN P and Q from inputs (x[:, 3] and x[:, 4])
    # Note: input_feature_filter keeps these values, masks unknowns to 0
    p_inj[pq_mask] = x[pq_mask, 3]
    q_inj[pq_mask] = x[pq_mask, 4]
    
    # For PV buses: Use KNOWN P from inputs + PREDICTED Q from model
    p_inj[pv_mask] = x[pv_mask, 3]  # Known P (input)
    q_inj[pv_mask] = pred[pv_mask, 3]  # Predicted Q (model output)
    
    # For Slack bus: Use PREDICTED P and Q from model
    p_inj[slack_mask] = pred[slack_mask, 2]  # Predicted P
    q_inj[slack_mask] = pred[slack_mask, 3]  # Predicted Q
    
    # ============================================
    # CONSTRUCT VOLTAGE FOR PHYSICS CHECK
    # ============================================
    
    # Initialize voltage magnitude and angle
    v_mag = torch.zeros(num_nodes, device=device)
    v_ang = torch.zeros(num_nodes, device=device)
    
    # For PQ buses: Use PREDICTED V_mag and V_ang
    v_mag[pq_mask] = pred[pq_mask, 0]
    v_ang[pq_mask] = pred[pq_mask, 1]
    
    # For PV buses: Use KNOWN V_mag (input) + PREDICTED V_ang (model)
    v_mag[pv_mask] = x[pv_mask, 5]  # Known V_mag (input)
    v_ang[pv_mask] = pred[pv_mask, 1]  # Predicted V_ang (model output)
    
    # For Slack bus: Use KNOWN V_mag and V_ang from inputs
    v_mag[slack_mask] = x[slack_mask, 5]  # Known V_mag
    v_ang[slack_mask] = x[slack_mask, 6]  # Known V_ang
    
    # ============================================
    # COMPUTE POWER FLOW EQUATIONS
    # ============================================
    
    # Compute complex voltage
    v_real = v_mag * torch.cos(v_ang)
    v_imag = v_mag * torch.sin(v_ang)
    
    # Compute currents: I = Y * V
    I_real = torch.matmul(Y_real, v_real) - torch.matmul(Y_imag, v_imag)
    I_imag = torch.matmul(Y_real, v_imag) + torch.matmul(Y_imag, v_real)
    
    # Compute power from voltages: S = V * conj(I)
    p_calc = v_real * I_real + v_imag * I_imag
    q_calc = v_imag * I_real - v_real * I_imag
    
    # ============================================
    # COMPUTE RESIDUALS (Power Balance Check)
    # ============================================
    
    # Power flow equations: Calculated power should match injections
    # This is the fundamental physics constraint
    p_residual = (p_calc - p_inj) ** 2
    q_residual = (q_calc - q_inj) ** 2
    
    # Combined residual (mean over all buses)
    physics_loss = torch.mean(p_residual + q_residual)
    
    return physics_loss



def compute_admittance_matrix(network, device='cpu', return_format='torch'):
    """
    Compute admittance matrix Y for a PyPSA network with robust topology handling.
    
    Improvements over original:
    - Handles arbitrary bus naming/ordering (topology-safe)
    - GPU-capable PyTorch tensors
    - Optional complex or split format
    - Maintains shunt admittance accuracy
    
    Args:
        network: PyPSA network object
        device: Device for PyTorch tensors ('cpu' or 'cuda')
        return_format: 'torch' (real/imag split), 'numpy' (complex), or 'complex_torch'
        
    Returns:
        If return_format='torch': Tuple of (Y_real, Y_imag) as PyTorch tensors
        If return_format='numpy': Complex numpy array
        If return_format='complex_torch': Complex PyTorch tensor
    """
    num_buses = len(network.buses)
    
    # Build robust bus name to index mapping 
    bus_to_idx = {bus_name: idx for idx, bus_name in enumerate(network.buses.index)}
    
    # Initialize as complex numpy array 
    Y = np.zeros((num_buses, num_buses), dtype=complex)
    
    # ============================================
    # PROCESS LINES
    # ============================================
    for line_name, line in network.lines.iterrows():
        # Use mapping instead of string parsing (topology-safe)
        i = bus_to_idx[line.bus0]
        j = bus_to_idx[line.bus1]
        
        # Line parameters
        r = line.r
        x = line.x
        b = line.b
        
        # Series admittance: y = 1/z where z = r + jx
        z = r + 1j * x
        if abs(z) > 1e-10:
            y_series = 1.0 / z
        else:
            y_series = 0.0
            if abs(z) > 0:
                logger.warning(f"Line {line_name} has very small impedance z={z}")
        
        # Shunt admittance (capacitance)
        y_shunt = 1j * b
        
        # Build Y matrix
        # Diagonal: self-admittance
        Y[i, i] += y_series + y_shunt / 2
        Y[j, j] += y_series + y_shunt / 2
        
        # Off-diagonal: mutual admittance
        Y[i, j] -= y_series
        Y[j, i] -= y_series
    
    # ============================================
    # PROCESS TRANSFORMERS
    # ============================================
    for trans_name, trans in network.transformers.iterrows():
        i = bus_to_idx[trans.bus0]
        j = bus_to_idx[trans.bus1]
        
        # Transformer parameters
        r = trans.r
        x = trans.x
        b = getattr(trans, 'b', 0.0)  # Transformers may have shunt admittance
        tap = getattr(trans, 'tap_ratio', 1.0)
        
        # Series admittance
        z = r + 1j * x
        if abs(z) > 1e-10:
            y_series = 1.0 / z
        else:
            y_series = 0.0
            logger.warning(f"Transformer {trans_name} has very small impedance z={z}")
        
        # Shunt admittance
        y_shunt = 1j * b
        
        # Transformer model with tap ratio
        # Primary side (bus i)
        Y[i, i] += y_series / (tap ** 2) + y_shunt / 2
        
        # Secondary side (bus j)
        Y[j, j] += y_series + y_shunt / 2
        
        # Mutual admittance (accounts for tap ratio)
        Y[i, j] -= y_series / tap
        Y[j, i] -= y_series / tap
    
    # ============================================
    # RETURN IN REQUESTED FORMAT
    # ============================================
    if return_format == 'numpy':
        return Y
    
    elif return_format == 'torch':
        # Split into real and imaginary parts for PyTorch
        Y_real = torch.FloatTensor(Y.real).to(device)
        Y_imag = torch.FloatTensor(Y.imag).to(device)
        return (Y_real, Y_imag)
    
    elif return_format == 'complex_torch':
        # PyTorch complex tensor (requires PyTorch >= 1.10)
        Y_complex = torch.tensor(Y, dtype=torch.complex64, device=device)
        return Y_complex
    
    else:
        raise ValueError(f"Unknown return_format: {return_format}")


# ============================================
# HELPER: Convert between formats
# ============================================
def ybus_to_torch_split(Y_complex, device='cpu'):
    """Convert complex numpy Y-bus to split PyTorch tensors"""
    Y_real = torch.FloatTensor(Y_complex.real).to(device)
    Y_imag = torch.FloatTensor(Y_complex.imag).to(device)
    return (Y_real, Y_imag)


def ybus_from_torch_split(Y_real, Y_imag):
    """Convert split PyTorch tensors back to complex numpy"""
    return Y_real.cpu().numpy() + 1j * Y_imag.cpu().numpy()

def precompute_Y_matrices(networks, device='cpu'):
    """
    Precompute and cache Y-matrices for all networks in a list.
    
    This is safe because:
    - Each network in the list gets a unique index (0, 1, 2, ...)
    - The dataset's network_idx directly maps to this list
    - No mixing possible between train/val/test splits
    
    Args:
        networks: List of PyPSA networks
        device: Device to store tensors
        
    Returns:
        Dict[int, Tuple]: Dictionary mapping network index to (Y_real, Y_imag)
    """
    Y_cache = {}
    for i, network in enumerate(networks):
        Y_cache[i] = compute_admittance_matrix(network, device=device)
        # Verify size
        Y_real, Y_imag = Y_cache[i]
        num_buses = len(network.buses)
        assert Y_real.size(0) == num_buses, f"Network {i}: Y-matrix size mismatch"
        
        logger.debug(f"Cached Y-matrix for network {i}: {num_buses}x{num_buses} buses")
    logger.info(f"Cached {len(Y_cache)} Y-matrices")
    return Y_cache



In [ ]:
def get_ybus_from_pypsa(network, snapshot=None):
    """
    Extract the admittance matrix (Y-bus) from a PyPSA network
    
    Args:
        network: PyPSA network object
        snapshot: Optional snapshot to use for time-dependent parameters
    
    Returns:
        Y_bus: Complex numpy array representing the admittance matrix
    """

    
    # Get number of buses
    n_buses = len(network.buses)
    
    # Initialize Y-bus matrix as sparse matrix
    Y_bus = lil_matrix((n_buses, n_buses), dtype=complex)
    
    # Process lines
    for line_idx, line in network.lines.iterrows():
        # Get bus indices (convert from 1-indexed to 0-indexed)
        from_bus = int(line['bus0'].split()[-1]) - 1
        to_bus = int(line['bus1'].split()[-1]) - 1
        
        # Get line parameters
        r = line['r']
        x = line['x']
        b = line['b']
        
        # Calculate admittance
        y_series = 1 / complex(r, x)
        y_shunt = complex(0, b/2)
        
        # Add to Y-bus matrix
        # Diagonal elements
        Y_bus[from_bus, from_bus] += y_series + y_shunt
        Y_bus[to_bus, to_bus] += y_series + y_shunt
        
        # Off-diagonal elements
        Y_bus[from_bus, to_bus] -= y_series
        Y_bus[to_bus, from_bus] -= y_series
    
    # Process transformers if any
    for trafo_idx, trafo in network.transformers.iterrows():
        # Get bus indices
        from_bus = int(trafo['bus0'].split()[-1]) - 1
        to_bus = int(trafo['bus1'].split()[-1]) - 1
        
        # Get transformer parameters
        r = trafo['r']
        x = trafo['x']
        tap = trafo.get('tap_ratio', 1.0)
        
        # Calculate admittance
        y_series = 1 / complex(r, x)
        
        # Add to Y-bus matrix with tap ratio consideration
        # Diagonal elements
        Y_bus[from_bus, from_bus] += y_series / (tap**2)
        Y_bus[to_bus, to_bus] += y_series
        
        # Off-diagonal elements
        Y_bus[from_bus, to_bus] -= y_series / tap
        Y_bus[to_bus, from_bus] -= y_series / tap
    
    # Convert to dense numpy array for easier use with PyTorch
    Y_bus = Y_bus.toarray()
    
    return Y_bus


In [ ]:

def generate_training_data_with_topology_old(
    num_scenarios: int = 100,
    steps_per_scenario: int = 15,
    
    # Generator setpoint base values and ranges
    p_set_base: Dict[str, float] = None,
    volatility_range: float = 0.5,
    
    # Topology variation parameters
    include_topology_variants: bool = True,
    gen_bus_options: List[List[int]] = None,  # User provides this - any positions allowed
    load_bus_options: List[List[int]] = None,
    
    # Line modification probability
    line_modification_prob: float = 0.3,
    line_param_variation: float = 0.15,
    
    # Extended topology
    enable_extended_topology: bool = False,
    extended_bus_prob: float = 0.2,
    
    # System parameters
    sbase: float = 1.0,
    seed: Optional[int] = None,
    verbose: bool = True
) -> List[pypsa.Network]:
    """Generate diverse training data with topology variations for GNN power flow."""
    
    # Initialize RNG
    if seed is not None:
        rng = np.random.default_rng(seed)
    else:
        rng = np.random.default_rng()
    
    # Set defaults
    if p_set_base is None:
        p_set_base = {'gen2': 1.63, 'gen3': 0.70}
    
    # Simple defaults - if user doesn't provide, use original topology
    if gen_bus_options is None:
        gen_bus_options = [[1, 2, 3]]  # Just use original if not specified
    
    if load_bus_options is None:
        load_bus_options = [[5, 7, 9]]  # Just use original if not specified
    
    # Line names for potential modification (from base IEEE 9-bus)
    modifiable_lines = ['Line 2', 'Line 3', 'Line 5', 'Line 6', 'Line 8', 'Line 9']
    modifiable_transformers = ['Transformer 1', 'Transformer 4', 'Transformer 7']
    
    networks = []
    topology_stats = {
        'gen_configs': {},
        'load_configs': {},
        'extended_count': 0,
        'line_modified_count': 0
    }
    
    if verbose:
        logger.info(f"Generating {num_scenarios} scenarios with topology variations...")
        logger.info(f"Generator options: {len(gen_bus_options)} configurations")
        logger.info(f"Load options: {len(load_bus_options)} configurations")
    
    for i in range(num_scenarios):
        try:
            # ================================================
            # 1. SELECT BASE TOPOLOGY
            # ================================================
            
            # Randomly select generator and load configurations
            gen_buses = list(gen_bus_options[rng.integers(0, len(gen_bus_options))])
            load_buses = list(load_bus_options[rng.integers(0, len(load_bus_options))])
            
            # Track topology distribution
            gen_key = str(gen_buses)
            load_key = str(load_buses)
            topology_stats['gen_configs'][gen_key] = topology_stats['gen_configs'].get(gen_key, 0) + 1
            topology_stats['load_configs'][load_key] = topology_stats['load_configs'].get(load_key, 0) + 1
            
            # ================================================
            # 2. EXTENDED TOPOLOGY (BEFORE creating network!)
            # ================================================

            additional_lines = None
            if enable_extended_topology and rng.random() < extended_bus_prob:
                num_extra_buses = rng.integers(1, 3)
                additional_lines = []
                
                # FIXED: Get all buses that will actually exist in the network
                # This includes: gen_buses, load_buses, and base IEEE buses 1-9
                all_existing_buses = set(range(1, 10))  # Base IEEE 9-bus always has 1-9
                all_existing_buses.update(gen_buses)
                all_existing_buses.update(load_buses)
                current_max_bus = max(all_existing_buses)
                
                # Convert to sorted list for easier selection
                existing_buses_list = sorted(list(all_existing_buses))                
                
                for extra in range(num_extra_buses):
                    new_bus = current_max_bus + 1 + extra
                    
                    # FIXED: Connect to existing buses (not hardcoded 4-9)
                    # Choose from buses that actually exist (1 to current_max_bus)
                    # Prefer connecting to non-generator buses (4 and above if available)
                    potential_connections = list(range(4, current_max_bus + 1))
                    
                    # If no buses >= 4 exist, use any existing bus except slack
                    if not potential_connections:
                        potential_connections = list(range(2, current_max_bus + 1))
                    
                    # If still nothing, use any existing bus
                    if not potential_connections:
                        potential_connections = list(range(1, current_max_bus + 1))
                    
                    connect_to = rng.choice(potential_connections)
                    
                    additional_lines.append({
                        "from": connect_to,
                        "to": new_bus,
                        "r": rng.uniform(0.008, 0.015),
                        "x": rng.uniform(0.06, 0.10),
                        "b": rng.uniform(0.12, 0.20),
                        "s_nom": rng.uniform(0.8, 1.2)
                    })
                    
                    if rng.random() < 0.7:
                        load_buses.append(new_bus)

                            # Update existing buses for next iteration
                    all_existing_buses.add(new_bus)
                    existing_buses_list.append(new_bus)
                    current_max_bus = new_bus
                topology_stats['extended_count'] += 1
            
            # ================================================
            # 3. RANDOMIZE GENERATOR SETPOINTS
            # ================================================
            
            p_set_generators = {}
            non_slack_gens = [b for b in gen_buses if b != gen_buses[0]]
            
            setpoint_keys = ['gen2', 'gen3']
            for idx, bus in enumerate(non_slack_gens):
                if idx < len(setpoint_keys):
                    base_val = p_set_base.get(setpoint_keys[idx], 0.5)
                else:
                    base_val = 0.5
                
                adjusted_p_set = rng.uniform(
                    base_val * (1 - volatility_range),
                    base_val * (1 + volatility_range)
                )
                p_set_generators[bus] = adjusted_p_set
            
            # ================================================
            # 4. RANDOMIZE LOAD VOLATILITY
            # ================================================
            
            load_volatility = rng.uniform(0.1, volatility_range)
            
            # ================================================
            # 5. LINE/TRANSFORMER MODIFICATIONS
            # ================================================
            
            line_modifications = None
            if rng.random() < line_modification_prob:
                line_modifications = {}
                
                for line_name in modifiable_lines:
                    if rng.random() < 0.5:
                        line_modifications[line_name] = {}
                        
                        if rng.random() < 0.7:
                            r_multiplier = rng.uniform(
                                1 - line_param_variation,
                                1 + line_param_variation
                            )
                            line_modifications[line_name]['r_mult'] = r_multiplier
                        
                        if rng.random() < 0.7:
                            x_multiplier = rng.uniform(
                                1 - line_param_variation,
                                1 + line_param_variation
                            )
                            line_modifications[line_name]['x_mult'] = x_multiplier
                
                for trans_name in modifiable_transformers:
                    if rng.random() < 0.3:
                        if trans_name not in line_modifications:
                            line_modifications[trans_name] = {}
                        
                        if rng.random() < 0.8:
                            x_multiplier = rng.uniform(
                                1 - line_param_variation,
                                1 + line_param_variation
                            )
                            line_modifications[trans_name]['x_mult'] = x_multiplier
                
                if line_modifications:
                    topology_stats['line_modified_count'] += 1
            
            line_mods_actual = _apply_line_multipliers(line_modifications)
            
            # ================================================
            # 6. CREATE NETWORK
            # ================================================
            
            network = create_9_bus_network_topology_variants(
                gen_buses=gen_buses,
                load_buses=load_buses,
                steps=steps_per_scenario,
                p_set_generators=p_set_generators,
                load_volatility=load_volatility,
                line_modifications=line_mods_actual,
                additional_lines=additional_lines,
                sbase=sbase,
                plot=False,
                seed=seed + i if seed is not None else None
            )
            
            # ================================================
            # 7. SOLVE POWER FLOW
            # ================================================
            
            network.pf(use_seed=True)
            
            # Check if power flow converged
            if not all(np.isfinite(network.buses_t.v_mag_pu.values.flatten())):
                logger.warning(f"Scenario {i+1}: Power flow did not converge. Skipping.")
                continue
            
            networks.append(network)
            
            # ================================================
            # 8. PROGRESS REPORTING
            # ================================================
            
            if verbose and (i + 1) % 10 == 0:
                logger.info(f"Generated {i+1}/{num_scenarios} networks")
                logger.info(f"  Gen buses: {gen_buses}, Load buses: {load_buses}")
                
                for bus, p_set in p_set_generators.items():
                    gen_idx = gen_buses.index(bus)
                    logger.info(f"  Gen {gen_idx+1} (Bus {bus}): p_set = {p_set:.3f} p.u.")
                
                if line_mods_actual:
                    logger.info(f"  Line modifications applied: {len(line_mods_actual)} components")
                
                if additional_lines:
                    logger.info(f"  Extended topology: {len(additional_lines)} additional lines")
        
        except Exception as e:
            import traceback
            logger.error(f"Scenario {i+1} failed with {type(e).__name__}: {str(e)}")
            logger.debug(f"Full traceback:\n{traceback.format_exc()}")
            continue

    
    # ================================================
    # FINAL STATISTICS
    # ================================================
    
    if verbose:
        logger.info(f"\n{'='*60}")
        logger.info(f"Training data generation complete!")
        logger.info(f"{'='*60}")
        logger.info(f"Successfully generated: {len(networks)}/{num_scenarios} networks")
        logger.info(f"Extended topologies: {topology_stats['extended_count']}")
        logger.info(f"Networks with line modifications: {topology_stats['line_modified_count']}")
        
        logger.info(f"\nGenerator configuration distribution:")
        for config, count in topology_stats['gen_configs'].items():
            logger.info(f"  {config}: {count} ({100*count/num_scenarios:.1f}%)")
        
        logger.info(f"\nLoad configuration distribution:")
        for config, count in topology_stats['load_configs'].items():
            logger.info(f"  {config}: {count} ({100*count/num_scenarios:.1f}%)")
    
    return networks



def generate_training_data(base_network, num_scenarios=100, steps_per_scenario=15,p_set_gen2=1.63, p_set_gen3 = 0.70, voilatility_range=0.5):
    """
    Generate multiple network scenarios by varying load and generation
    
    Args:
        base_network: Base PyPSA network
        num_scenarios: Number of different scenarios to generate
        steps_per_scenario: Number of snapshots per scenario
    
    Returns:
        List of PyPSA networks with solved power flow
    """
    networks = []
    
    for i in range(num_scenarios):
        # Create a new network with random variations based on the voilatility range
        #P_set is calculated as the base with var
        #chose random setpoint for generators based on voilatility range
        adjusted_p_set_gen2 = np.random.uniform(p_set_gen2*(1-voilatility_range),p_set_gen2*(1+voilatility_range))
        adjusted_p_set_gen3 = np.random.uniform(p_set_gen3*(1-voilatility_range),p_set_gen3*(1+voilatility_range))
        load_volatility = np.random.uniform(0.1, voilatility_range)

        
        # Create network with these parameters
        network = create_9_bus_network(
            steps=steps_per_scenario,
            p_set_gen2=adjusted_p_set_gen2,
            p_set_gen3=adjusted_p_set_gen3,
            load_voilatility=load_volatility,
            plot=False
        )
        
        # Solve power flow
        network.pf(use_seed=True)
        
        # Add to list
        networks.append(network)
        
        if i % 10 == 0:
            print(f"Generated {i+1}/{num_scenarios} networks")
            print(f"Gen set p for generator 2 = {adjusted_p_set_gen2}")
            print(f"Gen set p for generator 3 = {adjusted_p_set_gen3}")
    
    return networks

def generate_training_data_with_topology(
    num_scenarios: int = 100,
    steps_per_scenario: int = 15,
    max_attempts: int=None,
    
    # Generator setpoint base values and ranges
    p_set_base: Dict[str, float] = None,
    volatility_range: float = 0.5,
    
    # Topology variation parameters
    include_topology_variants: bool = True,
    gen_bus_options: List[List[int]] = None,
    load_bus_options: List[List[int]] = None,
    
    # Line modification probability
    line_modification_prob: float = 0.3,
    line_param_variation: float = 0.15,
    
    # Extended topology
    enable_extended_topology: bool = False,
    extended_bus_prob: float = 0.2,
    
    # System parameters
    sbase: float = 1.0,
    seed: Optional[int] = None,
    verbose: bool = True
) -> List[pypsa.Network]:
    """Generate diverse training data with topology variations for GNN power flow."""
    
    # Initialize RNG
    if seed is not None:
        rng = np.random.default_rng(seed)
    else:
        rng = np.random.default_rng()
    # Set max attempts (default: try up to 2x desired scenarios)
    if max_attempts is None:
        max_attempts = num_scenarios * 2
    # Set defaults
    if p_set_base is None:
        p_set_base = {'gen2': 1.63, 'gen3': 0.70}
    
    # Simple defaults - if user doesn't provide, use original topology
    if gen_bus_options is None:
        gen_bus_options = [[1, 2, 3]]
    
    if load_bus_options is None:
        load_bus_options = [[5, 7, 9]]
    
    # Line names for potential modification (from base IEEE 9-bus)
    modifiable_lines = ['Line 2', 'Line 3', 'Line 5', 'Line 6', 'Line 8', 'Line 9']
    modifiable_transformers = ['Transformer 1', 'Transformer 4', 'Transformer 7']
    
    networks = []
    topology_stats = {
        'gen_configs': {},
        'load_configs': {},
        'extended_count': 0,
        'line_modified_count': 0
    }
    
    # Track failure reasons
    failure_reasons = {
        'disconnected': 0,
        'invalid_params': 0,
        'solver_failed': 0,
        'no_convergence': 0,
        'nan_values': 0,
        'other': 0
    }
    
    if verbose:
        logger.info(f"Generating {num_scenarios} scenarios with topology variations...")
        logger.info(f"Generator options: {len(gen_bus_options)} configurations")
        logger.info(f"Load options: {len(load_bus_options)} configurations")
    attempt = 0
    # CHANGED: Loop until we have enough valid networks OR hit max attempts
    while len(networks) < num_scenarios and attempt < max_attempts:
        attempt += 1
        if attempt % 10 == 0 and verbose:
            print(f"Attempt {attempt}: Generated {len(networks)}/{num_scenarios} valid networks so far.")
        try:
            # ================================================
            # 1. SELECT BASE TOPOLOGY
            # ================================================
            
            gen_buses = list(gen_bus_options[rng.integers(0, len(gen_bus_options))])
            load_buses = list(load_bus_options[rng.integers(0, len(load_bus_options))])
            
            # Track topology distribution
            gen_key = str(gen_buses)
            load_key = str(load_buses)
            topology_stats['gen_configs'][gen_key] = topology_stats['gen_configs'].get(gen_key, 0) + 1
            topology_stats['load_configs'][load_key] = topology_stats['load_configs'].get(load_key, 0) + 1
            
            # ================================================
            # 2. EXTENDED TOPOLOGY
            # ================================================
            
            additional_lines = None
            if enable_extended_topology and rng.random() < extended_bus_prob:
                num_extra_buses = rng.integers(1, 3)
                additional_lines = []
                
                all_existing_buses = set(range(1, 10))
                all_existing_buses.update(gen_buses)
                all_existing_buses.update(load_buses)
                current_max_bus = max(all_existing_buses)
                
                existing_buses_list = sorted(list(all_existing_buses))
                
                for extra in range(num_extra_buses):
                    new_bus = current_max_bus + 1 + extra
                    
                    potential_connections = list(range(4, current_max_bus + 1))
                    
                    if not potential_connections:
                        potential_connections = list(range(2, current_max_bus + 1))
                    
                    if not potential_connections:
                        potential_connections = list(range(1, current_max_bus + 1))
                    
                    connect_to = rng.choice(potential_connections)
                    
                    additional_lines.append({
                        "from": connect_to,
                        "to": new_bus,
                        "r": rng.uniform(0.008, 0.015),
                        "x": rng.uniform(0.06, 0.10),
                        "b": rng.uniform(0.12, 0.20),
                        "s_nom": rng.uniform(0.8, 1.2)
                    })
                    
                    if rng.random() < 0.7:
                        load_buses.append(new_bus)
                    
                    all_existing_buses.add(new_bus)
                    existing_buses_list.append(new_bus)
                    current_max_bus = new_bus
                
                topology_stats['extended_count'] += 1
            
            # ================================================
            # 3. RANDOMIZE GENERATOR SETPOINTS
            # ================================================
            
            p_set_generators = {}
            non_slack_gens = [b for b in gen_buses if b != gen_buses[0]]
            
            setpoint_keys = ['gen2', 'gen3']
            for idx, bus in enumerate(non_slack_gens):
                if idx < len(setpoint_keys):
                    base_val = p_set_base.get(setpoint_keys[idx], 0.5)
                else:
                    base_val = 0.5
                
                adjusted_p_set = rng.uniform(
                    base_val * (1 - volatility_range),
                    base_val * (1 + volatility_range)
                )
                p_set_generators[bus] = adjusted_p_set
            
            # ================================================
            # 4. RANDOMIZE LOAD VOLATILITY
            # ================================================
            
            load_volatility = rng.uniform(0.1, volatility_range)
            
            # ================================================
            # 5. LINE/TRANSFORMER MODIFICATIONS
            # ================================================
            
            line_modifications = None
            if rng.random() < line_modification_prob:
                line_modifications = {}
                
                for line_name in modifiable_lines:
                    if rng.random() < 0.5:
                        line_modifications[line_name] = {}
                        
                        if rng.random() < 0.7:
                            r_multiplier = rng.uniform(
                                1 - line_param_variation,
                                1 + line_param_variation
                            )
                            line_modifications[line_name]['r_mult'] = r_multiplier
                        
                        if rng.random() < 0.7:
                            x_multiplier = rng.uniform(
                                1 - line_param_variation,
                                1 + line_param_variation
                            )
                            line_modifications[line_name]['x_mult'] = x_multiplier
                
                for trans_name in modifiable_transformers:
                    if rng.random() < 0.3:
                        if trans_name not in line_modifications:
                            line_modifications[trans_name] = {}
                        
                        if rng.random() < 0.8:
                            x_multiplier = rng.uniform(
                                1 - line_param_variation,
                                1 + line_param_variation
                            )
                            line_modifications[trans_name]['x_mult'] = x_multiplier
                
                if line_modifications:
                    topology_stats['line_modified_count'] += 1
            
            line_mods_actual = _apply_line_multipliers(line_modifications)
            
            # ================================================
            # 6. CREATE NETWORK
            # ================================================
            
            network = create_9_bus_network_topology_variants(
                gen_buses=gen_buses,
                load_buses=load_buses,
                steps=steps_per_scenario,
                p_set_generators=p_set_generators,
                load_volatility=load_volatility,
                line_modifications=line_mods_actual,
                additional_lines=additional_lines,
                sbase=sbase,
                plot=False,
                seed=seed + attempt if seed is not None else None
            )
            
            # ================================================
            # VALIDATION: Minimum loads required
            # ================================================
            if len(network.loads) < 2:
                if verbose and attempt % 20 == 0:
                    logger.debug(f"Attempt {attempt}: Insufficient loads ({len(network.loads)}). Skipping.")
                failure_reasons['insufficient_loads'] += 1
                continue
            
            # ================================================
            # VALIDATION: Check network connectivity
            # ================================================
            import networkx as nx
            G = nx.Graph()
            
            for idx, line in network.lines.iterrows():
                G.add_edge(line['bus0'], line['bus1'])
            for idx, trafo in network.transformers.iterrows():
                G.add_edge(trafo['bus0'], trafo['bus1'])
            
            if not nx.is_connected(G):
                if verbose and attempt % 20 == 0:
                    logger.debug(f"Attempt {attempt}: Network is disconnected. Skipping.")
                failure_reasons['disconnected'] += 1
                continue
            
            # ================================================
            # VALIDATION: Check line parameters
            # ================================================
            invalid_lines = []
            for idx, line in network.lines.iterrows():
                if line['r'] <= 0 or line['x'] <= 0:
                    invalid_lines.append(f"{idx}: r={line['r']:.6f}, x={line['x']:.6f}")
            
            for idx, trafo in network.transformers.iterrows():
                if trafo['x'] <= 0:
                    invalid_lines.append(f"{idx}: x={trafo['x']:.6f}")
            
            if invalid_lines:
                if verbose and attempt % 20 == 0:
                    logger.debug(f"Attempt {attempt}: Invalid line parameters. Skipping.")
                failure_reasons['invalid_params'] += 1
                continue
            
            # ================================================
            # 7. SOLVE POWER FLOW
            # ================================================
            original_level = logging.getLogger('pypsa').level
            logging.getLogger('pypsa').setLevel(logging.ERROR)
            pf_results = None
            pf_failed = False
            try:
                pf_results=network.pf(use_seed=True)
            except Exception as pf_error:
                pf_failed = True
                if verbose and attempt % 20 == 0:
                    logger.debug(f"Attempt {attempt}: Power flow solver failed: {type(pf_error).__name__}")
                failure_reasons['solver_failed'] += 1
            finally:
                logging.getLogger('pypsa').setLevel(original_level)
            
            if pf_failed:
                continue
            
            # Check convergence from the returned dictionary
            converged = False
            if pf_results is not None and 'converged' in pf_results:
                # pf_results['converged'] is a DataFrame with snapshots x sub_networks
                # Check if all snapshots converged for all sub-networks
                converged_df = pf_results['converged']
                if isinstance(converged_df, pd.DataFrame):
                    converged = converged_df.all().all()  # All sub-networks, all snapshots
                elif isinstance(converged_df, pd.Series):
                    converged = converged_df.all()
                else:
                    converged = bool(converged_df)

            if not converged:
                if verbose and attempt % 20 == 0:
                    logger.debug(f"Attempt {attempt}: Power flow did not converge")
                    if pf_results is not None:
                        logger.debug(f"  Convergence status: {pf_results.get('converged', 'N/A')}")
                        logger.debug(f"  Number of iterations: {pf_results.get('n_iter', 'N/A')}")
                        logger.debug(f"  Final error: {pf_results.get('error', 'N/A')}")
                failure_reasons['no_convergence'] += 1
                continue
            # After successful power flow:
            if converged:
                # Store pf_results for later inspection
                network.pf_results = pf_results
            # Check for NaN/Inf values
            if not all(np.isfinite(network.buses_t.v_mag_pu.values.flatten())):
                if verbose and attempt % 20 == 0:
                    logger.debug(f"Attempt {attempt}: Power flow resulted in NaN/Inf values.")
                failure_reasons['nan_values'] += 1
                continue
            
            # ================================================
            # SUCCESS - Add to networks list
            # ================================================
            networks.append(network)
            
            # ================================================
            # PROGRESS REPORTING
            # ================================================
            
            if verbose and len(networks) % 10 == 0:
                logger.info(f"Generated {len(networks)}/{num_scenarios} valid networks (attempt {attempt})")
                logger.info(f"  Success rate so far: {100*len(networks)/attempt:.1f}%")
        
        except Exception as e:
            import traceback
            logger.error(f"Attempt {attempt} failed with {type(e).__name__}: {str(e)}")
            if verbose:
                logger.debug(f"Full traceback:\n{traceback.format_exc()}")
            failure_reasons['other'] += 1
            continue
    
    # ================================================
    # FINAL STATISTICS
    # ================================================
    
    if verbose:
        logger.info(f"\n{'='*60}")
        logger.info(f"Training data generation complete!")
        logger.info(f"{'='*60}")
        logger.info(f"Successfully generated: {len(networks)}/{num_scenarios} networks")
        logger.info(f"Total attempts: {attempt}")
        logger.info(f"Overall success rate: {100*len(networks)/attempt:.1f}%")
        
        if len(networks) < num_scenarios:
            logger.warning(
                f"WARNING: Only generated {len(networks)}/{num_scenarios} requested networks "
                f"after {attempt} attempts. Consider adjusting parameters or increasing max_attempts."
            )
        
        # Failure breakdown
        total_failures = sum(failure_reasons.values())
        if total_failures > 0:
            logger.info(f"\nFailure breakdown ({total_failures} total):")
            for reason, count in sorted(failure_reasons.items(), key=lambda x: x[1], reverse=True):
                if count > 0:
                    logger.info(f"  {reason}: {count} ({100*count/total_failures:.1f}%)")
        
        logger.info(f"\nTopology statistics:")
        logger.info(f"  Extended topologies: {topology_stats['extended_count']}")
        logger.info(f"  Networks with line modifications: {topology_stats['line_modified_count']}")
        
        logger.info(f"\nGenerator configuration distribution:")
        for config, count in topology_stats['gen_configs'].items():
            logger.info(f"  {config}: {count} ({100*count/attempt:.1f}% of attempts)")
        
        logger.info(f"\nLoad configuration distribution:")
        for config, count in topology_stats['load_configs'].items():
            logger.info(f"  {config}: {count} ({100*count/attempt:.1f}% of attempts)")
    
    return networks


def _apply_line_multipliers(line_modifications: Optional[Dict]) -> Optional[Dict]:
    """
    Convert line parameter multipliers to actual values based on IEEE 9-bus base data.
    
    Parameters
    ----------
    line_modifications : Optional[Dict]
        Dictionary with multipliers for line parameters
        
    Returns
    -------
    Optional[Dict]
        Dictionary with actual parameter values
    """
    if line_modifications is None or not line_modifications:
        return None
    
    # Base IEEE 9-bus line data: {name: {'r': value, 'x': value, 's_nom': value}}
    base_line_params = {
        'Line 2': {'r': 0.0100, 'x': 0.0920, 's_nom': 1.00},
        'Line 3': {'r': 0.0390, 'x': 0.1700, 's_nom': 0.75},
        'Line 5': {'r': 0.0119, 'x': 0.1008, 's_nom': 1.00},
        'Line 6': {'r': 0.0085, 'x': 0.0720, 's_nom': 1.00},
        'Line 8': {'r': 0.0320, 'x': 0.1610, 's_nom': 1.00},
        'Line 9': {'r': 0.0100, 'x': 0.0850, 's_nom': 1.00},
        'Transformer 1': {'r': 0.0000, 'x': 0.0576, 's_nom': 1.50},
        'Transformer 4': {'r': 0.0000, 'x': 0.0586, 's_nom': 1.50},
        'Transformer 7': {'r': 0.0000, 'x': 0.0625, 's_nom': 1.75},
    }
    
    actual_mods = {}
    
    for component_name, mods in line_modifications.items():
        if component_name not in base_line_params:
            continue
        
        actual_mods[component_name] = {}
        base_params = base_line_params[component_name]
        
        # Apply multipliers to get actual values
        if 'r_mult' in mods:
            actual_mods[component_name]['r'] = base_params['r'] * mods['r_mult']
        
        if 'x_mult' in mods:
            actual_mods[component_name]['x'] = base_params['x'] * mods['x_mult']
        
        if 's_nom_mult' in mods:
            actual_mods[component_name]['s_nom'] = base_params['s_nom'] * mods['s_nom_mult']
    
    return actual_mods if actual_mods else None



def analyze_networks(networks: List[pypsa.Network]) -> pd.DataFrame:
    """
    Analyze a list of PyPSA networks and create a summary table.
    
    Parameters
    ----------
    networks : List[pypsa.Network]
        List of PyPSA network objects to analyze
        
    Returns
    -------
    pd.DataFrame
        Summary table with network properties
    """
    
    results = []
    
    for idx, network in enumerate(networks):
        # Basic counts
        num_buses = len(network.buses)
        num_lines = len(network.lines)
        num_transformers = len(network.transformers)
        num_generators = len(network.generators)
        num_loads = len(network.loads)
        
        # Generator buses
        gen_buses = sorted(network.generators['bus'].unique().tolist())
        gen_buses_str = ', '.join(map(str, gen_buses))
        
        # Load buses
        load_buses = sorted(network.loads['bus'].unique().tolist())
        load_buses_str = ', '.join(map(str, load_buses))
        
        # Line examples (first 2 lines)
        line_examples = []
        for i, (line_idx, line) in enumerate(network.lines.iterrows()):
            if i >= 2:
                break
            line_examples.append(
                f"{line_idx}: {line['bus0']}->{line['bus1']} "
                f"(r={line['r']:.4f}, x={line['x']:.4f})"
            )
        line_examples_str = ' | '.join(line_examples) if line_examples else "N/A"
        
        # Generator example (first generator)
        if len(network.generators) > 0:
            gen_idx, gen = list(network.generators.iterrows())[0]
            gen_example = (
                f"{gen_idx}: Bus {gen['bus']}, "
                f"P_nom={gen['p_nom']:.2f} MW, "
                f"Control={gen['control']}"
            )
        else:
            gen_example = "N/A"
        
        # Load example (first load)
        if len(network.loads) > 0:
            load_idx, load = list(network.loads.iterrows())[0]
            # Get first timestep load values
            p_set = network.loads_t.p_set[load_idx].iloc[0] if load_idx in network.loads_t.p_set.columns else 0
            q_set = network.loads_t.q_set[load_idx].iloc[0] if load_idx in network.loads_t.q_set.columns else 0
            load_example = (
                f"{load_idx}: Bus {load['bus']}, "
                f"P={p_set:.3f} MW, Q={q_set:.3f} MVAr"
            )
        else:
            load_example = "N/A"
        
        # Validity checks
        validity_checks = []
        
        # Check 1: Network connectivity
        G = nx.Graph()
        for _, line in network.lines.iterrows():
            G.add_edge(line['bus0'], line['bus1'])
        for _, trafo in network.transformers.iterrows():
            G.add_edge(trafo['bus0'], trafo['bus1'])
        
        is_connected = nx.is_connected(G) if len(G.nodes()) > 0 else False
        validity_checks.append(f"Connected: {'✓' if is_connected else '✗'}")
        
        # Check 2: Line parameters valid
        invalid_lines = 0
        for _, line in network.lines.iterrows():
            if line['r'] <= 0 or line['x'] <= 0:
                invalid_lines += 1
        for _, trafo in network.transformers.iterrows():
            if trafo['x'] <= 0:
                invalid_lines += 1
        validity_checks.append(f"Valid params: {'✓' if invalid_lines == 0 else f'✗ ({invalid_lines} invalid)'}")
        
        # Check 3: Power flow converged
        converged = False

        # Try to check from pf_results if they were stored
        if hasattr(network, 'pf_results') and network.pf_results is not None:
            if 'converged' in network.pf_results:
                converged_data = network.pf_results['converged']
                if isinstance(converged_data, pd.DataFrame):
                    converged = converged_data.all().all()
                elif isinstance(converged_data, pd.Series):
                    converged = converged_data.all()
                else:
                    converged = bool(converged_data)

        # Fallback: If pf_results not stored, check if we have valid voltage results
        if not converged and hasattr(network, 'buses_t') and hasattr(network.buses_t, 'v_mag_pu'):
            if len(network.buses_t.v_mag_pu) > 0:
                v_mag = network.buses_t.v_mag_pu.values.flatten()
                # If voltages are finite and reasonable, assume it converged
                if all(np.isfinite(v_mag)) and all((v_mag > 0.5) & (v_mag < 1.5)):
                    converged = True

        validity_checks.append(f"PF converged: {'✓' if converged else '✗'}")
        
        # Check 4: No NaN values
        has_valid_results = False
        if hasattr(network, 'buses_t') and hasattr(network.buses_t, 'v_mag_pu'):
            has_valid_results = all(np.isfinite(network.buses_t.v_mag_pu.values.flatten()))
        validity_checks.append(f"No NaN: {'✓' if has_valid_results else '✗'}")
        
        # Check 5: Sufficient loads
        sufficient_loads = num_loads >= 2
        validity_checks.append(f"Min loads (≥2): {'✓' if sufficient_loads else '✗'}")
        
        validity_str = ' | '.join(validity_checks)
        
        # Compile results
        results.append({
            'Network_Idx': idx,
            'Buses': num_buses,
            'Lines': num_lines,
            'Transformers': num_transformers,
            'Generators': num_generators,
            'Loads': num_loads,
            'Gen_Buses': gen_buses_str,
            'Load_Buses': load_buses_str,
            'Line_Examples': line_examples_str,
            'Gen_Example': gen_example,
            'Load_Example': load_example,
            'Validity_Checks': validity_str
        })
    
    # Create DataFrame
    df = pd.DataFrame(results)
    
    return df


def print_network_summary(networks: List[pypsa.Network], 
                         max_rows: int = None,
                         save_to_csv: str = None):
    """
    Print a formatted summary of networks and optionally save to CSV.
    
    Parameters
    ----------
    networks : List[pypsa.Network]
        List of networks to analyze
    max_rows : int, optional
        Maximum number of rows to display (None = all)
    save_to_csv : str, optional
        Filename to save CSV (e.g., 'network_summary.csv')
    """
    
    # Analyze networks
    df = analyze_networks(networks)
    
    # Print summary statistics
    print(f"\n{'='*80}")
    print(f"NETWORK COLLECTION SUMMARY")
    print(f"{'='*80}")
    print(f"Total networks: {len(networks)}")
    print(f"Buses range: {df['Buses'].min()}-{df['Buses'].max()}")
    print(f"Lines range: {df['Lines'].min()}-{df['Lines'].max()}")
    print(f"Generators range: {df['Generators'].min()}-{df['Generators'].max()}")
    print(f"Loads range: {df['Loads'].min()}-{df['Loads'].max()}")
    print(f"{'='*80}\n")
    
    # Display table
    if max_rows is not None:
        print(f"Showing first {max_rows} networks:\n")
        display_df = df.head(max_rows)
    else:
        display_df = df
    
    # Set pandas display options for better formatting
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 50)
    
    print(display_df.to_string(index=False))
    
    # Save to CSV if requested
    if save_to_csv:
        df.to_csv(save_to_csv, index=False)
        print(f"\n✓ Saved full summary to: {save_to_csv}")
    
    return df



In [ ]:

def evaluate_model(model, network, snapshot_idx=0, use_edge_features=True):
    """
    Evaluate model on a specific network and snapshot
    #[STSI26.02.06]: Extended to include per-bus and per-line detailed results
    #[STSI26.02.10]: Extended to report separate timing for
    #                (a) pure GNN inference and
    #                (b) data construction + inference.
    Args:
        model: Trained PowerFlowGNN model
        network: PyPSA network to evaluate on
        snapshot_idx: Snapshot index to evaluate
    
    Returns:
        Dictionary of metrics and predictions
    """
     # [STSI26.02.10]: Time data construction + inference + line flow calcuations together
    t0 = time.time()
    # Create dataset from this network
    dataset = PowerFlowDataset([network], use_edge_features=use_edge_features)
    # Get data for the specific snapshot
    data = dataset[snapshot_idx]
    
    t_data = time.time() - t0  # time to build dataset + index one sample
    # [STSI26.02.10]: Time pure GNN forward separately on same Data
    # Make prediction
    model.eval() # set model to evaluation mode

    with torch.no_grad():# no gradient calculation for evaluation
        t1= time.time() # start time for inference
        pred = model(data) # get the prediction from the model based on the data for the specific snapshot
        t_forward = time.time() - t1 # time for pure GNN forward pass
    
    # Extract true values from network
    true_v_mag = network.buses_t.v_mag_pu.iloc[snapshot_idx].values
    true_v_ang = network.buses_t.v_ang.iloc[snapshot_idx].values
    true_p = network.buses_t.p.iloc[snapshot_idx].values
    true_q = network.buses_t.q.iloc[snapshot_idx].values
    
    # Extract predicted values
    pred_v_mag = torch.zeros_like(torch.tensor(true_v_mag, dtype=torch.float)) # initializing tensors for predicted values
    pred_v_ang = torch.zeros_like(torch.tensor(true_v_ang, dtype=torch.float)) # initializing tensors for predicted values
    pred_p = torch.zeros_like(torch.tensor(true_p, dtype=torch.float)) # initializing tensors for predicted values
    pred_q = torch.zeros_like(torch.tensor(true_q, dtype=torch.float)) # initializing tensors for predicted values
    
    #[STSI26.02.06]: Track per-bus errors by bus name/index (not aggregated by type)
    per_bus_results = {
        'bus_name': [],
        'bus_idx': [],
        'bus_type': [],
        'true_v_mag': [],
        'pred_v_mag': [],
        'true_v_ang': [],
        'pred_v_ang': [],
        'true_p': [],
        'pred_p': [],
        'true_q': [],
        'pred_q': [],
        'v_mag_error': [],
        'v_ang_error': [],
        'p_error': [],
        'q_error': []
    }
    
    # Fill in predictions based on bus type
    for i, bus in enumerate(network.buses.index):
        bus_type = network.buses.loc[bus, 'type']
        
        if bus_type == 'PQ':
            # For PQ buses, predict v_mag, v_ang
            pred_v_mag[i] = pred[i, 0]
            pred_v_ang[i] = pred[i, 1]
            pred_p[i] = true_p[i]  # Known
            pred_q[i] = true_q[i]  # Known
        elif bus_type == 'PV':
            # For PV buses, predict v_ang, q
            pred_v_mag[i] = true_v_mag[i]  # Known
            pred_v_ang[i] = pred[i, 1]
            pred_p[i] = true_p[i]  # Known
            pred_q[i] = pred[i, 3]
        else:  # Slack
            # For slack bus, predict p, q
            pred_v_mag[i] = true_v_mag[i]  # Known
            pred_v_ang[i] = true_v_ang[i]  # Known
            pred_p[i] = pred[i, 2]
            pred_q[i] = pred[i, 3]
        
        #[STSI26.02.06]: Store per-bus predictions and errors (individual buses)
        per_bus_results['bus_name'].append(bus)
        per_bus_results['bus_idx'].append(i)
        per_bus_results['bus_type'].append(bus_type)
        per_bus_results['true_v_mag'].append(true_v_mag[i])
        per_bus_results['pred_v_mag'].append(pred_v_mag[i].item())
        per_bus_results['true_v_ang'].append(true_v_ang[i])
        per_bus_results['pred_v_ang'].append(pred_v_ang[i].item())
        per_bus_results['true_p'].append(true_p[i])
        per_bus_results['pred_p'].append(pred_p[i].item())
        per_bus_results['true_q'].append(true_q[i])
        per_bus_results['pred_q'].append(pred_q[i].item())
        per_bus_results['v_mag_error'].append(abs(pred_v_mag[i].item() - true_v_mag[i]))
        per_bus_results['v_ang_error'].append(abs(pred_v_ang[i].item() - true_v_ang[i]))
        per_bus_results['p_error'].append(abs(pred_p[i].item() - true_p[i]))
        per_bus_results['q_error'].append(abs(pred_q[i].item() - true_q[i]))
    
    
    # Calculate line flows using predicted values
    t2= time.time() # start time for line flow calculations
    pred_line_flows = calculate_line_flows(network, pred_v_mag.numpy(), pred_v_ang.numpy(), snapshot_idx)
    t_line=time.time() - t2 # time for line flow calculations
    
    true_line_flows = {
        'p0': network.lines_t.p0.iloc[snapshot_idx].values,
        'p1': network.lines_t.p1.iloc[snapshot_idx].values,
        'q0': network.lines_t.q0.iloc[snapshot_idx].values,
        'q1': network.lines_t.q1.iloc[snapshot_idx].values
        }
    #[STSI26.02.06]: Store per-line predictions and errors (individual lines)
    per_line_results = {
        'line_name': [],
        'line_idx': [],
        'bus0': [],
        'bus1': [],
        'true_p0': [],
        'pred_p0': [],
        'true_p1': [],
        'pred_p1': [],
        'true_q0': [],
        'pred_q0': [],
        'true_q1': [],
        'pred_q1': [],
        'p0_error': [],
        'p1_error': [],
        'q0_error': [],
        'q1_error': []
    }
    
    for i, line in enumerate(network.lines.index):
        per_line_results['line_name'].append(line)
        per_line_results['line_idx'].append(i)
        per_line_results['bus0'].append(network.lines.loc[line, 'bus0'])
        per_line_results['bus1'].append(network.lines.loc[line, 'bus1'])
        per_line_results['true_p0'].append(true_line_flows['p0'][i])
        per_line_results['pred_p0'].append(pred_line_flows['p0'][i])
        per_line_results['true_p1'].append(true_line_flows['p1'][i])
        per_line_results['pred_p1'].append(pred_line_flows['p1'][i])
        per_line_results['true_q0'].append(true_line_flows['q0'][i])
        per_line_results['pred_q0'].append(pred_line_flows['q0'][i])
        per_line_results['true_q1'].append(true_line_flows['q1'][i])
        per_line_results['pred_q1'].append(pred_line_flows['q1'][i])
        per_line_results['p0_error'].append(abs(pred_line_flows['p0'][i] - true_line_flows['p0'][i]))
        per_line_results['p1_error'].append(abs(pred_line_flows['p1'][i] - true_line_flows['p1'][i]))
        per_line_results['q0_error'].append(abs(pred_line_flows['q0'][i] - true_line_flows['q0'][i]))
        per_line_results['q1_error'].append(abs(pred_line_flows['q1'][i] - true_line_flows['q1'][i]))
    
    t_pipeline=t_data + t_forward + t_line # total time for data construction + inference + line flow calculations


    # Calculate metrics (Not included in the timing)
    v_mag_mae = torch.mean(torch.abs(pred_v_mag - torch.tensor(true_v_mag, dtype=torch.float)))
    v_ang_rmse = torch.sqrt(torch.mean((pred_v_ang - torch.tensor(true_v_ang, dtype=torch.float))**2))
    p_mae = torch.mean(torch.abs(pred_p - torch.tensor(true_p, dtype=torch.float)))
    q_mae = torch.mean(torch.abs(pred_q - torch.tensor(true_q, dtype=torch.float)))

    # Line flow metrics
    p_flow_mae = np.mean(np.abs(pred_line_flows['p0'] - true_line_flows['p0']))
    q_flow_mae = np.mean(np.abs(pred_line_flows['q0'] - true_line_flows['q0']))
    
    # Return metrics and predictions
    return {
        'metrics': {
            'v_mag_mae': v_mag_mae.item(),
            'v_ang_rmse': v_ang_rmse.item(),
            'p_mae': p_mae.item(),
            'q_mae': q_mae.item(),
            'p_flow_mae': p_flow_mae,
            'q_flow_mae': q_flow_mae
        },
        'predictions': {
            'v_mag': pred_v_mag.numpy(),
            'v_ang': pred_v_ang.numpy(),
            'p': pred_p.numpy(),
            'q': pred_q.numpy(),
            'line_flows': pred_line_flows
        },
        'true_values': {
            'v_mag': true_v_mag,
            'v_ang': true_v_ang,
            'p': true_p,
            'q': true_q,
            'line_flows': true_line_flows
        },
        'per_bus_results': per_bus_results,  #[STSI26.02.06]: Changed from per_bus_errors to per_bus_results
        'per_line_results': per_line_results,  #[STSI26.02.06]: Changed from per_line_errors to per_line_results
        'timing': {
            'data_only': t_data,            #[STSI26.02.10]: Added to time the data construction GNN pipeline for comparison with conventional
            'forward_only': t_forward,      #[STSI26.02.10]: Added to time the prediction part of GNN pipeline for comparison with conventional
            'line_flows_only': t_line,       #[STSI26.02.10]: Added to time the line flow of GNN pipeline for comparison with conventional
            'pipeline_total': t_pipeline    #[STSI26.02.10]: Added to time the complete GNN pipeline for comparison with conventional
        }
    }



#prediction function to predict node values and line flows for comparison with true values

#prediction function to predict node values and line flows for comparison with true values
def test_model_on_networks(model, test_networks, num_snapshots_per_network=None, compare_with_pypsa=True, use_edge_features=True, max_n_test=None):
    """
    Comprehensive testing using evaluate_model on all test networks
    #[STSI26.02.06]: Created to provide detailed per-network testing metrics
    #[STSI26.02.06]: Extended to include per-bus, per-line analysis and execution time comparison
    #[STSI26.02.06]: Changed to store per-bus and per-line individual results (not aggregated by type)
    #[STSI26.02.10]: Uses detailed timing from evaluate_model:
    #                - data_only, forward_only, line_flows_only, pipeline_total
    #                - Added max_n_test to limit number of test networks.
    Args:
        model: Trained PowerFlowGNN model
        test_networks: List of PyPSA networks for testing
        num_snapshots_per_network: Number of snapshots to test per network
                                   If None, tests all snapshots
        compare_with_pypsa: Whether to compare execution time with PyPSA power flow
    
    Returns:
        Dictionary with aggregated metrics, per-network results, and timing comparison
    """

    # [STSI26.02.10]: Optionally limit number of networks tested
    if max_n_test is not None:
        #shuffle networks to get a random subset if max_n_test is less than total
        if max_n_test < len(test_networks):
            np.random.shuffle(test_networks)
        test_networks = test_networks[:max_n_test]  

    all_results = []
    aggregate_metrics = {
        'v_mag_mae': [],
        'v_ang_rmse': [],
        'p_mae': [],
        'q_mae': [],
        'p_flow_mae': [],
        'q_flow_mae': []
    }
    
    #[STSI26.02.06]: Store all per-bus results across networks (individual buses)
    all_per_bus_results = []
    
    #[STSI26.02.06]: Store all per-line results across networks (individual lines)
    all_per_line_results = []
    
    #[STSI26.02.06]: Also track by bus type for summary statistics
    all_per_bus_errors_by_type = {
        'PQ': {'v_mag': [], 'v_ang': [], 'p': [], 'q': []},
        'PV': {'v_mag': [], 'v_ang': [], 'p': [], 'q': []},
        'Slack': {'v_mag': [], 'v_ang': [], 'p': [], 'q': []}
    }
    
    # [STSI26.02.10]: Track execution times
    gnn_times_pipeline = []       # data + forward + line flows
    gnn_times_forward_only = []   # forward only
    gnn_times_data_only = []      # data construction only
    gnn_times_line_only = []      # line_flows_only
    pypsa_times = []
    
    
    print(f"\nTesting on {len(test_networks)} networks")
    
    for net_idx, network in enumerate(test_networks):
        # Determine how many snapshots to test
        total_snapshots = len(network.snapshots)
        if num_snapshots_per_network is None:
            snapshots_to_test = total_snapshots
            snapshot_indices = range(total_snapshots)
        else:
            snapshots_to_test = min(num_snapshots_per_network, total_snapshots)
            #[STSI26.02.06]: Sample evenly across snapshots instead of first N
            snapshot_indices = np.linspace(0, total_snapshots-1, snapshots_to_test, dtype=int)
        
        print(f"\nNetwork {net_idx+1}/{len(test_networks)} "
              f"(testing {snapshots_to_test}/{total_snapshots} snapshots)")
        
        network_metrics = {key: [] for key in aggregate_metrics.keys()}
        network_gnn_times_pipeline = []
        network_gnn_times_forward_only = []
        network_gnn_times_data_only = []
        network_gnn_times_line_only = []
        network_pypsa_times = []
        
        #[STSI26.02.06]: Use evaluate_model for each snapshot to get detailed metrics
        for snap_idx in snapshot_indices:
            # GNN evaluation with detailed timing
            result = evaluate_model(
                model,
                network,
                snapshot_idx=snap_idx,
                use_edge_features=use_edge_features
            )

            t_data = result['timing']['data_only']
            t_forward = result['timing']['forward_only']
            t_line = result['timing']['line_flows_only']
            t_pipeline = result['timing']['pipeline_total']

            network_gnn_times_data_only.append(t_data) #[STSI26.02.10]: Store data construction time for this snapshot
            network_gnn_times_forward_only.append(t_forward) #[STSI26.02.10]: Store forward pass time for this snapshot
            network_gnn_times_line_only.append(t_line) #[STSI26.02.10]: Store line flow calculation time for this snapshot
            network_gnn_times_pipeline.append(t_pipeline) #[STSI26.02.10]: Store total pipeline time for this snapshot

            gnn_times_data_only.append(t_data) #[STSI26.02.10]: Store data construction time for this snapshot
            gnn_times_forward_only.append(t_forward)
            gnn_times_line_only.append(t_line)
            gnn_times_pipeline.append(t_pipeline)
            
            # Time PyPSA power flow (if requested)
            #[STSI26.02.06]: Fixed PyPSA pf() call - use snapshots parameter with list
            if compare_with_pypsa:
                network_copy = network.copy()
                start_time = time.time()
                results_pypsa=network_copy.pf(snapshots=[network.snapshots[snap_idx]])
                pypsa_time = time.time() - start_time
                network_pypsa_times.append(pypsa_time)
                pypsa_times.append(pypsa_time)
            
            # Collect metrics
            for key in aggregate_metrics.keys():
                metric_value = result['metrics'][key]
                network_metrics[key].append(metric_value)
                aggregate_metrics[key].append(metric_value)
            
            #[STSI26.02.06]: Store per-bus results with network and snapshot info
            for i in range(len(result['per_bus_results']['bus_name'])):
                bus_result = {
                    'network_idx': net_idx,
                    'snapshot_idx': snap_idx,
                    'bus_name': result['per_bus_results']['bus_name'][i],
                    'bus_idx': result['per_bus_results']['bus_idx'][i],
                    'bus_type': result['per_bus_results']['bus_type'][i],
                    'true_v_mag': result['per_bus_results']['true_v_mag'][i],
                    'pred_v_mag': result['per_bus_results']['pred_v_mag'][i],
                    'true_v_ang': result['per_bus_results']['true_v_ang'][i],
                    'pred_v_ang': result['per_bus_results']['pred_v_ang'][i],
                    'true_p': result['per_bus_results']['true_p'][i],
                    'pred_p': result['per_bus_results']['pred_p'][i],
                    'true_q': result['per_bus_results']['true_q'][i],
                    'pred_q': result['per_bus_results']['pred_q'][i],
                    'v_mag_error': result['per_bus_results']['v_mag_error'][i],
                    'v_ang_error': result['per_bus_results']['v_ang_error'][i],
                    'p_error': result['per_bus_results']['p_error'][i],
                    'q_error': result['per_bus_results']['q_error'][i]
                }
                all_per_bus_results.append(bus_result)
                
                # Also aggregate by type for summary
                bus_type = result['per_bus_results']['bus_type'][i]
                all_per_bus_errors_by_type[bus_type]['v_mag'].append(bus_result['v_mag_error'])
                all_per_bus_errors_by_type[bus_type]['v_ang'].append(bus_result['v_ang_error'])
                all_per_bus_errors_by_type[bus_type]['p'].append(bus_result['p_error'])
                all_per_bus_errors_by_type[bus_type]['q'].append(bus_result['q_error'])
            
            #[STSI26.02.06]: Store per-line results with network and snapshot info
            for i in range(len(result['per_line_results']['line_name'])):
                line_result = {
                    'network_idx': net_idx,
                    'snapshot_idx': snap_idx,
                    'line_name': result['per_line_results']['line_name'][i],
                    'line_idx': result['per_line_results']['line_idx'][i],
                    'bus0': result['per_line_results']['bus0'][i],
                    'bus1': result['per_line_results']['bus1'][i],
                    'true_p0': result['per_line_results']['true_p0'][i],
                    'pred_p0': result['per_line_results']['pred_p0'][i],
                    'true_p1': result['per_line_results']['true_p1'][i],
                    'pred_p1': result['per_line_results']['pred_p1'][i],
                    'true_q0': result['per_line_results']['true_q0'][i],
                    'pred_q0': result['per_line_results']['pred_q0'][i],
                    'true_q1': result['per_line_results']['true_q1'][i],
                    'pred_q1': result['per_line_results']['pred_q1'][i],
                    'p0_error': result['per_line_results']['p0_error'][i],
                    'p1_error': result['per_line_results']['p1_error'][i],
                    'q0_error': result['per_line_results']['q0_error'][i],
                    'q1_error': result['per_line_results']['q1_error'][i]
                }
                all_per_line_results.append(line_result)
        
        # Calculate average metrics for this network
        network_avg = {key: np.mean(values) for key, values in network_metrics.items()}
        network_std = {key: np.std(values) for key, values in network_metrics.items()}
        
        all_results.append({
            'network_idx': net_idx,
            'num_snapshots_tested': snapshots_to_test,
            'metrics_mean': network_avg,
            'metrics_std': network_std,
            # [STSI26.02.10]: Per-network timing statistics
            'gnn_pipeline_mean': np.mean(network_gnn_times_pipeline),
            'gnn_pipeline_std': np.std(network_gnn_times_pipeline),
            'gnn_forward_only_mean': np.mean(network_gnn_times_forward_only),
            'gnn_forward_only_std': np.std(network_gnn_times_forward_only),
            'gnn_data_only_mean': np.mean(network_gnn_times_data_only),
            'gnn_data_only_std': np.std(network_gnn_times_data_only),
            'gnn_line_only_mean': np.mean(network_gnn_times_line_only),
            'gnn_line_only_std': np.std(network_gnn_times_line_only),
            'pypsa_time_mean': np.mean(network_pypsa_times) if compare_with_pypsa and network_pypsa_times else None,
            'pypsa_time_std': np.std(network_pypsa_times) if compare_with_pypsa and network_pypsa_times else None,
            'speedup_pipeline': (
                np.mean(network_pypsa_times) / np.mean(network_gnn_times_pipeline)
                if compare_with_pypsa and network_pypsa_times else None
            ),
            'speedup_forward_only': (
                np.mean(network_pypsa_times) / np.mean(network_gnn_times_forward_only)
                if compare_with_pypsa and network_pypsa_times else None
            )
        })
        
        print(f"  Metrics (mean ± std):")
        print(f"    V_mag MAE:    {network_avg['v_mag_mae']:.6f} ± {network_std['v_mag_mae']:.6f} pu")
        print(f"    V_ang RMSE:   {network_avg['v_ang_rmse']:.6f} ± {network_std['v_ang_rmse']:.6f} rad")
        print(f"    P MAE:        {network_avg['p_mae']:.6f} ± {network_std['p_mae']:.6f} MW")
        print(f"    Q MAE:        {network_avg['q_mae']:.6f} ± {network_std['q_mae']:.6f} MVAr")
        print(f"    P_flow MAE:   {network_avg['p_flow_mae']:.6f} ± {network_std['p_flow_mae']:.6f} MW")
        print(f"    Q_flow MAE:   {network_avg['q_flow_mae']:.6f} ± {network_std['q_flow_mae']:.6f} MVAr")
    
        # [STSI26.02.06]: Calculate overall statistics across all test networks and snapshots
    overall_metrics = {
        key: {
            'mean': np.mean(values),
            'std': np.std(values),
            'min': np.min(values),
            'max': np.max(values),
            'median': np.median(values)
        }
        for key, values in aggregate_metrics.items()
    }
    
    # [STSI26.02.06]: Calculate per-bus-type statistics (for summary)
    per_bus_type_metrics = {}
    for bus_type, errors in all_per_bus_errors_by_type.items():
        per_bus_type_metrics[bus_type] = {
            'v_mag_mae': np.mean(errors['v_mag']) if errors['v_mag'] else 0,
            'v_ang_mae': np.mean(errors['v_ang']) if errors['v_ang'] else 0,
            'p_mae': np.mean(errors['p']) if errors['p'] else 0,
            'q_mae': np.mean(errors['q']) if errors['q'] else 0
        }
    
    # [STSI26.02.10]: Calculate timing statistics using new timing breakdown
    timing_stats = {
        'gnn_pipeline_mean': np.mean(gnn_times_pipeline) if gnn_times_pipeline else 0.0,
        'gnn_pipeline_std': np.std(gnn_times_pipeline) if gnn_times_pipeline else 0.0,
        'gnn_pipeline_median': np.median(gnn_times_pipeline) if gnn_times_pipeline else 0.0,
        'gnn_forward_only_mean': np.mean(gnn_times_forward_only) if gnn_times_forward_only else 0.0,
        'gnn_forward_only_std': np.std(gnn_times_forward_only) if gnn_times_forward_only else 0.0,
        'gnn_forward_only_median': np.median(gnn_times_forward_only) if gnn_times_forward_only else 0.0,
        'gnn_data_only_mean': np.mean(gnn_times_data_only) if gnn_times_data_only else 0.0,
        'gnn_data_only_std': np.std(gnn_times_data_only) if gnn_times_data_only else 0.0,
        'gnn_line_only_mean': np.mean(gnn_times_line_only) if gnn_times_line_only else 0.0,
        'gnn_line_only_std': np.std(gnn_times_line_only) if gnn_times_line_only else 0.0
    }
    
    if compare_with_pypsa and pypsa_times:
        timing_stats.update({
            'pypsa_mean': np.mean(pypsa_times),
            'pypsa_std': np.std(pypsa_times),
            'pypsa_median': np.median(pypsa_times),
            'speedup_pipeline_mean': np.mean(pypsa_times) / np.mean(gnn_times_pipeline),
            'speedup_pipeline_median': np.median(pypsa_times) / np.median(gnn_times_pipeline),
            'speedup_forward_only_mean': np.mean(pypsa_times) / np.mean(gnn_times_forward_only),
            'speedup_forward_only_median': np.median(pypsa_times) / np.median(gnn_times_forward_only)
        })
    
    print(f"\n{'='*60}")
    print("OVERALL TEST SET PERFORMANCE")
    print(f"{'='*60}")
    for key, stats in overall_metrics.items():
        print(f"{key:12s}: {stats['mean']:.6f} ± {stats['std']:.6f}")
        print(f"{'':14s}[min: {stats['min']:.6f}, median: {stats['median']:.6f}, max: {stats['max']:.6f}]")
    
    print(f"\n{'='*60}")
    print("PER-BUS-TYPE PERFORMANCE")
    print(f"{'='*60}")
    for bus_type, metrics in per_bus_type_metrics.items():
        print(f"{bus_type:8s}: V_mag={metrics['v_mag_mae']:.6f}, V_ang={metrics['v_ang_mae']:.6f}, "
              f"P={metrics['p_mae']:.6f}, Q={metrics['q_mae']:.6f}")
    
    if compare_with_pypsa and pypsa_times:
        print(f"\n{'='*60}")
        print("EXECUTION TIME COMPARISON")
        print(f"{'='*60}")
        print(f"GNN pipeline (data+forward+flows): {timing_stats['gnn_pipeline_mean']*1000:.3f} "
              f"± {timing_stats['gnn_pipeline_std']*1000:.3f} ms "
              f"(median: {timing_stats['gnn_pipeline_median']*1000:.3f} ms)")
        print(f"GNN forward only: {timing_stats['gnn_forward_only_mean']*1000:.3f} "
              f"± {timing_stats['gnn_forward_only_std']*1000:.3f} ms "
              f"(median: {timing_stats['gnn_forward_only_median']*1000:.3f} ms)")
        print(f"PyPSA: {timing_stats['pypsa_mean']*1000:.3f} "
              f"± {timing_stats['pypsa_std']*1000:.3f} ms "
              f"(median: {timing_stats['pypsa_median']*1000:.3f} ms)")
        print(f"Speedup (pipeline): {timing_stats['speedup_pipeline_mean']:.2f}x (mean), "
              f"{timing_stats['speedup_pipeline_median']:.2f}x (median)")
        print(f"Speedup (forward):  {timing_stats['speedup_forward_only_mean']:.2f}x (mean), "
              f"{timing_stats['speedup_forward_only_median']:.2f}x (median)")
    
    return {
        'per_network_results': all_results,
        'overall_metrics': overall_metrics,
        'per_bus_type_metrics': per_bus_type_metrics,
        'timing_stats': timing_stats,
        'raw_metrics': aggregate_metrics,
        'all_per_bus_results': all_per_bus_results,
        'all_per_line_results': all_per_line_results
    }




def calculate_line_flows(network, v_mag, v_ang, snapshot_idx):
    """
    Calculate line flows from voltage magnitudes and angles
    
    Args:
        network: PyPSA network
        v_mag: Voltage magnitudes (per unit)
        v_ang: Voltage angles (radians)
        snapshot_idx: Snapshot index
    
    Returns:
        Dictionary with line flows
    """
    # Convert angles to radians
    #v_ang_rad = np.deg2rad(v_ang)
    # [STSI 24.09.25] Angles where incorrectly converted to radians, but where already in radians from the model output, so no need to convert again.

    # Initialize arrays for line flows
    num_lines = len(network.lines)
    p0 = np.zeros(num_lines)
    p1 = np.zeros(num_lines)
    q0 = np.zeros(num_lines)
    q1 = np.zeros(num_lines)
    
    # Calculate line flows for each line
    for i, (idx, line) in enumerate(network.lines.iterrows()):
        # Get bus indices
        from_bus = int(line['bus0'].split()[-1]) - 1  # Convert 'Bus X' to index X-1
        to_bus = int(line['bus1'].split()[-1]) - 1
        
        # Get line parameters
        r = line['r']
        x = line['x']
        b = line['b']
        
        # Calculate admittance
        y = 1 / complex(r, x)
        g = y.real
        b_line = y.imag
        
        # Get voltage at from and to buses
        v_from = v_mag[from_bus]
        v_to = v_mag[to_bus]
        theta_from = v_ang[from_bus]
        theta_to = v_ang[to_bus]
        
        # Calculate angle difference
        theta_diff = theta_from - theta_to
        
        # Calculate active power flow
        p0[i] = (v_from**2 * g - v_from * v_to * g * np.cos(theta_diff) 
                - v_from * v_to * b_line * np.sin(theta_diff))
        p1[i] = (v_to**2 * g - v_from * v_to * g * np.cos(theta_diff) 
                + v_from * v_to * b_line * np.sin(theta_diff))
        
        # Calculate reactive power flow
        q0[i] = (-v_from**2 * (b_line + b/2) + v_from * v_to * b_line * np.cos(theta_diff) 
                - v_from * v_to * g * np.sin(theta_diff))
        q1[i] = (-v_to**2 * (b_line + b/2) + v_from * v_to * b_line * np.cos(theta_diff) 
                + v_from * v_to * g * np.sin(theta_diff))
    
    # Convert to MW and MVAr (assuming base power of 100 MVA)
    base_mva = network.sbase
    p0 *= base_mva
    p1 *= base_mva
    q0 *= base_mva
    q1 *= base_mva
    
    return {'p0': p0, 'p1': p1, 'q0': q0, 'q1': q1}

def visualize_results(results, network):
    """
    Visualize model predictions vs true values
    
    Args:
        results: Results dictionary from evaluate_model
        network: PyPSA network
    """
    # Extract predictions and true values
    pred_v_mag = results['predictions']['v_mag']
    true_v_mag = results['true_values']['v_mag']
    pred_v_ang = results['predictions']['v_ang']
    true_v_ang = results['true_values']['v_ang']
    pred_p = results['predictions']['p']
    true_p = results['true_values']['p']
    pred_q = results['predictions']['q']
    true_q = results['true_values']['q']
    
    # Create bus labels with types
    bus_labels = []
    for i, bus in enumerate(network.buses.index):
        bus_type = network.buses.loc[bus, 'type']
        bus_labels.append(f"{bus} ({bus_type})")
    
    # Create figure with subplots
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot voltage magnitude
    axs[0, 0].bar(np.arange(len(bus_labels)), true_v_mag, width=0.4, label='True', alpha=0.7)
    axs[0, 0].bar(np.arange(len(bus_labels))+0.4, pred_v_mag, width=0.4, label='Predicted', alpha=0.7)
    axs[0, 0].set_xticks(np.arange(len(bus_labels))+0.2)
    axs[0, 0].set_xticklabels(bus_labels, rotation=45, ha='right')
    axs[0, 0].set_ylabel('Voltage Magnitude (p.u.)')
    axs[0, 0].set_title('Voltage Magnitude Comparison')
    axs[0, 0].legend()
    axs[0, 0].grid(True, alpha=0.3)
    
    # Plot voltage angle
    axs[0, 1].bar(np.arange(len(bus_labels)), true_v_ang, width=0.4, label='True', alpha=0.7)
    axs[0, 1].bar(np.arange(len(bus_labels))+0.4, pred_v_ang, width=0.4, label='Predicted', alpha=0.7)
    axs[0, 1].set_xticks(np.arange(len(bus_labels))+0.2)
    axs[0, 1].set_xticklabels(bus_labels, rotation=45, ha='right')
    axs[0, 1].set_ylabel('Voltage Angle (rad)')
    axs[0, 1].set_title('Voltage Angle Comparison')
    axs[0, 1].legend()
    axs[0, 1].grid(True, alpha=0.3)
    
    # Plot active power
    axs[1, 0].bar(np.arange(len(bus_labels)), true_p, width=0.4, label='True', alpha=0.7)
    axs[1, 0].bar(np.arange(len(bus_labels))+0.4, pred_p, width=0.4, label='Predicted', alpha=0.7)
    axs[1, 0].set_xticks(np.arange(len(bus_labels))+0.2)
    axs[1, 0].set_xticklabels(bus_labels, rotation=45, ha='right')
    axs[1, 0].set_ylabel('Active Power (MW)')
    axs[1, 0].set_title('Active Power Comparison')
    axs[1, 0].legend()
    axs[1, 0].grid(True, alpha=0.3)
    
    # Plot reactive power
    axs[1, 1].bar(np.arange(len(bus_labels)), true_q, width=0.4, label='True', alpha=0.7)
    axs[1, 1].bar(np.arange(len(bus_labels))+0.4, pred_q, width=0.4, label='Predicted', alpha=0.7)
    axs[1, 1].set_xticks(np.arange(len(bus_labels))+0.2)
    axs[1, 1].set_xticklabels(bus_labels, rotation=45, ha='right')
    axs[1, 1].set_ylabel('Reactive Power (MVAr)')
    axs[1, 1].set_title('Reactive Power Comparison')
    axs[1, 1].legend()
    axs[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Plot line flow comparison
    pred_flows = results['predictions']['line_flows']
    true_flows = results['true_values']['line_flows']
    
    # Create line labels
    line_labels = []
    for i, line in enumerate(network.lines.index):
        from_bus = network.lines.loc[line, 'bus0']
        to_bus = network.lines.loc[line, 'bus1']
        line_labels.append(f"{line}\n({from_bus}->{to_bus})")
    
    # Create figure for line flows
    fig, axs = plt.subplots(2, 1, figsize=(14, 10))
    
    # Plot active power flow
    axs[0].bar(np.arange(len(line_labels)), true_flows['p0'], width=0.4, label='True', alpha=0.7)
    axs[0].bar(np.arange(len(line_labels))+0.4, pred_flows['p0'], width=0.4, label='Predicted', alpha=0.7)
    axs[0].set_xticks(np.arange(len(line_labels))+0.2)
    axs[0].set_xticklabels(line_labels, rotation=45, ha='right')
    axs[0].set_ylabel('Active Power Flow (MW)')
    axs[0].set_title('Line Active Power Flow Comparison (From Bus)')
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)
    
    # Plot reactive power flow
    axs[1].bar(np.arange(len(line_labels)), true_flows['q0'], width=0.4, label='True', alpha=0.7)
    axs[1].bar(np.arange(len(line_labels))+0.4, pred_flows['q0'], width=0.4, label='Predicted', alpha=0.7)
    axs[1].set_xticks(np.arange(len(line_labels))+0.2)
    axs[1].set_xticklabels(line_labels, rotation=45, ha='right')
    axs[1].set_ylabel('Reactive Power Flow (MVAr)')
    axs[1].set_title('Line Reactive Power Flow Comparison (From Bus)')
    axs[1].legend()
    axs[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    
    # Print metrics
    print("Model Performance Metrics:")
    print(f"Voltage Magnitude MAE: {results['metrics']['v_mag_mae']:.6f} p.u.")
    print(f"Voltage Angle RMSE: {results['metrics']['v_ang_rmse']:.6f}")
    print(f"Active Power MAE: {results['metrics']['p_mae']:.6f} MW")
    print(f"Reactive Power MAE: {results['metrics']['q_mae']:.6f} MVAr")
    print(f"Line Active Power Flow MAE: {results['metrics']['p_flow_mae']:.6f} MW")
    print(f"Line Reactive Power Flow MAE: {results['metrics']['q_flow_mae']:.6f} MVAr")

def compare_execution_time(model, network, num_runs=100):
    """
    Compare execution time between GNN and traditional power flow
    
    Args:
        model: Trained PowerFlowGNN model
        network: PyPSA network
        num_runs: Number of runs for timing
    """

    
    # Time GNN inference
    dataset = PowerFlowDataset([network])
    data = dataset[0]
    
    # Warm-up run
    model.eval()
    with torch.no_grad():
        _ = model(data)
    
    # Time GNN inference
    gnn_times = []
    for _ in range(num_runs):
        start_time = time.time()
        with torch.no_grad():
            _ = model(data)
        gnn_times.append(time.time() - start_time)
    
    # Time traditional power flow
    pf_times = []
    for _ in range(num_runs):
        # Create a copy to avoid modifying the original
        net_copy = network.copy()
        start_time = time.time()
        net_copy.pf(use_seed=True)
        pf_times.append(time.time() - start_time)
    
    # Calculate statistics
    gnn_avg = np.mean(gnn_times) * 1000  # Convert to ms
    gnn_std = np.std(gnn_times) * 1000
    pf_avg = np.mean(pf_times) * 1000
    pf_std = np.std(pf_times) * 1000
    speedup = pf_avg / gnn_avg
    
    # Print results
    print(f"GNN Inference Time: {gnn_avg:.2f} ± {gnn_std:.2f} ms")
    print(f"Traditional PF Time: {pf_avg:.2f} ± {pf_std:.2f} ms")
    print(f"Speedup: {speedup:.2f}x")
    
    # Plot comparison
    plt.figure(figsize=(10, 6))
    plt.bar(['GNN', 'Traditional PF'], [gnn_avg, pf_avg], yerr=[gnn_std, pf_std], alpha=0.7)
    plt.ylabel('Execution Time (ms)')
    plt.title('Execution Time Comparison')
    plt.grid(True, alpha=0.3)
    plt.show()


### Create multiple networks to get a richer dataset with different generator setpoints

In [ ]:
logger = logging.getLogger()
logger.setLevel(logging.ERROR)  # Suppress PyPSA warnings for cleaner output

# Generate training data
# changed the generator setpoint to a stochastic variable with voilatility range of 0.3, 50 scenarios, 24 steps per scenario. 
#



networks_small = generate_training_data_with_topology(
    num_scenarios=50,
    steps_per_scenario= 24,
    
    # Generator setpoint base values and ranges
    p_set_base={
        'gen2':1.63, 
        'gen3':0.7}, 
    volatility_range = 0.5,
    
    # Topology variation parameters
    include_topology_variants = True,
    gen_bus_options= [
        [1,2,3],
        [3,2,1],
        [4,5,6],
        [6,7,9],
        [2,4,6]
        ],
        load_bus_options=[
        # Original safe options
        [5, 7, 9],
        [7, 8, 9],
        [4, 6, 8],
        [1, 3, 5, 7, 9],
        
        # New 3-load combinations for variety
        [2, 4, 6],
        [3, 5, 7],
        [1, 4, 7],
        [2, 5, 8],
        [3, 6, 9],
        [1, 5, 9],
        [2, 6, 8],
        [4, 7, 9],
        
        # 4-load combinations for robustness
        [1, 3, 6, 8],
        [2, 4, 7, 9],
        [1, 4, 6, 9],
        [3, 5, 7, 9],
        [2, 5, 6, 8],
        
        # 5-load combinations
        [1, 2, 5, 7, 9],
        [2, 3, 6, 8, 9],
        [1, 4, 5, 7, 8],
    ],  
    
    # Line modification probability

    line_modification_prob = 0.35,  # 50% chance to modify lines
    line_param_variation = 0.15,   # ±15% parameter variation
    
    # Extended topology
    enable_extended_topology = True,
    extended_bus_prob = 0.4,  # 50% chance for extended topology
    
    # System parameters
    sbase= 1.0,
    seed = 42,
    verbose= True
)



In [ ]:

networks_large = generate_training_data_with_topology(num_scenarios=500,steps_per_scenario= 24*7,
    
    # Generator setpoint base values and ranges
    p_set_base={
        'gen2':1.63, 
        'gen3':0.7}, 
    volatility_range = 0.5,
    
    # Topology variation parameters
    include_topology_variants = True,
    gen_bus_options= [
        [1,2,3],
        [3,2,1],
        [4,5,6],
        [6,7,9],
        [2,4,6]
        ],
        load_bus_options=[
        # Original safe options
        [5, 7, 9],
        [7, 8, 9],
        [4, 6, 8],
        [1, 3, 5, 7, 9],
        
        # New 3-load combinations for variety
        [2, 4, 6],
        [3, 5, 7],
        [1, 4, 7],
        [2, 5, 8],
        [3, 6, 9],
        [1, 5, 9],
        [2, 6, 8],
        [4, 7, 9],
        
        # 4-load combinations for robustness
        [1, 3, 6, 8],
        [2, 4, 7, 9],
        [1, 4, 6, 9],
        [3, 5, 7, 9],
        [2, 5, 6, 8],
        
        # 5-load combinations
        [1, 2, 5, 7, 9],
        [2, 3, 6, 8, 9],
        [1, 4, 5, 7, 8],
    ],  
    
    # Line modification probability

    line_modification_prob = 0.5,  # 50% chance to modify lines
    line_param_variation = 0.25,   # ±25% parameter variation
    
    # Extended topology
    enable_extended_topology = True,
    extended_bus_prob = 0.4,  # 50% chance for extended topology
    
    # System parameters
    sbase= 1.0,
    seed = 42,
    verbose= True
)




In [ ]:
df = print_network_summary(
    networks_small, 
    max_rows=10,  # Show first 10 in console
    save_to_csv='network_small_summary.csv'  # Save all to CSV
)


### Sample plot and solution from the networks used for data generation

In [ ]:
# choose which network to evaluate the model on, for example the first one in the small dataset
networks=networks_small
#networks=networks_large

In [ ]:
# Solve a sample from the networks for reference
logger.setLevel(logging.ERROR) 
model_results(networks[39], plot_results=True, print_results=False)

### Preparing the data from multiple networks

In [ ]:
#Create the dataset from the generated networks to see that it works.
dataset = PowerFlowDataset(networks)

#### Check the initial weights

In [ ]:
model = PowerFlowGNN(
        node_features=dataset[0].x.size(1),
        edge_features=dataset[0].edge_attr.size(1),
        hidden_dim=64,
        num_layers=3
    )
    
for name, param in model.state_dict().items():
    print(f"\nParameter: {name}")
    print(f"Shape: {param.shape}")
    print(f"Sample values: {param.flatten()[:15]}")

### Train GNN based on the networks

In [ ]:
#select 15% of networks for validation randomly from the generated networks, another 15% for testing, rest for training

num_train_networks = int(0.7 * len(networks))
num_val_networks = int(0.85 * len(networks)) - num_train_networks

random.shuffle(networks)    # Shuffle networks before splitting
train_networks = networks[:num_train_networks]
val_networks = networks[num_train_networks:num_train_networks + num_val_networks]
test_networks = networks[num_train_networks + num_val_networks:]


train_dataset = PowerFlowDataset(train_networks)
train_Y_cache = precompute_Y_matrices(train_networks)
train_loader = DataLoader(
        train_dataset, 
        batch_size=1, 
        shuffle=True,   # Shuffle makes sure that the data is randomly sampled during training, training dataset contains multiple networks with different topologies
        follow_batch=['network_idx'] #[STSI 15.01.2026] added to keep track of which network used
        ) 
    

# Quick verification
print("\n=== Verification ===")
print(f"Train networks: {len(train_networks)}")
print(f"Train dataset samples: {len(train_dataset)}")
print(f"Train Y-cache: {len(train_Y_cache)}")

# Check one batch
for batch in train_loader:
    net_idx = batch.network_idx.item() if batch.network_idx.dim() > 0 else batch.network_idx
    network = train_networks[net_idx]
    Y_matrix = train_Y_cache[net_idx]
    
    print(f"\nFirst batch:")
    print(f"  network_idx: {net_idx}")
    print(f"  num_nodes: {batch.num_nodes}")
    print(f"  network.buses: {len(network.buses)}")
    print(f"  Y-matrix size: {Y_matrix[0].shape}")
    print(f"  Match: {batch.num_nodes == len(network.buses) == Y_matrix[0].size(0)}")
    break

print("=== Ready to train! ===\n")


### Train in small networksample (50 scenarios x 24 timesteps)
shows how batch size of one makes the training overfit to singel topology. increasing batch size helps to reduce the overfitting
physics weigth makes validation losses increase after some time

In [ ]:
# Train the model
logger.setLevel(logging.ERROR) 
print("\n=== Training ===\n with no edge features" )
model, losses, test_results = train_power_flow_gnn(networks, num_epochs=50, batch_size=16, weight_physics=0.1,lr=0.0005, use_edge_features=False)
print("\n=== Training ===\n with edge features" )
model, losses, test_results = train_power_flow_gnn(networks, num_epochs=50, batch_size=16, weight_physics=0.1,lr=0.0005, use_edge_features=True)

In [ ]:
# Train the model with batch of one

print("\n=== Training ===\n with no edge features" )
model, losses, test_results = train_power_flow_gnn(networks, num_epochs=50, batch_size=1, weight_physics=0.1,lr=0.0005, use_edge_features=False)


In [ ]:
# Train the model
logger.setLevel(logging.ERROR) 
model_no_physics, losses_no_physics, test_results_no_physics = train_power_flow_gnn(networks_small,batch_size=16, num_epochs=50, weight_physics=0.0,lr=0.0005)

In [ ]:
model_physics_03, losses_physics_03, test_results_physics_03 = train_power_flow_gnn(networks_small, batch_size=16, num_epochs=50, weight_physics=0.3,lr=0.0005)

### Training with larger dataset (more scenarios and more timesteps)
Shows how larger dataset tends to have validation losses diverging when batches are to small or learning rate to big. Find good combination between lr and batch size that does not overfitt model to specific data. 


In [ ]:
# Training on larger dataset with more epochs
model_large, losses_large, test_results_large = train_power_flow_gnn(networks_large, batch_size=16, num_epochs=50, weight_physics=0.3,lr=0.0005)

In [ ]:
model_large, losses_large, test_results_large = train_power_flow_gnn(networks_large, batch_size=32, num_epochs=50, weight_physics=0.3,lr=0.0005)

In [ ]:
model_large, losses_large, test_results_large = train_power_flow_gnn(networks_large, batch_size=32, num_epochs=50, weight_physics=0.1,lr=0.0005)

In [ ]:
model_large, losses_large, test_results_large = train_power_flow_gnn(networks_large, batch_size=32, num_epochs=50, weight_physics=0.1,lr=0.0001) #reduced learning rate for better convergence
model_large, losses_large, test_results_large = train_power_flow_gnn(networks_large, batch_size=64, num_epochs=50, weight_physics=0.1,lr=0.0005) # increased batch size for faster training

In [ ]:
model_large, losses_large, test_results_large = train_power_flow_gnn(networks_large, batch_size=64, num_epochs=30, weight_physics=0.1,lr=0.0001,max_n_test=15) #reduced learning rate and increased batch sizefor better convergence
model_large, losses_large, test_results_large = train_power_flow_gnn(networks_large, batch_size=132, num_epochs=30, weight_physics=0.1,lr=0.0001,max_n_test=15) # even larger batch size for faster training

In [ ]:
model_large, losses_large, test_results_large = train_power_flow_gnn(networks_large, batch_size=132, num_epochs=30, weight_physics=0.3,lr=0.0001,max_n_test=15) # even larger batch size for faster training

### Test with PTDF as edge feature

## Testing.
only using one network for testing, so i need to make sure that all the test networks in the dataset is used for testing...

In [ ]:
# Create a test network with different parameters
test_network = create_9_bus_network(steps=10, p_set_gen2=1.70, p_set_gen3=1.00,load_voilatility=0.3, plot=False)
test_network.pf(use_seed=True)

# Evaluate the model
results = evaluate_model(model, test_network)

# Visualize results
visualize_results(results, test_network)

# Compare execution time
compare_execution_time(model, test_network)

# Below this point is not adatped to topology change

## Plot functions to compare timestep results (GNN vs conventional)

In [ ]:
def plot_comparison_results_with_masks(bus_results_model, bus_results_conv, line_results_model, line_results_conv, network, prediction_masks):
    """
    Plot comparison between model prediction results and conventional calculation results
    Only highlighting the values that were actually predicted by the model
    """

    
    # Plot bus results comparison
    n_buses = len(network.buses)
    fig, axes = plt.subplots(4, 1, figsize=(20, 3*n_buses), sharex=True)
    properties = [('P (MW)', 'P'), ('Q (MVAr)', 'Q'), ('V (pu)', 'V'), ('Angle (deg)', 'Angle')]
    labels = ['Active Power (P)', 'Reactive Power (Q)', 'Voltage Magnitude (p.u.)', 'Voltage Angle']

    for bus in network.buses.index:
        # Determine bus type for legend
        if bus in network.loads.bus.values:
            bus_type = f"{bus} (Load)"
        elif bus in network.generators.bus.values:
            gen_idx = network.generators[network.generators.bus == bus].index[0]
            if network.generators.loc[gen_idx, 'control'] == 'Slack':
                bus_type = f"{bus} (Gen-Slack)"
            else:
                bus_type = f"{bus} (Gen-PV)"
        else:
            bus_type = bus

        for i, (prop, mask_key) in enumerate(properties):
            # Get prediction mask for this bus and property
            is_predicted = prediction_masks[mask_key][bus]
            
            # Plot all values (predicted and known) with different styles
            # Model values
            model_values = bus_results_model[bus][prop]
            conv_values = bus_results_conv[bus][prop]
            
            # Plot predicted values with solid lines and markers
            predicted_indices = model_values.index[is_predicted]
            known_indices = model_values.index[~is_predicted]
            
            if len(predicted_indices) > 0:
                axes[i].plot(predicted_indices, model_values[predicted_indices], 
                           marker='o', linestyle='-', linewidth=2, markersize=6,
                           label=f'{bus_type} - Model (Predicted)', alpha=0.8)
                axes[i].plot(predicted_indices, conv_values[predicted_indices], 
                           marker='x', linestyle='--', linewidth=2, markersize=8,
                           label=f'{bus_type} - Conventional (Predicted)', alpha=0.8)
            
            # Plot known values with lighter style
            if len(known_indices) > 0:
                axes[i].plot(known_indices, model_values[known_indices], 
                           marker='s', linestyle=':', linewidth=1, markersize=4,
                           label=f'{bus_type} - Known Values', alpha=0.4, color='gray')
            
            axes[i].set_ylabel(labels[i], fontsize=12)
            axes[i].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
            axes[i].grid(True, alpha=0.3)
            axes[i].set_title(f'{labels[i]} Comparison (Predicted vs Known)', fontsize=12, fontweight='bold')

    axes[-1].set_xlabel('Time', fontsize=12)
    
    plt.suptitle('Bus Results: Model vs Conventional (Highlighted Predictions)', fontsize=12, fontweight='bold',y=0.98)
    

    plt.tight_layout(rect=[0, 0.02, 0.9, 0.98])
    plt.show()

    # Plot line results comparison (all line flows are calculated from predicted voltages)
    n_lines = len(network.lines)
    fig, axs = plt.subplots(n_lines, 2, figsize=(18, 3*n_lines), sharex=True)
    
    # Handle case of single line
    if n_lines == 1:
        axs = axs.reshape(1, -1)
    
    for i, line in enumerate(network.lines.index):
        from_bus = network.lines.loc[line, 'bus0']
        to_bus = network.lines.loc[line, 'bus1']
        line_name = f"{line} ({from_bus}→{to_bus})"
        s_nom = network.lines.loc[line, 's_nom']
        
        # Active Power (P) - Left subplot
        axs[i, 0].plot(line_results_model.index, line_results_model[line]['P0 (MW)'], 
                      marker='o', linestyle='-', linewidth=2, markersize=6,
                      label=f"P-in Model (from {from_bus})")
        axs[i, 0].plot(line_results_conv.index, line_results_conv[line]['P0 (MW)'], 
                      marker='x', linestyle='--', linewidth=2, markersize=8,
                      label=f"P-in Conv (from {from_bus})")
        axs[i, 0].plot(line_results_model.index, -line_results_model[line]['P1 (MW)'], 
                      marker='s', linestyle='-', linewidth=2, markersize=6,
                      label=f"P-out Model (to {to_bus})")
        axs[i, 0].plot(line_results_conv.index, -line_results_conv[line]['P1 (MW)'], 
                      marker='+', linestyle='--', linewidth=2, markersize=10,
                      label=f"P-out Conv (to {to_bus})")
        
        # Add S_nom reference lines
        axs[i, 0].axhline(y=s_nom, color='red', linestyle=':', linewidth=2, alpha=0.7,
                         label=f"±S_nom = ±{s_nom} MVA")
        axs[i, 0].axhline(y=-s_nom, color='red', linestyle=':', linewidth=2, alpha=0.7)
        
        axs[i, 0].set_ylabel('Active Power (MW)', fontsize=12)
        axs[i, 0].set_title(f"Active Power Flow - {line_name}", fontsize=12, fontweight='bold')
        axs[i, 0].legend(fontsize=10)
        axs[i, 0].grid(True, alpha=0.3)
        
        # Reactive Power (Q) - Right subplot
        axs[i, 1].plot(line_results_model.index, line_results_model[line]['Q0 (MVAr)'], 
                      marker='o', linestyle='-', linewidth=2, markersize=6,
                      label=f"Q-in Model (from {from_bus})")
        axs[i, 1].plot(line_results_conv.index, line_results_conv[line]['Q0 (MVAr)'], 
                      marker='x', linestyle='--', linewidth=2, markersize=8,
                      label=f"Q-in Conv (from {from_bus})")
        axs[i, 1].plot(line_results_model.index, -line_results_model[line]['Q1 (MVAr)'], 
                      marker='s', linestyle='-', linewidth=2, markersize=6,
                      label=f"Q-out Model (to {to_bus})")
        axs[i, 1].plot(line_results_conv.index, -line_results_conv[line]['Q1 (MVAr)'], 
                      marker='+', linestyle='--', linewidth=2, markersize=10,
                      label=f"Q-out Conv (to {to_bus})")
        
        axs[i, 1].set_ylabel('Reactive Power (MVAr)', fontsize=12)
        axs[i, 1].set_title(f"Reactive Power Flow - {line_name}", fontsize=12, fontweight='bold')
        axs[i, 1].legend(fontsize=10)
        axs[i, 1].grid(True, alpha=0.3)
        
    fig.text(0.5, 0.02, 'Time', ha='center', fontsize=14, fontweight='bold')
    plt.suptitle('Line Flow Results: Model vs Conventional (All Calculated from Predictions)', fontsize=16, fontweight='bold',y=0.98)
    
    plt.tight_layout(rect=[0, 0.02, 0.9, 0.98])
    plt.show()


In [ ]:
def predict_network_results_with_masks(model, network,debug=False):
    """
    Use trained GNN model to predict network results for all snapshots
    Returns results in the same format as model_results() function along with prediction masks
    
    Args:
        model: Trained PowerFlowGNN model
        network: PyPSA network to predict on
    
    Returns:
        bus_results: DataFrame with predicted bus results
        line_results: DataFrame with predicted line results
        prediction_masks: Dictionary containing masks for what was actually predicted
    """

    
    # Create dataset from the network
    dataset = PowerFlowDataset([network])
    
    # Create DataFrames with same structure as model_results()
    bus_results = pd.DataFrame(
        index=network.snapshots,
        columns=pd.MultiIndex.from_product([
            network.buses.index,
            ['P (MW)', 'Q (MVAr)', 'V (pu)', 'Angle (deg)']
        ])
    )
    
    line_results = pd.DataFrame(
        index=network.snapshots,
        columns=pd.MultiIndex.from_product([
            network.lines.index,
            ['P0 (MW)', 'P1 (MW)', 'Q0 (MVAr)', 'Q1 (MVAr)']
        ])
    )
    
    # Create prediction masks - True where model predicts, False where values are known
    prediction_masks = {
        'P': pd.DataFrame(False, index=network.snapshots, columns=network.buses.index),
        'Q': pd.DataFrame(False, index=network.snapshots, columns=network.buses.index),
        'V': pd.DataFrame(False, index=network.snapshots, columns=network.buses.index),
        'Angle': pd.DataFrame(False, index=network.snapshots, columns=network.buses.index)
    }
    
    # Set prediction masks based on bus types (same for all snapshots)
    for bus in network.buses.index:
        bus_type = network.buses.loc[bus, 'type']
        
        if bus_type == 'PQ':
            # PQ buses: predict V_mag and V_ang
            prediction_masks['V'].loc[:, bus] = True
            prediction_masks['Angle'].loc[:, bus] = True
            
        elif bus_type == 'PV':
            # PV buses: predict V_ang and Q
            prediction_masks['Angle'].loc[:, bus] = True
            prediction_masks['Q'].loc[:, bus] = True
            
        else:  # Slack
            # Slack bus: predict P and Q
            prediction_masks['P'].loc[:, bus] = True
            prediction_masks['Q'].loc[:, bus] = True
    
    model.eval()
    
    # Predict for each snapshot
    for t_idx, t in enumerate(network.snapshots):
        # Get data for this snapshot
        data = dataset[t_idx]
        
        # Make prediction
        with torch.no_grad():
            pred = model(data)
        
        # Extract predicted values for each bus
        pred_v_mag = torch.zeros(len(network.buses))
        pred_v_ang = torch.zeros(len(network.buses))
        pred_p = torch.zeros(len(network.buses))
        pred_q = torch.zeros(len(network.buses))
        
        # Fill predictions based on bus type
        for i, bus in enumerate(network.buses.index):
            bus_type = network.buses.loc[bus, 'type']
            
            if bus_type == 'PQ':
                # For PQ buses: predict v_mag, v_ang
                pred_v_mag[i] = pred[i, 0]
                pred_v_ang[i] = pred[i, 1]
                # P and Q are known (from loads/generators)
                pred_p[i] = network.buses_t.p.loc[t, bus]
                pred_q[i] = network.buses_t.q.loc[t, bus]
                
            elif bus_type == 'PV':
                # For PV buses: known v_mag, predict v_ang, q
                pred_v_mag[i] = network.buses_t.v_mag_pu.loc[t, bus]  # Known
                pred_v_ang[i] = pred[i, 1]
                pred_p[i] = network.buses_t.p.loc[t, bus]  # Known
                pred_q[i] = pred[i, 3]
                
            else:  # Slack
                # For slack bus: known v_mag, v_ang, predict p, q
                pred_v_mag[i] = network.buses_t.v_mag_pu.loc[t, bus]  # Known
                pred_v_ang[i] = network.buses_t.v_ang.loc[t, bus]     # Known
                pred_p[i] = pred[i, 2]
                pred_q[i] = pred[i, 3]
        
        # Fill bus results
        for i, bus in enumerate(network.buses.index):
            bus_results.loc[t, (bus, 'P (MW)')] = pred_p[i].item()
            bus_results.loc[t, (bus, 'Q (MVAr)')] = pred_q[i].item()
            bus_results.loc[t, (bus, 'V (pu)')] = pred_v_mag[i].item()
            bus_results.loc[t, (bus, 'Angle (deg)')] = pred_v_ang[i].item()
        
        # Debugging to check angle dimensions:
        if debug:
            print(f"Predicted angles (should be small radians): {pred_v_ang[:3].numpy()}")
            print(f"Predicted angles in degrees: {np.degrees(pred_v_ang[:3]).numpy()}")
            print(f"Angle range: {pred_v_ang.min():.6f} to {pred_v_ang.max():.6f} radians")

        # Calculate line flows from predicted voltages
        pred_line_flows = calculate_line_flows(
            network, pred_v_mag.numpy(), pred_v_ang.numpy(),t_idx)
        
        # Fill line results
        for i, line in enumerate(network.lines.index):
            line_results.loc[t, (line, 'P0 (MW)')] = pred_line_flows['p0'][i]
            line_results.loc[t, (line, 'P1 (MW)')] = pred_line_flows['p1'][i]
            line_results.loc[t, (line, 'Q0 (MVAr)')] = pred_line_flows['q0'][i]
            line_results.loc[t, (line, 'Q1 (MVAr)')] = pred_line_flows['q1'][i]
    
    return bus_results, line_results, prediction_masks


In [ ]:
def compare_model_with_conventional_masked(network, model):
    """
    Complete comparison function using prediction masks to compare only predicted values
    
    Args:
        network: PyPSA network
        model: Trained PowerFlowGNN model
    
    Returns:
        All results for further analysis including prediction masks
    """
    print("Running conventional power flow calculation...")
    # Get conventional results using your existing function
    bus_results_conv, line_results_conv = model_results(network, print_results=False, plot_results=False)
    
    print("Running GNN model prediction...")
    # Get model prediction results with masks
    bus_results_model, line_results_model, prediction_masks = predict_network_results_with_masks(model, network)
    
    print("\n=== BUS TYPE PREDICTION SUMMARY ===")
    for bus in network.buses.index:
        bus_type = network.buses.loc[bus, 'type']
        predicted_props = []
        
        if prediction_masks['P'].loc[network.snapshots[0], bus]:
            predicted_props.append('P')
        if prediction_masks['Q'].loc[network.snapshots[0], bus]:
            predicted_props.append('Q')
        if prediction_masks['V'].loc[network.snapshots[0], bus]:
            predicted_props.append('V_mag')
        if prediction_masks['Angle'].loc[network.snapshots[0], bus]:
            predicted_props.append('V_ang')
            
    #    print(f"{bus} ({bus_type}): Predicts {', '.join(predicted_props) if predicted_props else 'Nothing'}")
    
    print("\nPlotting comparison...")
    # Plot comparison with masks
    plot_comparison_results_with_masks(bus_results_model, bus_results_conv, 
                                     line_results_model, line_results_conv, 
                                     network, prediction_masks)
    
    return {
        'model_bus': bus_results_model,
        'conv_bus': bus_results_conv,
        'model_line': line_results_model,
        'conv_line': line_results_conv,
        'prediction_masks': prediction_masks
    }


In [ ]:
def calculate_accuracy_metrics_with_masks(results, prediction_masks):
    """
    Calculate and print accuracy metrics between model and conventional results
    Only for values that were actually predicted by the model
    """

    
    model_bus = results['model_bus']
    conv_bus = results['conv_bus']
    
    print("\n=== ACCURACY METRICS (PREDICTED VALUES ONLY) ===")
    
    # Bus metrics - only for predicted values
    properties = [('P (MW)', 'P'), ('Q (MVAr)', 'Q'), ('V (pu)', 'V'), ('Angle (deg)', 'Angle')]
    
    for prop_name, mask_key in properties:
        # Collect only predicted values across all buses and snapshots
        model_vals = []
        conv_vals = []
        
        for bus in model_bus.columns.get_level_values(0).unique():
            if (bus, prop_name) in model_bus.columns:
                # Get mask for this bus and property
                mask = prediction_masks[mask_key][bus]
                
                # Only include values where mask is True (predicted values)
                bus_model_vals = model_bus[bus][prop_name][mask].values
                bus_conv_vals = conv_bus[bus][prop_name][mask].values
                
                model_vals.extend(bus_model_vals)
                conv_vals.extend(bus_conv_vals)
        
        if len(model_vals) > 0:  # Only calculate if we have predicted values
            model_vals = np.array(model_vals)
            conv_vals = np.array(conv_vals)
            
            # Calculate metrics
            mae = np.mean(np.abs(model_vals - conv_vals))
            rmse = np.sqrt(np.mean((model_vals - conv_vals)**2))
            mape = np.mean(np.abs((model_vals - conv_vals) / (conv_vals + 1e-8))) * 100
            
            print(f"{prop_name} (Predicted only - {len(model_vals)} values):")
            print(f"  MAE: {mae:.6f}")
            print(f"  RMSE: {rmse:.6f}")
            print(f"  MAPE: {mape:.2f}%")
        else:
            print(f"{prop_name}: No predicted values to compare")


In [ ]:
# Compare on a single network
results = compare_model_with_conventional_masked(networks[20], model)

# Calculate accuracy metrics
calculate_accuracy_metrics_with_masks(results, results['prediction_masks'])



#### Compare for all networks

In [ ]:
#======= Creates a lot (50x2) of plots=========================
# Run comparison on multiple networks
def run_complete_comparison(networks, model):
    for i, network in enumerate(networks):
        print(f"\n=== Comparison for Network {i+1} ===")
        results = compare_model_with_conventional_masked(network, model)
        calculate_accuracy_metrics_with_masks(results,results['prediction_masks'])


run_complete_comparison(networks, model)

## Physics Loss Comparison Experiment

### Traning several modells with variations on physics and MSE loss (earlier version...)

In [ ]:

def comprehensive_physics_loss_comparison(networks, test_networks=None, num_epochs=100):
    """
    Comprehensive pipeline to compare models with different physics loss fractions
    
    Args:
        networks: List of training networks
        test_networks: List of test networks (if None, creates new ones)
        num_epochs: Number of training epochs
    
    Returns:
        Dictionary with all results and trained models
    """
    
    # Define physics loss weights to compare
    physics_loss_weights = [0.0, 0.3, 0.5]
    
    # Storage for results
    results = {
        'models': {},
        'training_history': {},
        'test_metrics': {},
        'detailed_predictions': {},
        'execution_times': {}
    }
    
    print("="*60)
    print("PHYSICS-INFORMED LOSS COMPARISON PIPELINE")
    print("="*60)
    
    # Step 1: Train models with different physics loss fractions
    for weight in physics_loss_weights:
        print(f"\n🔄 Training model with physics loss fraction: {weight}")
        print("-" * 50)
        
        # Train the model
        model, (train_losses, val_losses) = train_power_flow_gnn(
            networks, 
            num_epochs=num_epochs, 
            weight_physics=weight
        )
        
        # Store results
        results['models'][weight] = model
        results['training_history'][weight] = {
            'train_losses': train_losses,
            'val_losses': val_losses
        }
        
        print(f"✅ Model with physics weight {weight} trained successfully")
    
    # Step 2: Create diverse test scenarios
    if test_networks is None:
        print(f"\n🔄 Generating test scenarios...")
        test_networks = generate_diverse_test_scenarios()
    
    # Step 3: Evaluate all models on test scenarios
    print(f"\n🔄 Evaluating models on {len(test_networks)} test scenarios...")
    
    for weight in physics_loss_weights:
        print(f"\nEvaluating model with physics weight {weight}...")
        
        scenario_metrics = []
        scenario_predictions = []
        execution_times = []
        
        for i, test_network in enumerate(test_networks):
            # Evaluate model
            eval_results = evaluate_model(results['models'][weight], test_network)
            scenario_metrics.append(eval_results['metrics'])
            scenario_predictions.append(eval_results['predictions'])
            
            # Measure execution time
            exec_time = measure_inference_time(results['models'][weight], test_network)
            execution_times.append(exec_time)
        
        # Store aggregated results
        results['test_metrics'][weight] = aggregate_metrics(scenario_metrics)
        results['detailed_predictions'][weight] = scenario_predictions
        results['execution_times'][weight] = {
            'mean': np.mean(execution_times),
            'std': np.std(execution_times),
            'all_times': execution_times
        }
    
    # Step 4: Generate comprehensive comparison
    comparison_results = generate_comparison_analysis(results)
    
    return results, comparison_results

def generate_diverse_test_scenarios(num_scenarios=10):
    """Generate diverse test scenarios with varying load and generation patterns"""
    test_networks = []
    
    # Scenario 1: Base case
    test_networks.append(create_9_bus_network(steps=1, p_set_gen2=170, p_set_gen3=100, plot=False))
    
    # Scenario 2-4: High load scenarios
    for gen2, gen3 in [(180, 110), (185, 115), (190, 120)]:
        test_networks.append(create_9_bus_network(steps=1, p_set_gen2=gen2, p_set_gen3=gen3, plot=False))
    
    # Scenario 5-7: Low load scenarios
    for gen2, gen3 in [(150, 80), (145, 75), (140, 70)]:
        test_networks.append(create_9_bus_network(steps=1, p_set_gen2=gen2, p_set_gen3=gen3, plot=False))
    
    # Scenario 8-10: Unbalanced scenarios
    for gen2, gen3 in [(200, 60), (130, 130), (175, 95)]:
        test_networks.append(create_9_bus_network(steps=1, p_set_gen2=gen2, p_set_gen3=gen3, plot=False))
    
    # Solve power flow for all test networks
    for network in test_networks:
        network.pf(use_seed=True)
    
    return test_networks

def measure_inference_time(model, network, num_runs=50):
    """Measure average inference time for a model"""

    
    dataset = PowerFlowDataset([network])
    data = dataset[0]
    
    # Warm-up
    model.eval()
    with torch.no_grad():
        _ = model(data)
    
    # Measure times
    times = []
    for _ in range(num_runs):
        start = time.time()
        with torch.no_grad():
            _ = model(data)
        times.append(time.time() - start)
    
    return np.mean(times) * 1000  # Convert to ms

def aggregate_metrics(scenario_metrics):
    """Aggregate metrics across multiple scenarios"""
    aggregated = {}
    
    # Calculate mean and std for each metric
    for metric_name in scenario_metrics[0].keys():
        values = [scenario[metric_name] for scenario in scenario_metrics]
        aggregated[metric_name] = {
            'mean': np.mean(values),
            'std': np.std(values),
            'min': np.min(values),
            'max': np.max(values),
            'all_values': values
        }
    
    return aggregated

def generate_comparison_analysis(results):
    """Generate comprehensive comparison analysis"""
    physics_weights = list(results['models'].keys())
    
    # Create comparison DataFrame
    comparison_data = []
    
    for weight in physics_weights:
        metrics = results['test_metrics'][weight]
        exec_time = results['execution_times'][weight]
        
        row = {
            'Physics Weight': weight,
            'V_mag MAE (mean)': metrics['v_mag_mae']['mean'],
            'V_mag MAE (std)': metrics['v_mag_mae']['std'],
            'V_ang RMSE (mean)': metrics['v_ang_rmse']['mean'],
            'V_ang RMSE (std)': metrics['v_ang_rmse']['std'],
            'P MAE (mean)': metrics['p_mae']['mean'],
            'P MAE (std)': metrics['p_mae']['std'],
            'Q MAE (mean)': metrics['q_mae']['mean'],
            'Q MAE (std)': metrics['q_mae']['std'],
            'Exec Time (ms)': exec_time['mean'],
            'Exec Time Std': exec_time['std']
        }
        comparison_data.append(row)
    
    comparison_df = pd.DataFrame(comparison_data)
    
    return comparison_df

def visualize_comprehensive_comparison(results, comparison_df):
    """Create comprehensive visualization of results"""
    
    # 1. Training curves comparison
    plt.figure(figsize=(15, 12))
    
    # Training loss curves
    plt.subplot(2, 3, 1)
    for weight in results['training_history'].keys():
        train_losses = results['training_history'][weight]['train_losses']
        plt.plot(train_losses, label=f'Physics Weight {weight}', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Training Loss')
    plt.title('Training Loss Comparison')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Validation loss curves
    plt.subplot(2, 3, 2)
    for weight in results['training_history'].keys():
        val_losses = results['training_history'][weight]['val_losses']
        plt.plot(val_losses, label=f'Physics Weight {weight}', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Validation Loss')
    plt.title('Validation Loss Comparison')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Performance metrics comparison
    metrics = ['v_mag_mae', 'v_ang_rmse', 'p_mae', 'q_mae']
    metric_titles = ['Voltage Magnitude MAE', 'Voltage Angle RMSE', 'Active Power MAE', 'Reactive Power MAE']
    
    for i, (metric, title) in enumerate(zip(metrics, metric_titles)):
        plt.subplot(2, 3, i + 3)
        
        means = [results['test_metrics'][weight][metric]['mean'] for weight in results['models'].keys()]
        stds = [results['test_metrics'][weight][metric]['std'] for weight in results['models'].keys()]
        
        x = range(len(results['models'].keys()))
        plt.bar(x, means, yerr=stds, capsize=5, alpha=0.7)
        plt.xticks(x, [f'{w}' for w in results['models'].keys()])
        plt.xlabel('Physics Loss Weight')
        plt.ylabel(title)
        plt.title(f'{title} Comparison')
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 2. Execution time comparison
    plt.figure(figsize=(10, 6))
    weights = list(results['models'].keys())
    exec_times = [results['execution_times'][weight]['mean'] for weight in weights]
    exec_stds = [results['execution_times'][weight]['std'] for weight in weights]
    
    plt.bar(range(len(weights)), exec_times, yerr=exec_stds, capsize=5, alpha=0.7)
    plt.xticks(range(len(weights)), [f'{w}' for w in weights])
    plt.xlabel('Physics Loss Weight')
    plt.ylabel('Execution Time (ms)')
    plt.title('Inference Time Comparison')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # 3. Print detailed comparison table
    print("\n" + "="*80)
    print("DETAILED PERFORMANCE COMPARISON")
    print("="*80)
    print(comparison_df.to_string(index=False, float_format='%.6f'))
    
    # 4. Determine best model
    print("\n" + "="*80)
    print("PERFORMANCE RANKING")
    print("="*80)
    
    # Rank by different criteria
    rankings = {}
    for metric in ['V_mag MAE (mean)', 'V_ang RMSE (mean)', 'P MAE (mean)', 'Q MAE (mean)']:
        sorted_df = comparison_df.sort_values(metric)
        rankings[metric] = sorted_df['Physics Weight'].tolist()
        print(f"\nBest to worst for {metric}:")
        for i, weight in enumerate(rankings[metric], 1):
            value = sorted_df[sorted_df['Physics Weight'] == weight][metric].iloc[0]
            print(f"  {i}. Physics Weight {weight}: {value:.6f}")



In [ ]:
def plot_predictions_comparison_per_node(all_results, network, physics_weights=[0.0, 0.3, 0.5]):
    """
    Plot prediction vs true values per node for different physics loss weights
    
    Args:
        all_results: Dictionary where keys are physics weights and values are results from evaluate_model
        network: PyPSA network object
        physics_weights: List of physics loss weights to compare
    """
    # Create bus labels with types
    bus_labels = []
    for i, bus in enumerate(network.buses.index):
        bus_type = network.buses.loc[bus, 'type']
        bus_labels.append(f"Bus {bus}\n({bus_type})")
    
    # Create figure with subplots for each target variable
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    
    # Define colors for different physics weights
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    
    # Target variables and their indices
    targets = [
        ('v_mag', 'Voltage Magnitude (p.u.)', (0, 0)),
        ('v_ang', 'Voltage Angle (rad)', (0, 1)), 
        ('p', 'Active Power (MW)', (1, 0)),
        ('q', 'Reactive Power (MVAr)', (1, 1))
    ]
    
    # Plot each target variable
    for target_name, ylabel, (row, col) in targets:
        ax = axes[row, col]
        
        # Get true values (same across all models)
        true_values = all_results[physics_weights[0]]['true_values'][target_name]
        
        # Plot true values as baseline
        x_positions = np.arange(len(bus_labels))
        ax.bar(x_positions - 0.3, true_values, width=0.15, 
               label='True', alpha=0.8, color='black')
        
        # Plot predictions for each physics weight
        for i, weight in enumerate(physics_weights):
            if weight in all_results:
                pred_values = all_results[weight]['predictions'][target_name]
                offset = -0.15 + (i * 0.15)  # Distribute bars
                ax.bar(x_positions + offset, pred_values, width=0.15,
                       label=f'Physics Weight {weight}', alpha=0.7, 
                       color=colors[i % len(colors)])
        
        # Formatting
        ax.set_xticks(x_positions)
        ax.set_xticklabels(bus_labels, rotation=45, ha='right')
        ax.set_ylabel(ylabel)
        ax.set_title(f'{ylabel.split(" (")[0]} Comparison Across Physics Weights')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(True, alpha=0.3)
        
        # Add error annotations
        for i, weight in enumerate(physics_weights):
            if weight in all_results:
                pred_values = all_results[weight]['predictions'][target_name]
                errors = np.abs(pred_values - true_values)
                max_error_idx = np.argmax(errors)
                ax.annotate(f'Max Error: {errors[max_error_idx]:.4f}', 
                           xy=(0.02, 0.98), xycoords='axes fraction',
                           bbox=dict(boxstyle="round,pad=0.3", facecolor=colors[i % len(colors)], alpha=0.3),
                           verticalalignment='top', fontsize=8)
    
    plt.tight_layout()
    plt.show()

def plot_error_comparison_per_node(all_results, network, physics_weights=[0.0, 0.3, 0.5]):
    """
    Plot prediction errors per node for different physics loss weights
    """
    # Create bus labels
    bus_labels = []
    for i, bus in enumerate(network.buses.index):
        bus_type = network.buses.loc[bus, 'type']
        bus_labels.append(f"Bus {bus}\n({bus_type})")
    
    # Create figure
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    
    # Define colors
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    
    # Target variables
    targets = [
        ('v_mag', 'Voltage Magnitude Error (p.u.)', (0, 0)),
        ('v_ang', 'Voltage Angle Error (degrees)', (0, 1)),
        ('p', 'Active Power Error (MW)', (1, 0)),
        ('q', 'Reactive Power Error (MVAr)', (1, 1))
    ]
    
    # Plot errors for each target variable
    for target_name, ylabel, (row, col) in targets:
        ax = axes[row, col]
        
        x_positions = np.arange(len(bus_labels))
        
        # Calculate and plot errors for each physics weight
        for i, weight in enumerate(physics_weights):
            if weight in all_results:
                true_values = all_results[weight]['true_values'][target_name]
                pred_values = all_results[weight]['predictions'][target_name]
                errors = np.abs(pred_values - true_values)
                
                offset = (i - len(physics_weights)/2) * 0.15
                ax.bar(x_positions + offset, errors, width=0.15,
                       label=f'Physics Weight {weight}', alpha=0.7,
                       color=colors[i % len(colors)])
        
        # Formatting
        ax.set_xticks(x_positions)
        ax.set_xticklabels(bus_labels, rotation=45, ha='right')
        ax.set_ylabel(ylabel)
        ax.set_title(f'{ylabel.split(" Error")[0]} Absolute Errors')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def create_detailed_comparison_table(all_results, network, physics_weights=[0.0, 0.3, 0.5]):
    """
    Create a detailed comparison table showing errors per node and variable
    """
       
    # Initialize results dictionary
    comparison_data = []
    
    # Get bus information
    for i, bus in enumerate(network.buses.index):
        bus_type = network.buses.loc[bus, 'type']
        
        for target in ['v_mag', 'v_ang', 'p', 'q']:
            row = {
                'Bus': bus,
                'Bus_Type': bus_type,
                'Target': target
            }
            
            # Add true value
            true_val = all_results[physics_weights[0]]['true_values'][target][i]
            row['True_Value'] = true_val
            
            # Add predictions and errors for each physics weight
            for weight in physics_weights:
                if weight in all_results:
                    pred_val = all_results[weight]['predictions'][target][i]
                    error = abs(pred_val - true_val)
                    row[f'Pred_W{weight}'] = pred_val
                    row[f'Error_W{weight}'] = error
            
            comparison_data.append(row)
    
    # Create DataFrame
    df = pd.DataFrame(comparison_data)
    
    # Print summary statistics
    print("="*80)
    print("DETAILED NODE-WISE COMPARISON")
    print("="*80)
    
    for target in ['v_mag', 'v_ang', 'p', 'q']:
        print(f"\n{target.upper()} COMPARISON:")
        target_df = df[df['Target'] == target]
        
        print("\nMean Absolute Errors by Physics Weight:")
        for weight in physics_weights:
            if weight in all_results:
                mean_error = target_df[f'Error_W{weight}'].mean()
                print(f"  Physics Weight {weight}: {mean_error:.6f}")
        
        print(f"\nWorst Performing Nodes for {target}:")
        for weight in physics_weights:
            if weight in all_results:
                worst_idx = target_df[f'Error_W{weight}'].idxmax()
                worst_bus = target_df.loc[worst_idx, 'Bus']
                worst_error = target_df.loc[worst_idx, f'Error_W{weight}']
                print(f"  Physics Weight {weight}: Bus {worst_bus} (Error: {worst_error:.6f})")
    
    return df

# Usage example for your comparison pipeline:
def visualize_physics_weight_comparison(results_dict, test_network):
    """
    Complete visualization suite for physics weight comparison
    
    Args:
        results_dict: Dictionary with physics weights as keys and evaluation results as values
        test_network: Test network used for evaluation
    """
    physics_weights = list(results_dict.keys())
    
    print("Generating comparison visualizations...")
    
    # 1. Node-wise prediction comparison
    plot_predictions_comparison_per_node(results_dict, test_network, physics_weights)
    
    # 2. Node-wise error comparison  
    plot_error_comparison_per_node(results_dict, test_network, physics_weights)
    
    # 3. Detailed comparison table
    comparison_df = create_detailed_comparison_table(results_dict, test_network, physics_weights)
    
    # 4. Best performing model per node
    print("\n" + "="*60)
    print("BEST PERFORMING MODEL PER NODE")
    print("="*60)
    
    for target in ['v_mag', 'v_ang', 'p', 'q']:
        print(f"\n{target.upper()}:")
        target_df = comparison_df[comparison_df['Target'] == target]
        
        for _, row in target_df.iterrows():
            bus = row['Bus']
            bus_type = row['Bus_Type']
            
            # Find best physics weight for this node
            errors = {w: row[f'Error_W{w}'] for w in physics_weights if f'Error_W{w}' in row}
            best_weight = min(errors.keys(), key=lambda w: errors[w])
            best_error = errors[best_weight]
            
            print(f"  Bus {bus} ({bus_type}): Physics Weight {best_weight} (Error: {best_error:.6f})")
    
    return comparison_df





In [ ]:


# Suppress warnings and set logging levels
#warnings.filterwarnings('ignore')
logging.getLogger('torch').setLevel(logging.ERROR)
logging.getLogger('matplotlib').setLevel(logging.ERROR)

In [ ]:
#Old - dont use...
# training data:
#networks = generate_training_data(n, num_scenarios=100, steps_per_scenario=24)
    
#    # Run comprehensive comparison
#results, comparison_df = comprehensive_physics_loss_comparison(networks, num_epochs=50)

    # Visualize results
#visualize_comprehensive_comparison(results, comparison_df)


### Improved version

In [ ]:
# %% [markdown]
# ## Physics Loss Comparison Functions
# Functions to compare different physics loss weights

# %%
def train_multiple_models(networks, physics_weights=[0.0, 0.01, 0.1, 0.5], num_epochs=50):
    """Train multiple models with different physics loss weights"""
    models = {}
    training_histories = {}
    
    for weight in physics_weights:
        print(f"\n🔄 Training model with physics weight: {weight}")
        print("-" * 50)
        
        model, (train_losses, val_losses) = train_power_flow_gnn(
            networks, num_epochs=num_epochs, weight_physics=weight
        )
        
        models[weight] = model
        training_histories[weight] = {
            'train_losses': train_losses,
            'val_losses': val_losses
        }
        
        print(f"✅ Model with physics weight {weight} trained successfully")
    
    return models, training_histories

# %%
def evaluate_all_models(models, test_network):
    """Evaluate all models on a test network and return results"""
    all_results = {}
    
    for weight, model in models.items():
        results = evaluate_model(model, test_network)
        all_results[weight] = results
        print(f"Model with physics weight {weight} evaluated")
    
    return all_results

# %%
def plot_training_comparison(training_histories):
    """Plot training curves for all models side by side"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Training loss comparison
    for weight, history in training_histories.items():
        ax1.plot(history['train_losses'], label=f'Physics Weight {weight}', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Training Loss')
    ax1.set_title('Training Loss Comparison')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Validation loss comparison
    for weight, history in training_histories.items():
        ax2.plot(history['val_losses'], label=f'Physics Weight {weight}', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Validation Loss')
    ax2.set_title('Validation Loss Comparison')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# %%
def plot_predictions_comparison_per_node(all_results, network, physics_weights):
    """Plot prediction vs true values per node for different physics loss weights"""
    bus_labels = [f"Bus {bus}\n({network.buses.loc[bus, 'type']})" for bus in network.buses.index]
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    colors = ['black', 'red', 'blue', 'green', 'orange']  # black for true values
    
    targets = [
        ('v_mag', 'Voltage Magnitude (p.u.)', (0, 0)),
        ('v_ang', 'Voltage Angle (degrees)', (0, 1)), 
        ('p', 'Active Power (MW)', (1, 0)),
        ('q', 'Reactive Power (MVAr)', (1, 1))
    ]
    
    for target_name, ylabel, (row, col) in targets:
        ax = axes[row, col]
        x_positions = np.arange(len(bus_labels))
        
        # Get true values (same across all models)
        true_values = all_results[physics_weights[0]]['true_values'][target_name]
        
        # Plot true values as baseline
        ax.bar(x_positions - 0.3, true_values, width=0.15, 
               label='True', alpha=0.8, color='black')
        
        # Plot predictions for each physics weight
        for i, weight in enumerate(physics_weights):
            if weight in all_results:
                pred_values = all_results[weight]['predictions'][target_name]
                offset = -0.15 + (i * 0.15)
                ax.bar(x_positions + offset, pred_values, width=0.15,
                       label=f'Physics Weight {weight}', alpha=0.7, 
                       color=colors[(i+1) % len(colors)])
        
        # Formatting
        ax.set_xticks(x_positions)
        ax.set_xticklabels(bus_labels, rotation=45, ha='right')
        ax.set_ylabel(ylabel)
        ax.set_title(f'{ylabel.split(" (")[0]} Comparison Across Physics Weights')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# %%
def plot_errors_comparison_per_node(all_results, network, physics_weights):
    """Plot prediction errors per node for different physics loss weights"""
    bus_labels = [f"Bus {bus}\n({network.buses.loc[bus, 'type']})" for bus in network.buses.index]
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    
    targets = [
        ('v_mag', 'Voltage Magnitude Error (p.u.)', (0, 0)),
        ('v_ang', 'Voltage Angle Error (degrees)', (0, 1)),
        ('p', 'Active Power Error (MW)', (1, 0)),
        ('q', 'Reactive Power Error (MVAr)', (1, 1))
    ]
    
    for target_name, ylabel, (row, col) in targets:
        ax = axes[row, col]
        x_positions = np.arange(len(bus_labels))
        
        # Calculate and plot errors for each physics weight
        for i, weight in enumerate(physics_weights):
            if weight in all_results:
                true_values = all_results[weight]['true_values'][target_name]
                pred_values = all_results[weight]['predictions'][target_name]
                errors = np.abs(pred_values - true_values)
                
                offset = (i - len(physics_weights)/2) * 0.2
                ax.bar(x_positions + offset, errors, width=0.15,
                       label=f'Physics Weight {weight}', alpha=0.7,
                       color=colors[i % len(colors)])
        
        # Formatting
        ax.set_xticks(x_positions)
        ax.set_xticklabels(bus_labels, rotation=45, ha='right')
        ax.set_ylabel(ylabel)
        ax.set_title(f'{ylabel.split(" Error")[0]} Absolute Errors')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# %%
def print_comparison_summary(all_results, physics_weights):
    """Print summary statistics comparing all models"""
    print("="*80)
    print("PHYSICS LOSS COMPARISON SUMMARY")
    print("="*80)
    
    # Create comparison table
    metrics = ['v_mag_mae', 'v_ang_rmse', 'p_mae', 'q_mae']
    metric_names = ['Voltage Mag MAE', 'Voltage Ang RMSE', 'Active Power MAE', 'Reactive Power MAE']
    
    print(f"{'Metric':<20}", end="")
    for weight in physics_weights:
        print(f"{'Physics ' + str(weight):<15}", end="")
    print()
    print("-" * (20 + 15 * len(physics_weights)))
    
    for metric, name in zip(metrics, metric_names):
        print(f"{name:<20}", end="")
        for weight in physics_weights:
            value = all_results[weight]['metrics'][metric]
            print(f"{value:<15.6f}", end="")
        print()
    
    # Find best performing model for each metric
    print(f"\n{'BEST PERFORMING MODEL PER METRIC':^80}")
    print("=" * 80)
    
    for metric, name in zip(metrics, metric_names):
        best_weight = min(physics_weights, 
                         key=lambda w: all_results[w]['metrics'][metric])
        best_value = all_results[best_weight]['metrics'][metric]
        print(f"{name}: Physics Weight {best_weight} ({best_value:.6f})")

# %%
def create_detailed_node_comparison_table(all_results, network, physics_weights):
    """Create detailed comparison table showing errors per node and variable"""
    comparison_data = []
    
    for i, bus in enumerate(network.buses.index):
        bus_type = network.buses.loc[bus, 'type']
        
        for target in ['v_mag', 'v_ang', 'p', 'q']:
            row = {'Bus': bus, 'Bus_Type': bus_type, 'Target': target}
            
            # Add true value
            true_val = all_results[physics_weights[0]]['true_values'][target][i]
            row['True_Value'] = true_val
            
            # Add predictions and errors for each physics weight
            for weight in physics_weights:
                if weight in all_results:
                    pred_val = all_results[weight]['predictions'][target][i]
                    error = abs(pred_val - true_val)
                    row[f'Pred_W{weight}'] = pred_val
                    row[f'Error_W{weight}'] = error
            
            comparison_data.append(row)
    
    df = pd.DataFrame(comparison_data)
    
    # Print summary for each target
    print("="*80)
    print("DETAILED NODE-WISE COMPARISON")
    print("="*80)
    
    for target in ['v_mag', 'v_ang', 'p', 'q']:
        print(f"\n{target.upper()} COMPARISON:")
        target_df = df[df['Target'] == target]
        
        print("\nMean Absolute Errors by Physics Weight:")
        for weight in physics_weights:
            if weight in all_results:
                mean_error = target_df[f'Error_W{weight}'].mean()
                print(f"  Physics Weight {weight}: {mean_error:.6f}")
        
        print(f"\nBest Performing Nodes for {target}:")
        for weight in physics_weights:
            if weight in all_results:
                best_idx = target_df[f'Error_W{weight}'].idxmin()
                best_bus = target_df.loc[best_idx, 'Bus']
                best_error = target_df.loc[best_idx, f'Error_W{weight}']
                print(f"  Physics Weight {weight}: Bus {best_bus} (Error: {best_error:.6f})")
    
    return df


In [ ]:
# %%
# Step 1: Generate training data (if not already done)
#if 'networks' not in globals():
networks = generate_training_data(n, num_scenarios=50, steps_per_scenario=24)

# %%
# Step 2: Train multiple models with different physics weights
physics_weights = [0.0, 0.0001, 0.001, 0.01, 0.1]
models, training_histories = train_multiple_models(networks, physics_weights, num_epochs=50)

# %%
# Step 3: Compare training curves
plot_training_comparison(training_histories)

# %%
# Step 4: Create test network and evaluate all models
test_network = create_9_bus_network(steps=1, p_set_gen2=1.75, p_set_gen3=0.95, plot=False)
test_network.pf(use_seed=True)

all_results = evaluate_all_models(models, test_network)

# %%
# Step 5: Compare predictions per node
plot_predictions_comparison_per_node(all_results, test_network, physics_weights)

# %%
# Step 6: Compare errors per node
plot_errors_comparison_per_node(all_results, test_network, physics_weights)

# %%
# Step 7: Print summary statistics
print_comparison_summary(all_results, physics_weights)

# %%
# Step 8: Detailed node-wise analysis
detailed_df = create_detailed_node_comparison_table(all_results, test_network, physics_weights)

# %%
# Optional: Save results for later analysis
# detailed_df.to_csv('physics_loss_comparison_results.csv', index=False)


In [ ]:

def visualize_graph(data):
    G = to_networkx(data, to_undirected=True)
    
    # Get node features for coloring (e.g., bus type)
    node_types = data.x[:, :3].argmax(dim=1).numpy()
    
    # Define colors for different bus types
    colors = ['red', 'green', 'blue']
    node_colors = [colors[node_type] for node_type in node_types]
    
    # Prepare node labels with features
    node_labels = {}
    for i, node in enumerate(G.nodes()):
        features = data.x[i].numpy()
        # Format features as string, e.g., bus type and first few features
        bus_type = ['Slack', 'PV', 'PQ'][node_types[i]]
        p = features[3]  # active power (index 3)
        q = features[4]  # reactive power (index 4)
        v_mag = features[5]  # voltage magnitude (index 5)
        v_ang = features[6]  # voltage angle (index 6)
        label = f"{node}\nType: {bus_type}\nP: {p:.2f}\nQ: {q:.2f}\nV: {v_mag:.2f}\nθ: {v_ang:.2f}"
        node_labels[node] = label
    
    # Prepare edge labels with features
    edge_labels = {}
    for u, v in G.edges():
        # Find edge index in data.edge_index
        edge_idx = None
        for idx in range(data.edge_index.size(1)):
            if (data.edge_index[0, idx].item() == u and data.edge_index[1, idx].item() == v) or \
               (data.edge_index[0, idx].item() == v and data.edge_index[1, idx].item() == u):
                edge_idx = idx
                break
                
        if edge_idx is not None:
            edge_feat = data.edge_attr[edge_idx].numpy()
            r, x, b, s_nom = edge_feat[:4]
            label = f"r:{r:.3f}\nx:{x:.3f}\nb:{b:.3f}\nS:{s_nom:.1f}"
            edge_labels[(u, v)] = label
        else:
            edge_labels[(u, v)] = ""
    
    # Plot
    plt.figure(figsize=(12, 12))
    pos = nx.spring_layout(G, seed=42)  # Fixed seed for reproducibility
    
    # Draw graph with nodes but without default labels
    nx.draw(G, pos, with_labels=False, node_color=node_colors, 
            node_size=700, width=2, edge_color='gray')
    
    # Adjust node label positions to be beside nodes
    label_pos = {}
    for node, (x, y) in pos.items():
        label_pos[node] = (x + 0.1, y)  # Shift label slightly to the right
    
    # Add custom node and edge labels
    nx.draw_networkx_labels(G, label_pos, labels=node_labels, font_size=8)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)
    
    plt.title("Power System Graph with Node and Edge Features")
    plt.axis('off')
    plt.show()


In [ ]:
# Assuming you have your dataset created
dataset = PowerFlowDataset(networks)
sample_data = dataset[0]  # Get the first graph from your dataset

# Call the visualization function
visualize_graph(sample_data)

In [ ]:
def visualize_pypsa_network(network):

    
    # Create a networkx graph
    G = nx.Graph()
    
    # Add nodes (buses)
    for i, bus in enumerate(network.buses.index):
        bus_type = network.buses.loc[bus, 'type']
        # Add node with attributes
        G.add_node(bus, type=bus_type)
    
    # Add edges (lines and transformers)
    for _, line in network.lines.iterrows():
        G.add_edge(line['bus0'], line['bus1'], 
                  r=line['r'], x=line['x'], 
                  component='line')
    
    for _, trafo in network.transformers.iterrows():
        G.add_edge(trafo['bus0'], trafo['bus1'], 
                  r=trafo['r'], x=trafo['x'], 
                  component='transformer')
    
    # Define node colors based on bus type
    color_map = {'Slack': 'red', 'PV': 'green', 'PQ': 'blue'}
    node_colors = [color_map[G.nodes[node]['type']] for node in G.nodes]
    
    # Define edge colors based on component type
    edge_colors = ['brown' if G.edges[edge]['component'] == 'transformer' else 'black' 
                  for edge in G.edges]
    
    # Get positions from PyPSA if available
    if 'x' in network.buses.columns and 'y' in network.buses.columns:
        pos = {bus: (network.buses.loc[bus, 'x'], network.buses.loc[bus, 'y']) 
              for bus in network.buses.index}
    else:
        pos = nx.spring_layout(G)
    
    # Plot
    plt.figure(figsize=(12, 10))
    nx.draw(G, pos, with_labels=True, node_color=node_colors, 
            node_size=700, width=2, edge_color=edge_colors, font_size=10)
    
    # Add legend
    bus_types = ['Slack', 'PV', 'PQ']
    legend_colors = [color_map[t] for t in bus_types]
    plt.legend(bus_types, loc='upper right', 
               prop={'size': 12}, 
               markerscale=1.5,
               scatterpoints=1,
               scatteryoffsets=[0.5],
               handletextpad=1,
               handlelength=1,
               handleheight=1,
               frameon=True)
    
    plt.title("Power System Network Structure")
    plt.tight_layout()
    plt.show()
    
    return G  # Return the graph for further analysis if needed


In [ ]:
# Create or load your PyPSA network
network = create_9_bus_network(steps=1, p_set_gen2=170, p_set_gen3=100)

# Visualize the network
visualize_pypsa_network(network)
